# PYTHON ITERATORS, ITERABLES, GENERATORS & LAZY EVALUATION

### *A Comprehensive Professional Handbook for Data, ML and AI Engineering*

From the for Loop to Async Generators: The Iteration Protocol, yield, itertools, Streaming Pipelines and Production Patterns

**Kajola Gbenga**  
*Data Scientist & AI Engineer*

---

This handbook is dedicated entirely to iteration: iterables, iterators, the iteration protocol, generators, yield, generator expressions, lazy evaluation, itertools, and asynchronous iteration. It assumes familiarity with, and does not reproduce, companion handbooks on Python Data Types, Methods, Functions, Operators, Keywords, OOP, Exception Handling, File Handling, and Modules & Packages; where those topics are touched (a class's __init__, a try/except block, opening a file), it is only to the minimal extent needed to explain iteration behavior.

---

## Table of Contents

- Part 1. Introduction to Iteration
- Part 2. Iterables
- Part 3. Iterators
- Part 4. The Iteration Protocol
- Part 5. iter() and next()
- Part 6. StopIteration
- Part 7. Building Custom Iterators
- Part 8. Iterable Classes vs Iterator Classes
- Part 9. Generators
- Part 10. yield
- Part 11. Generator Execution: The Complete Lifecycle
- Part 12. Generator Expressions
- Part 13. Memory Efficiency
- Part 14. Lazy Evaluation
- Part 15. Generators and Files
- Part 16. Generators and Data Science
- Part 17. Generators and Machine Learning
- Part 18. Generators and AI Engineering
- Part 19. yield from
- Part 20. Generator Return Values
- Part 21. Sending Values Into Generators (.send())
- Part 22. .throw() and .close()
- Part 23. Generators vs Custom Iterator Classes
- Part 24. itertools: A Comprehensive Practical Guide
- Part 25. itertools for Data Processing
- Part 26. Iteration and zip()
- Part 27. Iteration and enumerate()
- Part 28. Iteration and map(), filter(), and Related Tools
- Part 29. Iteration and Comprehensions
- Part 30. Iteration With Dictionaries
- Part 31. Iteration With Files
- Part 32. Async Iterators and Async Generators
- Part 33. Iterators and Performance
- Part 34. Common Iteration Mistakes
- Part 35. Real-World Projects
- Part 36. Interview Questions
- Part 37. Consolidated Practical Exercises
- Part 38. Complete Iteration Cheat Sheet

---

> This handbook progresses from absolute-beginner iteration concepts in Part 1 through async generators and production data/AI pipelines by Part 32, with dedicated performance, mistakes, projects, interview-question and cheat-sheet parts closing it out. Read start to finish once, then use the Table of Contents, Part 24's itertools reference, and Part 38's cheat sheet for ongoing lookup.

# Part 1. Introduction to Iteration

Iteration is the act of visiting the elements of a collection one at a time, in sequence. It is one of the most-used mechanisms in all of Python, and everything this handbook covers, from a simple for loop to an asynchronous streaming pipeline, is built on the same handful of underlying ideas introduced here.

## 1.1 What Iteration Means

Imagine a librarian handing you books from a shelf one at a time: you take one book, look at it, hand it back or set it aside, then the librarian hands you the next one, until the shelf is empty. That, exactly, is iteration: retrieving items from a collection one at a time, in order, until there are no more items left.

In Python, iteration is not just a convenience feature; it is a formal protocol (a defined set of rules an object must follow) that virtually every built-in collection type implements, and that you can implement yourself for custom objects, as this handbook will show in detail.

## 1.2 Why Iteration Is Important

- It is the primary way Python code processes collections: loops, comprehensions, and functions like sum() and sorted() are all built on iteration underneath.
- It generalizes across wildly different data sources: the exact same 'for item in x:' syntax works whether x is a list in memory, a file on disk, a database cursor, or a network stream, because they all speak the same iteration protocol.
- It enables lazy, memory-efficient processing of data that may be too large to hold in memory all at once, which is the foundation for everything in Parts 9 through 18 on generators and lazy evaluation.
- It underlies nearly every data pipeline in Data Science, Machine Learning, and AI Engineering: reading a huge CSV, streaming API responses, feeding mini-batches to a model, are all iteration at their core.

## 1.3 Iterating Over Collections

In [1]:
fruits = ["mango", "pawpaw", "banana"]
for fruit in fruits:
    print(fruit)

mango
pawpaw
banana


| Code / Item | Explanation |
|---|---|
| `for fruit in fruits:` | Requests items from the list fruits one at a time, binding each one in turn to the name fruit, until the list is exhausted. |

## 1.4 for Loops and Iteration

A Python for loop is, in effect, syntactic sugar: convenient, readable surface syntax that hides a more explicit, lower-level sequence of steps the interpreter performs on your behalf. Part 3 unpacks exactly what those steps are; for now, understand that 'for x in y:' always means 'iterate over y, using the iteration protocol, and bind each produced value to x'.

## 1.5 What Happens Internally During a for Loop (Preview)

At a high level, a for loop asks the collection for an iterator (a helper object that tracks progress), then repeatedly asks that iterator for the next value, stopping automatically the moment the iterator signals there is nothing left. Part 3 walks through this mechanism in complete, precise detail with worked examples; this is only a preview so the vocabulary in Part 2 makes sense.

## 1.6 Iteration vs Repetition

Repetition (looping a fixed number of times, such as 'for i in range(5):') and iteration (visiting each element of an existing collection) are related but distinct ideas. Repetition is about running code N times; iteration is specifically about walking through a sequence of values one at a time, produced by an iterable. range() itself happens to be iterable, which is exactly why 'for i in range(5):' works, but not every repeated action involves iterating over a meaningful collection of data.

## 1.7 Iteration vs Indexing

Indexing (accessing an element by its numeric position, like my_list[3]) is a different mechanism from iteration, even though the two often achieve similar-looking results for sequence types like list. Indexing requires the collection to support positional lookup by number; iteration only requires the collection to be able to produce its elements one at a time, in some order, which is a strictly weaker and more general requirement. This is precisely why some iterables, such as a set or a file object, support iteration perfectly well but do not support indexing at all.

In [2]:
numbers = {10, 20, 30}
for n in numbers:
    print(n)          # iteration works fine
try:
    numbers[0]         # indexing does NOT work on a set
except TypeError as e:
    print("TypeError:", e)

10
20
30
TypeError: 'set' object is not subscriptable


## Part 1 Practice Exercises

**Beginner:** In your own words, what is the difference between iteration and indexing?

*Answer:* Iteration means retrieving a collection's elements one at a time, in sequence, using the iteration protocol, and works on any object that supports it, regardless of whether that object has a notion of numeric position. Indexing means retrieving a specific element directly by its numeric position (my_list[3]), and only works on collections that explicitly support positional access.

**Intermediate:** Why does 'for n in {10, 20, 30}:' work even though {10, 20, 30}[0] raises a TypeError?

*Answer:* A set supports the iteration protocol (it can produce its elements one at a time through iter()/next()), which is all a for loop requires. A set does not support positional indexing, because it is unordered and has no meaningful concept of 'the element at position 0', so [] access is not implemented for it at all.

**Advanced:** Explain why 'for i in range(5):' is described as repetition using an iterable, rather than pure repetition, and why this distinction matters for understanding later parts of this handbook.

*Answer:* range(5) is not simply a repeat-this-N-times instruction; it is itself a concrete, iterable sequence object representing the values 0 through 4. The for loop's behavior in this case is identical to iterating over any other iterable, such as a list: it obtains an iterator from range(5) and pulls successive values from it. The loop only happens to repeat five times as a side effect of range(5) producing exactly five values. This distinction matters because it means everything this handbook explains about the iteration protocol, iterators, and StopIteration applies equally to 'for i in range(5):' as it does to iterating over a list, a file, or a generator; there is no separate, special-cased 'repetition mechanism' in Python distinct from iteration itself.

# Part 2. Iterables

An iterable is any object capable of returning its elements one at a time. This part defines the concept precisely, shows how to recognize an iterable, tours Python's common built-in iterables, and introduces the __iter__() method that formally makes an object iterable.

## 2.1 What an Iterable Is (Formal Definition)

Formally, an object is iterable if it implements a method called __iter__(), which, when called, returns an iterator (Part 3 defines exactly what an iterator is). In practice, this means: any object you can legally write after the 'in' keyword in a for loop is an iterable.

Being iterable says nothing about how the data is stored, whether it is ordered, or whether it can be indexed; it is purely a promise that the object knows how to hand out an iterator over its contents on request.

## 2.2 How to Identify an Iterable

In [3]:
def is_iterable(obj):
    return hasattr(obj, "__iter__")

print(is_iterable([1, 2, 3]))
print(is_iterable("hello"))
print(is_iterable(42))
print(is_iterable({1, 2, 3}))

True
True
False
True


| Code / Item | Explanation |
|---|---|
| `hasattr(obj, "__iter__")` | Directly checks whether the object's class defines the __iter__ method, which is the formal, defining trait of an iterable. |
| `is_iterable(42)` | Returns False, because int has no notion of 'elements' to hand out one at a time; a single number is not a collection. |

> **Note:** The more robust, idiomatic check is 'from collections.abc import Iterable; isinstance(obj, Iterable)', which also correctly recognizes objects that implement iteration through the older __getitem__-based protocol; hasattr is shown here because it makes the underlying mechanism explicit.

## 2.3 Common Python Iterables

| Code / Item | Explanation |
|---|---|
| `list` | Iterates over elements in their stored order, e.g. for x in [1, 2, 3]: |
| `tuple` | Iterates over elements in their stored order, identical behavior to list. |
| `str` | Iterates character by character, in order. |
| `dict` | Iterates over its keys by default (Part 30 covers .values(), .items() and views in full). |
| `set / frozenset` | Iterates over elements in an unspecified, hash-based order (Part 17 of the Data Types handbook covers this ordering behavior). |
| `file objects` | Iterate line by line when opened for reading text (Part 31). |
| `range` | Iterates over its defined arithmetic sequence of integers, computed lazily. |
| `Custom objects` | Any user-defined class that implements __iter__() (Part 7). |

## 2.4 A Quick Tour

In [4]:
for x in [1, 2, 3]:
    print("list:", x)
for x in (1, 2):
    print("tuple:", x)
for x in "ab":
    print("str:", x)
for x in {"a": 1, "b": 2}:
    print("dict (keys):", x)
for x in range(2):
    print("range:", x)

list: 1
list: 2
list: 3
tuple: 1
tuple: 2
str: a
str: b
dict (keys): a
dict (keys): b
range: 0
range: 1


## 2.5 The __iter__() Method and How It Relates to Iteration

__iter__() is a dunder (double-underscore) method that every iterable class defines. When Python encounters 'for x in obj:', or when you call iter(obj) yourself, it is __iter__() that gets invoked behind the scenes; whatever object __iter__() returns is treated as the iterator that will actually produce the values, one 'next' request at a time.

Crucially, __iter__() does not itself produce the elements; it produces (or returns) something else, an iterator, whose job is to produce the elements. This separation of roles, an iterable that can create iterators versus an iterator that actually walks through the data, is the single most important conceptual distinction in this entire handbook, and Part 3 and Part 8 examine it exhaustively.

In [5]:
numbers = [10, 20, 30]
iterator = numbers.__iter__()   # exactly what iter(numbers) does
print(iterator)
print(type(iterator))

<class 'list_iterator'>


| Code / Item | Explanation |
|---|---|
| `numbers.__iter__()` | Directly calls the dunder method; this is precisely what the built-in function iter(numbers) does on your behalf, more conventionally (Part 5). |
| `type(iterator)` | Reveals that a list's iterator is a distinct type, list_iterator, not a list itself; the iterable and the iterator it produces are genuinely different kinds of object. |

## Part 2 Practice Exercises

**Beginner:** What single method must an object's class implement for that object to be considered iterable?

*Answer:* __iter__().

**Intermediate:** Why does calling is_iterable(42) (checking for a __iter__ attribute) return False for the integer 42?

*Answer:* int represents a single scalar value with no internal collection of elements to hand out one at a time, so its class does not define an __iter__ method at all; without that method, an object does not satisfy Python's definition of an iterable, and attempting 'for x in 42:' raises TypeError: 'int' object is not iterable.

**Advanced:** Explain precisely what numbers.__iter__() returns, and why that returned object is not itself 'the list', in terms of the roles Part 2 introduces.

*Answer:* numbers.__iter__() returns a new object, an iterator (specifically a list_iterator object for a list), whose sole job is to track progress through numbers and produce its elements one at a time on request. It is not the list itself, and it is not a copy of the list's data; it is a separate, lightweight helper object that holds a reference to numbers plus an internal position marker. This separation exists so that the iterable (numbers) can remain a passive, reusable data source, while the iterator does the actual, stateful work of remembering 'how far through have we gotten', which, as Part 8 explains, is also exactly what allows multiple independent iterators to be created from the same iterable simultaneously, each tracking its own separate position.

# Part 3. Iterators

Where an iterable is a source that knows how to produce an iterator, an iterator is the object that actually does the work: it remembers position and hands out the next value on demand. This part defines iterators precisely and walks through, step by step, exactly what Python does internally when it executes a for loop.

## 3.1 What an Iterator Is (Formal Definition)

An iterator is an object that implements two methods: __iter__(), which must return the iterator itself, and __next__(), which returns the next value in the sequence each time it is called, and raises StopIteration once there are no more values to give. Any object satisfying both requirements is, by definition, an iterator.

## 3.2 Iterable vs Iterator: The Core Distinction

The table below (3.2a) contrasts the two roles directly; it is worth returning to throughout the rest of this handbook whenever the distinction feels blurry.

## 3.3 iter() and next()

iter(obj) is the built-in function form of calling obj.__iter__(); next(it) is the built-in function form of calling it.__next__(). Both exist as ordinary functions, rather than requiring you to call the dunder methods directly, following Python's general convention of providing a clean functional interface over dunder methods (compare len(obj) and obj.__len__()).

In [6]:
numbers = [10, 20, 30]
it = iter(numbers)
print(next(it))
print(next(it))
print(next(it))
try:
    next(it)
except StopIteration:
    print("StopIteration raised: the iterator is exhausted")

10
20
30
StopIteration raised: the iterator is exhausted


## 3.4 __iter__() and __next__() on an Iterator

An important, easily-missed detail: an iterator's own __iter__() method simply returns itself (return self). This is precisely what makes every iterator also, trivially, an iterable, which is what allows you to write 'for x in some_iterator:' directly, and is also why a for loop's internal machinery can treat 'the thing I'm iterating over' uniformly, whether it was originally a list or was already an iterator.

In [7]:
numbers = [1, 2, 3]
it = iter(numbers)
print(it is iter(it))   # True: an iterator's __iter__() returns itself

True


## 3.5 StopIteration

StopIteration is a built-in exception class with one job: signal 'there are no more values'. An iterator's __next__() method raises it (rather than, say, returning None or some other sentinel) specifically because raising an exception cannot be confused with a legitimately produced value; None, for instance, might be a perfectly valid item in the sequence itself, so a special exception is used instead of a special return value. Part 6 covers StopIteration in complete depth.

## 3.6 Exactly What Happens Internally During 'for item in collection:'

A for loop is precise, mechanical syntactic sugar for a specific sequence of steps using iter() and next(). Understanding this equivalence removes essentially all mystery from how iteration works in Python.

In [8]:
collection = ["a", "b", "c"]

# What 'for item in collection: print(item)' ACTUALLY does, spelled out:
iterator = iter(collection)      # Step 1: get an iterator from the iterable
while True:
    try:
        item = next(iterator)     # Step 2: request the next value
    except StopIteration:         # Step 3: catch the 'no more values' signal
        break                      # Step 4: exit the loop cleanly
    print(item)                    # Step 5: run the loop body with that value

a
b
c


| Code / Item | Explanation |
|---|---|
| `iterator = iter(collection)` | Step 1: obtains a fresh iterator from the iterable by calling its __iter__() method, exactly once, at the very start of the loop. |
| `item = next(iterator)` | Step 2: requests the next value from the iterator on every pass, by calling its __next__() method. |
| `except StopIteration: break` | Step 3-4: the for loop silently catches StopIteration for you and simply ends the loop, which is precisely why you never see a StopIteration traceback from an ordinary for loop, even though it is happening internally every single time one completes. |
| `print(item)` | Step 5: the loop body executes using whatever value next() most recently produced. |

> **Note:** This exact five-step pattern (iter(), then a while-True/next()/except-StopIteration/break loop) is the literal definition of what 'for x in y:' means in Python; every for loop over every iterable, without exception, reduces to this.

## 3.2a Iterable vs Iterator Comparison

| Aspect | Iterable | Iterator |
|---|---|---|
| Defining method(s) | __iter__() only (must return an iterator) | __iter__() (returns self) AND __next__() |
| Role | A source of data; can be asked to produce an iterator | The active worker; produces values and tracks position |
| State | Generally stateless with respect to iteration progress | Stateful: remembers exactly where it is |
| Reusability | Produces a fresh iterator (from the start) every time iter() is called | Exhausted after one full pass; cannot be restarted |
| Example | [1, 2, 3] (a list) | iter([1, 2, 3]) (a list_iterator) |

## Part 3 Practice Exercises

**Beginner:** What two methods must an object implement to qualify as an iterator?

*Answer:* __iter__() (which must return the iterator itself) and __next__() (which returns the next value, or raises StopIteration when exhausted).

**Intermediate:** Why does an iterator's __iter__() method simply 'return self'?

*Answer:* Doing so makes every iterator also satisfy the definition of an iterable (since it has a working __iter__() method), which is what allows an iterator to be used directly in a for loop, or passed to iter() again harmlessly, without needing any special-case handling to distinguish 'a plain iterable' from 'something that is already an iterator'.

**Advanced:** Manually rewrite 'for x in range(3): print(x * 2)' using only iter(), next(), a while loop, and a try/except block, without using the for keyword at all.

*Answer:* it = iter(range(3))
while True:
    try:
        x = next(it)
    except StopIteration:
        break
    print(x * 2)
This reproduces exactly what the for loop does internally: obtain one iterator from range(3) via iter(), repeatedly request values with next(), and stop the loop the instant a StopIteration exception is raised, running the loop body (print(x * 2)) once per successfully retrieved value in between.

# Part 4. The Iteration Protocol

The iteration protocol is the formal name for the small, precise contract that Parts 2 and 3 have been building toward: the exact rules an object must follow to participate correctly in Python's for loops, comprehensions, unpacking, and every other iteration-based construct. This part consolidates that contract into one complete, precise reference, with an execution diagram.

## 4.1 The Full Protocol, Stated Precisely

- An iterable must implement __iter__(self), which takes no other arguments and returns an iterator.
- An iterator must implement __iter__(self), which returns self (making every iterator also an iterable).
- An iterator must implement __next__(self), which takes no other arguments, returns the next value in the sequence, and raises StopIteration (with no meaningful return value) once exhausted.
- Once __next__() has raised StopIteration, well-behaved iterators are expected to continue raising StopIteration on all subsequent calls, rather than unexpectedly resuming (Part 34 discusses violations of this expectation as a common mistake).

## 4.2 Iterator State: What 'State' Means Here

State, in this context, means whatever internal data an iterator needs to remember between successive __next__() calls in order to know what to produce next: typically, at minimum, a reference to the underlying data and a position marker (an index, a pointer, or something equivalent). Because this state genuinely persists on the iterator object between calls, calling next() twice in a row on the same iterator never produces the same value twice (assuming the underlying data has more than one distinct position to offer); each call picks up exactly where the previous one left off.

## 4.3 Why Iterators Are Stateful (and Iterables Generally Are Not)

This statefulness is precisely what lets multiple independent loops walk through the same iterable simultaneously without interfering with each other, an idea Part 8 explores in full: the iterable itself stays a passive, unmodified data source, while each separate call to iter() on it produces a brand-new iterator with its own, independent position counter. If the iterable itself tracked position, two simultaneous loops over the same list would corrupt each other's progress.

In [9]:
numbers = [10, 20, 30]
it_a = iter(numbers)
it_b = iter(numbers)
print(next(it_a))   # 10
print(next(it_a))   # 20
print(next(it_b))   # 10 again: it_b has its OWN independent position

10
20
10


| Code / Item | Explanation |
|---|---|
| `it_a = iter(numbers) / it_b = iter(numbers)` | Two separate calls to iter() on the same list produce two separate, independent iterator objects, each starting fresh from the beginning. |
| `next(it_b) returning 10 after it_a already advanced` | Proves the two iterators do not share state; each iterator's position is tracked entirely on that iterator object, not on the shared list. |

## 4.4 Execution-Flow Diagram: The Iteration Protocol in Action

The following ASCII flow traces the exact sequence of protocol calls behind 'for x in iterable: BODY', matching the five-step breakdown from Part 3.6:

```
iterable
   |
   |  __iter__()
   v
iterator  <---------------------------+
   |                                    |
   |  __next__()                        |  loop back for next item
   v                                    |
 has a value? ----yes----> BODY runs ---+
   |
   no (raises StopIteration)
   |
   v
loop exits normally
```

> **Note:** This diagram is the single mental model to keep for the rest of this handbook: an iterable produces one iterator; that iterator is repeatedly asked for the next value; the loop body runs once per successfully produced value; the loop ends the instant StopIteration is raised.

## Part 4 Practice Exercises

**Beginner:** According to the iteration protocol, what must an iterator's __next__() method do once there are no more values left to produce?

*Answer:* It must raise StopIteration.

**Intermediate:** Explain why it is the iterator, not the iterable, that holds the position/progress state during iteration.

*Answer:* Keeping the position state on the iterator, rather than on the iterable, allows the iterable to remain a stable, reusable, passive data source that can produce any number of independent iterators, each with its own separate progress marker. If the iterable itself tracked position, only one iteration pass could ever happen at a time, and starting a second loop over the same collection while the first was still in progress would corrupt or interfere with the first loop's position.

**Advanced:** Using the concept of iterator state, explain why 'it = iter([1, 2, 3]); list(it); list(it)' produces [1, 2, 3] the first time and [] the second time.

*Answer:* iter([1, 2, 3]) creates one specific iterator object, it, with its own internal position marker starting at the beginning. The first list(it) call fully drains that iterator by repeatedly calling next() until StopIteration, advancing its internal position marker all the way to the end and collecting every value along the way into a new list, [1, 2, 3]. Because it is the same iterator object being reused (not a fresh one), the second list(it) call starts from that iterator's current, already-exhausted position; calling next() on it immediately raises StopIteration with no further values available, so list() sees no values to collect and returns an empty list, []. This demonstrates that an iterator, unlike its source iterable, has no way to 'reset' itself back to the beginning.

# Part 5. iter() and next()

This part focuses entirely on practical, hands-on use of the two built-in functions that expose the iteration protocol directly: iter() and next(). Mastering manual iteration with these two functions, without a for loop, is what makes advanced patterns later in this handbook (generator internals, .send(), custom pipelines) feel natural rather than magical.

## 5.1 Basic Usage of iter()

In [10]:
letters = ["a", "b", "c"]
it = iter(letters)
print(it)
print(type(it))

<class 'list_iterator'>


## 5.2 Basic Usage of next()

In [11]:
letters = ["a", "b", "c"]
it = iter(letters)
print(next(it))
print(next(it))
print(next(it))

a
b
c


## 5.3 Manual Iteration Without a for Loop

Because for loops hide the mechanism, deliberately avoiding them for a moment is the clearest way to internalize how iteration actually works.

In [12]:
colors = ["red", "green", "blue"]
it = iter(colors)
print(next(it))
print(next(it))
print(next(it))
try:
    print(next(it))
except StopIteration:
    print("No more colors")

red
green
blue
No more colors


## 5.4 The Default Value Argument in next()

next(iterator, default) accepts an optional second argument: if the iterator is exhausted, next() returns default instead of raising StopIteration. This is directly analogous to dict.get(key, default), and is extremely useful for writing manual iteration code that does not need a try/except block.

In [13]:
it = iter([1, 2])
print(next(it, "no more"))
print(next(it, "no more"))
print(next(it, "no more"))   # exhausted: returns the default instead of raising
print(next(it, "no more"))   # still returns the default on every subsequent call

1
2
no more
no more


| Code / Item | Explanation |
|---|---|
| `next(it, "no more")` | Once the underlying iterator is exhausted, every further call simply returns the supplied default value, 'no more', rather than raising StopIteration. |

## 5.5 Handling StopIteration Explicitly

In [14]:
it = iter(["x"])
while True:
    try:
        value = next(it)
    except StopIteration:
        print("Iteration complete")
        break
    print("Got:", value)

Got: x
Iteration complete


## 5.6 Practical Applications of Manual Iteration

- Peeking at the first item of an iterable before deciding how to process the rest, without consuming the whole thing into a list first.
- Pulling items from two iterators in a custom, non-lockstep pattern that zip() cannot express directly.
- Implementing your own version of itertools-style helper functions.
- Writing test code or debugging tools that need to step through a generator or iterator one value at a time, inspecting state in between.

In [15]:
def peek_first(iterable):
    it = iter(iterable)
    first = next(it, None)
    return first, it   # return the peeked value AND the still-usable iterator

first, remaining = peek_first([100, 200, 300])
print("First:", first)
print("Rest:", list(remaining))

First: 100
Rest: [200, 300]


| Code / Item | Explanation |
|---|---|
| `peek_first([100, 200, 300])` | Demonstrates a genuinely useful manual-iteration pattern: inspect the first value without losing access to the rest, since the same iterator object, remaining, still holds its position immediately after the first value and can be consumed normally afterward. |

## Part 5 Practice Exercises

**Beginner:** What does next(iterator) do when the iterator has no more values, and no default argument was supplied?

*Answer:* It raises the StopIteration exception.

**Intermediate:** Rewrite 'try: value = next(it)\nexcept StopIteration: value = 'DONE'' using next()'s built-in default-value argument instead of a try/except block.

*Answer:* value = next(it, "DONE") - functionally identical, but shorter and avoids an explicit try/except block entirely.

**Advanced:** Write a function first_n(iterable, n) that manually uses iter() and next() (not slicing or itertools.islice) to return a list of the first n items of any iterable, safely handling the case where the iterable has fewer than n items.

*Answer:* def first_n(iterable, n):
    it = iter(iterable)
    result = []
    for _ in range(n):
        value = next(it, None)
        if value is None and len(result) < n and _ >= 0:
            pass
        result.append(value)
    return result
A cleaner version uses a sentinel to avoid confusing a real None value in the data with exhaustion:
def first_n(iterable, n):
    _sentinel = object()
    it = iter(iterable)
    result = []
    for _ in range(n):
        value = next(it, _sentinel)
        if value is _sentinel:
            break
        result.append(value)
    return result
Using a locally created sentinel object (guaranteed to be identical to nothing else in the data, per Part 22 of the Data Types handbook on object identity) rather than None as the default avoids incorrectly treating a genuine None item in the underlying data as if the iterator were exhausted.

# Part 6. StopIteration

This part focuses entirely on the StopIteration exception itself: what it signals, exactly when it is raised, how for loops swallow it silently, and its special, central relationship to generators, which Part 20 revisits in advanced detail.

## 6.1 What StopIteration Means

StopIteration is a plain built-in exception class, a subclass of Exception, whose sole meaning by convention is 'this iterator has no more values to produce'. It carries no special information by default (though it can optionally carry a value, which Part 20 covers for generators specifically); its type alone is the entire signal.

## 6.2 When StopIteration Occurs

It occurs at exactly one moment: when an iterator's __next__() method is called after the iterator has already produced every value it will ever produce. It is never raised proactively or early; it is raised precisely on the call that discovers there is nothing left.

In [16]:
it = iter([1])
print(next(it))       # 1: succeeds
try:
    next(it)            # nothing left: raises StopIteration HERE, not before
except StopIteration:
    print("Exhausted exactly on this call")

1
Exhausted exactly on this call


## 6.3 How for Loops Handle StopIteration

As Part 3.6 demonstrated precisely, a for loop wraps its repeated next() calls in an implicit try/except StopIteration: break. This is a completely ordinary exception being caught by completely ordinary exception-handling machinery; there is no special-cased magic in the for loop beyond that.

## 6.4 Manually Handling StopIteration

In [17]:
def safe_next(it):
    try:
        return next(it)
    except StopIteration:
        return None

it = iter([])
print(safe_next(it))

None


## 6.5 Why StopIteration Normally Isn't Visible During Ordinary for Loops

You never see a StopIteration traceback from a normal, correctly-written for loop, precisely because the for loop's own internal machinery catches it before it can propagate any further; this is by design, since letting every for loop crash with a visible traceback the moment it finished would make loops unusable. It only becomes visible when you call next() manually yourself, outside of a for loop's implicit handling, exactly as every example in Part 5 deliberately does to make the mechanism observable.

## 6.6 Relationship With Generators (Preview)

Generators (Part 9 onward) are themselves iterators: a generator function's returned generator object also raises StopIteration once its code runs to completion (falls off the end, or hits a bare return). Historically, StopIteration accidentally raised inside a generator's own body (for instance, from calling next() on an exhausted inner iterator without catching it) used to propagate outward in a confusing way; PEP 479 (active by default since Python 3.7) changed this so that any StopIteration escaping a generator's body is automatically converted into a RuntimeError instead, specifically to prevent that class of confusing, silent early-termination bug. Part 20 revisits this in full.

## Part 6 Practice Exercises

**Beginner:** What type of Python object is StopIteration?

*Answer:* It is an exception class (a subclass of the built-in Exception).

**Intermediate:** Why do you never see a StopIteration traceback when running an ordinary, correctly written for loop?

*Answer:* Because the for loop's own internal implementation wraps its repeated next() calls in an implicit try/except StopIteration block, catching the exception itself and simply ending the loop cleanly, rather than letting it propagate up and crash the program.

**Advanced:** Explain the purpose of PEP 479's change (StopIteration raised inside a generator body being converted to RuntimeError) using a concrete scenario where the old behavior would have been confusing.

*Answer:* Consider a generator function that internally loops over another iterator using next() directly (not a for loop) and forgets to catch StopIteration from that inner call: 'def gen(): inner = iter([]); yield next(inner)'. Before PEP 479, the StopIteration raised by the inner next(inner) call would propagate straight out of the generator function itself, and because a generator's own iteration machinery also uses StopIteration to signal 'this generator is done', a caller doing 'for x in gen():' would see the generator silently and prematurely stop, indistinguishable from the generator having genuinely finished successfully, hiding what was actually a bug (an unhandled inner exhaustion) as if it were normal, successful completion. PEP 479 closes this ambiguity by converting any StopIteration that escapes a generator's body into a RuntimeError instead, which is a clearly visible, unambiguous crash rather than a silently, incorrectly 'successful' early stop, making this class of bug far easier to detect.

# Part 7. Building Custom Iterators

This part teaches you to build real, working iterator classes from scratch, progressing through five increasingly realistic examples. Every line of every example is explained, so you leave this part able to write a correct custom iterator for any data source you encounter.

## 7.1 The Blueprint Every Custom Iterator Follows

- Define __init__(self, ...) to store whatever initial state and configuration the iterator needs.
- Define __iter__(self) that returns self, satisfying the iterator half of the protocol (Part 4.1).
- Define __next__(self) that computes and returns the next value, updates internal state so the following call produces something new, and raises StopIteration once a defined termination condition is reached.

## 7.2 Example 1: A Counter Iterator

In [18]:
class Counter:
    def __init__(self, start, end):
        self.current = start
        self.end = end

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.end:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

for n in Counter(1, 5):
    print(n)

1
2
3
4
5


| Code / Item | Explanation |
|---|---|
| `self.current = start` | The internal state: this is exactly the position marker Part 4.2 described, tracked on the object itself between calls. |
| `if self.current > self.end: raise StopIteration` | The termination condition: the very first thing __next__() checks, before doing anything else, is whether iteration should end. |
| `value = self.current; self.current += 1` | Captures the value to return this call, then advances internal state, so the next call produces a genuinely different value. |

## 7.3 Example 2: A Countdown Iterator

In [19]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current < 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

for n in Countdown(3):
    print(n)

3
2
1
0


> **Note:** Structurally identical to Counter, but decrementing instead of incrementing; this is a deliberate demonstration that the blueprint from 7.1 is completely general, not tied to counting upward.

## 7.4 Example 3: A Range-Like Iterator (Supporting a Step)

In [20]:
class MyRange:
    def __init__(self, start, stop, step=1):
        self.current = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if (self.step > 0 and self.current >= self.stop) or \
           (self.step < 0 and self.current <= self.stop):
            raise StopIteration
        value = self.current
        self.current += self.step
        return value

print(list(MyRange(0, 10, 2)))
print(list(MyRange(5, 0, -1)))

[0, 2, 4, 6, 8]
[5, 4, 3, 2, 1]


| Code / Item | Explanation |
|---|---|
| `(self.step > 0 and self.current >= self.stop) or (self.step < 0 and ...)` | The termination check must now account for direction: a positive step means 'stop once current reaches or passes stop', a negative step means the opposite comparison, mirroring how the built-in range() itself behaves for ascending versus descending sequences. |

## 7.5 Example 4: A Fibonacci Iterator

In [21]:
class Fibonacci:
    def __init__(self, count):
        self.count = count
        self.produced = 0
        self.a, self.b = 0, 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.produced >= self.count:
            raise StopIteration
        value = self.a
        self.a, self.b = self.b, self.a + self.b
        self.produced += 1
        return value

print(list(Fibonacci(8)))

[0, 1, 1, 2, 3, 5, 8, 13]


| Code / Item | Explanation |
|---|---|
| `self.a, self.b = 0, 1` | Initializes the two 'running' Fibonacci values that the classic iterative algorithm needs to track. |
| `self.a, self.b = self.b, self.a + self.b` | Advances both tracked values by one Fibonacci step in a single, simultaneous tuple assignment, computing the next pair from the current pair. |
| `self.produced >= self.count` | A separate counter (self.produced), distinct from the mathematical sequence itself, is what actually enforces 'produce exactly count values and then stop', since the Fibonacci sequence itself has no natural stopping point. |

## 7.6 Example 5: A Dataset Iterator (Realistic, Data-Science-Flavored)

This final example models a common real pattern: an iterator over a collection of dictionaries (a small in-memory 'dataset'), that also skips over invalid or incomplete records automatically, rather than merely marching through raw positions.

In [22]:
class ValidRecordIterator:
    def __init__(self, records):
        self.records = records
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        while self.index < len(self.records):
            record = self.records[self.index]
            self.index += 1
            if record.get("score") is not None:   # skip invalid records
                return record
        raise StopIteration

dataset = [
    {"id": 1, "score": 88},
    {"id": 2, "score": None},   # invalid: will be skipped
    {"id": 3, "score": 91},
]
for record in ValidRecordIterator(dataset):
    print(record)

{'id': 1, 'score': 88}
{'id': 3, 'score': 91}


| Code / Item | Explanation |
|---|---|
| `while self.index < len(self.records): ...` | A loop *inside* __next__() itself, used to skip past any number of invalid records before either returning a valid one or reaching the end of the underlying data and raising StopIteration; a single __next__() call may internally advance self.index by more than one position. |
| `if record.get("score") is not None: return record` | The filtering logic: only records that pass this check are ever handed back to the caller; invalid records are silently skipped rather than returned or specially flagged. |

> **Note:** This pattern, filtering inside a custom __next__(), is exactly the kind of logic that a generator function (Part 9 onward) expresses far more simply and readably; this comparison is precisely why Part 23 exists.

## Part 7 Practice Exercises

**Beginner:** In a custom iterator class, which method is responsible for actually producing each successive value?

*Answer:* __next__().

**Intermediate:** Modify the Counter class from 7.2 so that it also accepts a step argument (default 1), and demonstrate it counting from 0 to 10 in steps of 2.

*Answer:* class Counter:
    def __init__(self, start, end, step=1):
        self.current = start
        self.end = end
        self.step = step
    def __iter__(self):
        return self
    def __next__(self):
        if self.current > self.end:
            raise StopIteration
        value = self.current
        self.current += self.step
        return value
print(list(Counter(0, 10, 2)))   # [0, 2, 4, 6, 8, 10]
The only change needed is storing an additional self.step attribute and using it in place of the hard-coded += 1, exactly mirroring how MyRange in 7.4 generalizes Counter and Countdown together.

**Advanced:** Explain why ValidRecordIterator's __next__() method needs its own internal while loop, rather than simply checking the current record and immediately raising StopIteration or returning, the way Counter's __next__() does.

*Answer:* Counter's stopping condition and its value-producing condition are one and the same check: if self.current is past self.end, stop; otherwise, that exact current value is always valid and returned immediately. ValidRecordIterator has two genuinely separate concerns: whether there is more underlying data left at all (self.index < len(self.records)), and, independently, whether the specific record at the current position happens to be valid according to application logic (record.get("score") is not None). A single record being invalid does not mean iteration should stop; it only means that one particular record should be skipped and the search should continue with the next one. The internal while loop is what allows __next__() to keep advancing self.index past any number of consecutive invalid records within a single call, only returning once it either finds a genuinely valid record to hand back, or exhausts the underlying data entirely (at which point, and only then, does it raise StopIteration).

# Part 8. Iterable Classes vs Iterator Classes

Parts 2 through 7 have been building toward this part's central point: an iterable and an iterator are formally different roles, and a well-designed class often deliberately separates them into two distinct classes, precisely so that a single dataset can support multiple, simultaneous, independent iteration passes.

## 8.1 The Problem With Making a Class Both Iterable and Its Own Iterator

Every custom class in Part 7 combined both roles into one object: __iter__() returned self, and __next__() tracked state directly on that same object. This is simple, but it has a real limitation: because the position state lives directly on the one object, that object can only ever support a single, in-progress iteration pass at a time. Starting a second for loop over the same Counter instance before the first one finishes would corrupt both loops, since they would share and fight over the exact same self.current.

In [23]:
counter = Counter(1, 5)
first_two = [next(counter), next(counter)]
print(first_two)          # [1, 2]: fine so far

# Starting a fresh for loop over the SAME instance does NOT restart it:
remaining = list(counter)
print(remaining)          # [3, 4, 5]: continues from where it left off, not from 1

[1, 2]
[3, 4, 5]


## 8.2 The Proper Design: Separate Iterable and Iterator Classes

The idiomatic fix is to make the outer, user-facing class a true iterable: its __iter__() method creates and returns a brand-new, separate iterator object every single time it is called, rather than returning self. The iterator class holds the actual position state; the iterable class holds only the underlying data and configuration, and never tracks progress itself.

In [24]:
class NumberRange:                      # the ITERABLE
    def __init__(self, start, end):
        self.start = start
        self.end = end

    def __iter__(self):
        return NumberRangeIterator(self.start, self.end)   # a FRESH iterator every time


class NumberRangeIterator:              # the ITERATOR
    def __init__(self, current, end):
        self.current = current
        self.end = end

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.end:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

nr = NumberRange(1, 3)
print(list(nr))   # [1, 2, 3]
print(list(nr))   # [1, 2, 3] again: a FRESH iterator was created, so this restarts cleanly

[1, 2, 3]
[1, 2, 3]


| Code / Item | Explanation |
|---|---|
| `return NumberRangeIterator(self.start, self.end)` | The critical line: every call to iter(nr) constructs and returns a brand-new NumberRangeIterator object, with its own independent self.current starting fresh from self.start. |
| `print(list(nr)) twice, both producing [1, 2, 3]` | Direct proof that NumberRange itself never gets exhausted, unlike Counter in Part 8.1, because NumberRange never tracks position on itself at all; only its short-lived iterator objects do. |

## 8.3 Iterable vs Iterator vs Collection vs Sequence: A Complete Comparison

These four terms are related but describe increasingly specific capabilities, each one a stricter requirement than the last.

## 8.4 Why an Iterable Can Produce Multiple Independent Iterators

This is now a direct, mechanical consequence of the design in 8.2: because __iter__() constructs a new iterator object on every call, nothing is shared between separately requested iterators except, at most, read-only access to the same underlying immutable configuration (self.start, self.end). Two simultaneous 'for x in nr:' loops over the very same NumberRange instance therefore never interfere with each other, each with its own private NumberRangeIterator tracking its own private self.current.

## 8.5 Why an Iterator Itself Maintains Iteration State

The iterator's entire reason for existing, as established since Part 3, is to be the stateful half of this partnership; an iterator with no state (that could not remember where it left off) would be unable to produce a meaningfully different value on each successive call, defeating the purpose of iteration itself.

## 8.3a Iterable vs Iterator vs Collection vs Sequence

| Concept | Core Requirement | Supports Indexing? | Example |
|---|---|---|---|
| Iterable | Implements __iter__() returning an iterator | Not necessarily | set, dict, a generator object |
| Iterator | Implements __iter__() (returns self) and __next__() | No, by definition | iter([1,2,3]), an open file object |
| Collection | Iterable, and also supports len() and 'in' (per collections.abc.Collection) | Not necessarily | list, set, dict |
| Sequence | A Collection that ALSO supports ordered positional indexing and slicing | Yes | list, tuple, str, range |

## Part 8 Practice Exercises

**Beginner:** In the corrected NumberRange design, which class actually holds the self.current position counter: the iterable or the iterator?

*Answer:* The iterator class, NumberRangeIterator; the iterable class, NumberRange, only stores self.start and self.end, and never tracks progress on itself.

**Intermediate:** Explain concretely why 'list(counter); list(counter)' on a Counter instance from Part 7.2 produces an empty list the second time, while the equivalent code on a NumberRange instance from 8.2 does not.

*Answer:* Counter combines the iterable and iterator roles into a single object: its __iter__() returns self, so there is only ever one position counter, self.current, shared by every loop over that same instance; once the first list(counter) call drains it fully, the shared self.current sits past self.end permanently, so any further iteration starts from that already-exhausted state, and 'list(counter)' the second time produces []. NumberRange, in contrast, never tracks position on itself; its __iter__() method constructs a brand-new NumberRangeIterator, with its own fresh self.current, every single time it is called. Because 'list(nr)' internally calls iter(nr) once at the start, each separate 'list(nr)' call gets its own new, unexhausted iterator, so both calls independently restart from the beginning and produce the full sequence.

**Advanced:** Design an iterable class WeekdayCycle whose iterator produces the weekday names Monday through Friday, and demonstrate that two separate for loops over the same WeekdayCycle instance, interleaved (alternating iterations between them), do not interfere with each other's progress.

*Answer:* class WeekdayCycle:
    DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
    def __iter__(self):
        return iter(self.DAYS)   # reuse list's own iterator machinery for a fresh pass each time

wc = WeekdayCycle()
it1 = iter(wc)
it2 = iter(wc)
print(next(it1))   # Monday
print(next(it1))   # Tuesday
print(next(it2))   # Monday: it2 is independent of it1's progress
print(next(it1))   # Wednesday
print(next(it2))   # Tuesday
This works correctly because __iter__() calls iter(self.DAYS), which itself creates a brand-new list_iterator over the shared DAYS list on every single call, exactly the same 'fresh iterator per call' pattern as NumberRangeIterator in 8.2, just delegating the actual iterator implementation to the built-in list type instead of writing a custom iterator class by hand; it1 and it2 end up as two entirely separate list_iterator objects, each independently tracking its own position through the same underlying DAYS list without any shared mutable position state between them.

# Part 9. Generators

Every custom iterator in Part 7 required a hand-written class with explicit state variables, an __iter__() returning self, and a __next__() with a manual termination check. A generator achieves the exact same result, a fully protocol-compliant iterator, using ordinary function syntax plus one new keyword: yield. This part introduces generators from first principles and explains precisely why they exist.

## 9.1 What a Generator Is

A generator is a special kind of iterator that Python creates automatically from a generator function, a function whose body contains at least one yield statement. Calling a generator function does not run its body at all; it immediately returns a generator object, which is a real, fully-compliant iterator (it has working __iter__() and __next__() methods) that will run the function's body, lazily, only as it is asked for successive values.

## 9.2 Generator Functions

In [25]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i
        i += 1

gen = count_up_to(3)
print(gen)
print(type(gen))

<generator object count_up_to at 0x7f9130640340>
<class 'generator'>


| Code / Item | Explanation |
|---|---|
| `def count_up_to(n): ... yield i ...` | The presence of yield anywhere in this function body is what makes Python treat count_up_to as a generator function rather than an ordinary function, changing what calling it actually does. |
| `gen = count_up_to(3)` | Calling a generator function does NOT execute any of its body yet; it only creates and returns a generator object, instantly, with the function's code paused before its very first line. |

## 9.3 Generator Objects Are Iterators

In [26]:
gen = count_up_to(3)
print(hasattr(gen, "__iter__"))
print(hasattr(gen, "__next__"))
print(next(gen))
print(next(gen))
print(next(gen))
try:
    next(gen)
except StopIteration:
    print("Generator exhausted")

True
True
1
2
3
Generator exhausted


> **Note:** This proves, directly, that a generator object satisfies the exact same iteration protocol from Part 4 as any hand-written iterator class; a generator is not a separate, special mechanism outside that protocol, it is simply a much more convenient way of producing an object that follows it.

## 9.4 Lazy Execution: Nothing Runs Until You Ask

This is the single most important property of generators. The function body only executes up to the next yield statement each time next() is called on the resulting generator object; code after the final yield (or code that would run before the first yield, if any exists) never runs until it is actually reached by an ongoing sequence of next() calls.

In [27]:
def noisy_counter(n):
    print("Generator started")
    for i in range(1, n + 1):
        print(f"About to yield {i}")
        yield i
    print("Generator finished")

gen = noisy_counter(2)
print("Generator created, but nothing printed yet")
print("First next():", next(gen))
print("Second next():", next(gen))
try:
    next(gen)
except StopIteration:
    pass

Generator created, but nothing printed yet
Generator started
About to yield 1
First next(): 1
About to yield 2
Second next(): 2
Generator finished


| Code / Item | Explanation |
|---|---|
| `gen = noisy_counter(2)` | Creates the generator object; notice 'Generator started' is NOT printed by this line at all, proving the function body has not executed yet. |
| `next(gen) (first call)` | Only now does the body actually begin running, printing 'Generator started' and 'About to yield 1', then pausing exactly at the yield statement and handing back 1. |
| `next(gen) (third call, after two values already produced)` | Resumes execution after the loop ends, runs the final print('Generator finished'), then the function falls off the end, which is what triggers the automatic StopIteration. |

## 9.5 Generator State and Pausing/Resuming Execution

When a generator hits a yield statement, its entire execution state, every local variable, the current position in the code, the state of any active loops, is frozen in place and preserved on the generator object itself. The next call to next() resumes execution from exactly that frozen point, with every local variable holding exactly the value it held at the moment of pausing, as though time had simply stopped and restarted for that one function call. Part 10 and Part 11 examine this pause/resume mechanism, and what happens to local variables specifically, in full detail.

## 9.6 Memory Efficiency (Preview)

Because a generator produces only one value at a time, on demand, it never needs to hold an entire result set in memory simultaneously the way a list built by a normal function would. Part 13 demonstrates this with concrete, measured memory comparisons; this is only a preview to motivate why generators matter so much for large datasets.

## 9.7 Normal Function vs Generator Function: Direct Comparison

In [28]:
def build_list(n):        # NORMAL function: eager
    result = []
    for i in range(1, n + 1):
        result.append(i)
    return result

def build_generator(n):   # GENERATOR function: lazy
    for i in range(1, n + 1):
        yield i

eager = build_list(5)
lazy = build_generator(5)
print(eager)          # the actual list, already fully computed
print(lazy)           # a generator object; nothing computed yet
print(list(lazy))     # NOW it is fully computed, on demand

[1, 2, 3, 4, 5]


<generator object build_generator at 0x7f91303ce340>
[1, 2, 3, 4, 5]


## 9.8 Normal Function vs Generator Function

| Aspect | Normal Function | Generator Function |
|---|---|---|
| Defining keyword | return | yield (anywhere in the body) |
| What calling it produces | The fully computed return value, immediately | A generator object, immediately, with the body not yet run |
| When the body executes | Entirely, in one go, when called | Incrementally, one 'chunk' per next() call |
| Can produce values one at a time? | No; must finish and return everything at once | Yes; that is its defining purpose |
| State between calls | None; each call starts fresh | Fully preserved between yields on the same generator object |
| Memory profile for N results | All N results held in memory at once (if collected) | Only one result (plus minimal state) held at a time |

## Part 9 Practice Exercises

**Beginner:** What keyword, if present anywhere in a function's body, turns it into a generator function?

*Answer:* yield.

**Intermediate:** What does calling a generator function actually do, and what does it NOT do?

*Answer:* Calling a generator function immediately creates and returns a generator object; it does NOT execute any of the function's body at that point. Execution of the body only begins the first time next() is called on the returned generator object.

**Advanced:** Using the noisy_counter(n) example from 9.4 as a model, explain precisely what would be printed, and in what order, if you called next(gen) exactly once on gen = noisy_counter(3), and nothing else afterward.

*Answer:* Exactly one next(gen) call runs the generator body from its paused starting point up to (and including) the very first yield statement it reaches, then pauses again. For noisy_counter(3), this means: 'Generator started' prints (the print before the loop begins), then the loop starts, i becomes 1, 'About to yield 1' prints, and then execution pauses at 'yield i' and hands back the value 1 to the caller. Nothing after that point, not 'About to yield 2', not 'Generator finished', prints at all, because the generator's execution genuinely stops (is suspended) at that exact yield statement and does not resume until another next() call is made, which this scenario does not include.

# Part 10. yield

This part is devoted entirely to the yield statement itself: precisely how it differs from return, what happens to local variables when execution pauses, and how multiple yield statements (including yield inside loops) behave across successive next() calls.

## 10.1 yield vs return: The Core Difference

Table 10.1a below contrasts the two directly. The essential idea: return terminates; yield merely pauses, preserving everything so execution can pick back up exactly where it left off.

## 10.2 A Direct Side-by-Side Example

In [29]:
def get_squares_return(n):
    result = []
    for i in range(1, n + 1):
        result.append(i ** 2)
    return result          # runs once, hands back everything at once, then the call is over

def get_squares_yield(n):
    for i in range(1, n + 1):
        yield i ** 2         # pauses here, n separate times across n next() calls

print(get_squares_return(4))          # [1, 4, 9, 16]: fully computed immediately
print(list(get_squares_yield(4)))     # [1, 4, 9, 16]: same final result, produced lazily

[1, 4, 9, 16]
[1, 4, 9, 16]


## 10.3 What Happens to Local Variables When a Generator Pauses

Every local variable that exists at the moment a yield statement is reached is preserved, completely intact, as part of the generator object's internal frame state, exactly as if the entire call stack for that function had been carefully frozen rather than discarded. The next next() call resumes execution immediately after that same yield statement, with every local variable holding precisely the value it held at the pause, including loop counters, accumulators, and anything else.

In [30]:
def running_total():
    total = 0
    while True:
        amount = yield total
        if amount is not None:
            total += amount

gen = running_total()
print(next(gen))          # 0: primes the generator, runs to the first yield
print(gen.send(10))       # total becomes 10; local var 'total' was fully preserved between calls
print(gen.send(5))        # total becomes 15

0
10
15


> **Note:** This example uses .send() (fully covered in Part 21) specifically to make the point vivid: the local variable total genuinely persists, and accumulates, across multiple separate next()/send() calls, something an ordinary function could never do, since an ordinary function's local variables cease to exist the instant it returns.

## 10.4 Multiple yield Statements

In [31]:
def three_stages():
    print("Stage 1")
    yield "first"
    print("Stage 2")
    yield "second"
    print("Stage 3")
    yield "third"

for stage in three_stages():
    print("Received:", stage)

Stage 1
Received: first
Stage 2
Received: second
Stage 3
Received: third


| Code / Item | Explanation |
|---|---|
| `for stage in three_stages():` | Each iteration of this for loop corresponds to exactly one yield being reached: the generator runs 'Stage 1' then pauses at the first yield, the loop body runs with 'first', then the loop calls next() again, resuming right after that yield to run 'Stage 2' and pause at the second yield, and so on. |

## 10.5 yield Inside Loops

Placing yield inside a loop, by far the most common pattern in real generator functions, means the loop itself drives how many values the generator can produce: one yield per loop iteration that actually executes, with the loop's own state (its counter, its iterable position) forming part of what gets preserved and resumed on every pause.

In [32]:
def evens_up_to(n):
    for i in range(n + 1):
        if i % 2 == 0:
            yield i

print(list(evens_up_to(10)))

[0, 2, 4, 6, 8, 10]


| Code / Item | Explanation |
|---|---|
| `for i in range(n + 1): if i % 2 == 0: yield i` | The loop provides the sequence of candidate values; yield inside the if selectively pauses and produces only some of them, one at a time, exactly the filtering behavior that Part 7.6's ValidRecordIterator implemented far more verbosely by hand. |

## 10.1a yield vs return

| Aspect | return | yield |
|---|---|---|
| Effect on the function | Permanently ends the call, discarding all local state | Pauses the function, preserving ALL local state for resumption |
| What happens on the next call | A fresh call starts execution from the top again | next() on the SAME generator resumes exactly where it paused |
| How many times it can happen | At most once per function call | As many times as there are yield statements reached in one run |
| Turns the function into a generator? | No | Yes, its mere presence anywhere in the body does |

## Part 10 Practice Exercises

**Beginner:** Does calling return inside a function turn it into a generator function?

*Answer:* No; only yield does. A function using only return behaves as an ordinary function.

**Intermediate:** Explain, using the running_total() example, why the local variable total does not reset to 0 between successive gen.send() calls.

*Answer:* Each time running_total()'s code pauses at 'amount = yield total', the generator object preserves the function's entire local execution state, including the current value of total, exactly as it stood at that pause. When gen.send(...) resumes execution, it resumes that exact same frozen frame rather than starting a fresh call, so total continues to hold whatever value it had accumulated so far, and the subsequent 'total += amount' line adds to that preserved value rather than to a freshly-initialized total = 0, which would only happen if the function were called again from scratch.

**Advanced:** Write a generator function running_max() that, similarly to running_total() in 10.3, receives numbers via successive .send() calls and always yields the highest number seen so far. Prime it correctly and demonstrate it with three .send() calls.

*Answer:* def running_max():
    current_max = None
    value = yield current_max
    while True:
        if value is not None and (current_max is None or value > current_max):
            current_max = value
        value = yield current_max

gen = running_max()
next(gen)                 # prime: runs to the first yield, returning None
print(gen.send(5))        # 5
print(gen.send(3))        # 5 (3 is not greater than the current max)
print(gen.send(9))        # 9
This mirrors 10.3's pattern precisely: the local variable current_max is preserved across every .send() call because the generator's execution is paused, not ended, at each yield; priming with next(gen) first is required so that the very first .send(value) has an already-suspended yield expression to deliver its value into, exactly as Part 21 explains for every generator that receives values via .send().

# Part 11. Generator Execution: The Complete Lifecycle

This part consolidates everything from Parts 9 and 10 into one precise, ordered lifecycle description, with a step-by-step execution diagram traced against a concrete example, so the entire pause/resume mechanism becomes fully explicit.

## 11.1 The Eight Lifecycle Stages

- 1. The generator function is defined (the def statement runs once, creating the function object itself; no generator exists yet).
- 2. The generator function is called, which creates and returns a generator object. The function body has NOT started running.
- 3. The first next() call begins execution, running from the very top of the function body.
- 4. Execution proceeds normally (running ordinary statements, loops, conditionals) until a yield expression is reached.
- 5. yield pauses execution at that exact point, preserving all local state, and hands the yielded value back to whoever called next().
- 6. A subsequent next() call resumes execution immediately after that same yield statement, continuing with all local state intact.
- 7. Stages 4-6 repeat for as many yield statements as the function body reaches.
- 8. When the function body finally runs to completion (falls off the end, or hits a bare return), the generator automatically raises StopIteration, and is now permanently exhausted.

## 11.2 Traced Example With Full Execution Diagram

In [33]:
def lifecycle_demo():
    print("A: before first yield")
    yield 1
    print("B: after first, before second yield")
    yield 2
    print("C: after second yield, function ending")

gen = lifecycle_demo()          # Stage 2: object created, body NOT run
print("Object created")

v1 = next(gen)                  # Stage 3-5: runs 'A:...', pauses at yield 1
print("Got:", v1)

v2 = next(gen)                  # Stage 6-7: resumes after yield 1, runs 'B:...', pauses at yield 2
print("Got:", v2)

try:
    next(gen)                    # Stage 6, then 8: resumes after yield 2, runs 'C:...', falls off the end
except StopIteration:
    print("StopIteration: generator is now exhausted")

Object created
A: before first yield
Got: 1
B: after first, before second yield
Got: 2
C: after second yield, function ending
StopIteration: generator is now exhausted


| Code / Item | Explanation |
|---|---|
| `gen = lifecycle_demo()` | Stage 2 only: an object is created, but 'A: before first yield' has NOT printed yet, proving the body truly has not run. |
| `v1 = next(gen)` | Stages 3 through 5 in one call: prints 'A: ...', then pauses at the first yield, handing back 1. |
| `v2 = next(gen)` | Stages 6 and 7: resumes exactly after the first yield (not from the top), prints 'B: ...', then pauses at the second yield, handing back 2. |
| `The third next(gen) call` | Stage 6 again (resumes after the second yield), prints 'C: ...', then the function body simply ends, which is Stage 8: StopIteration is raised automatically, with no further yield to pause at. |

## 11.3 A Text-Based Execution Diagram

```
lifecycle_demo()
     |
     v
[generator object created] -- body not yet started
     |
   next(gen) #1
     v
run: 'A: before first yield'
     |
   yield 1  <--------- PAUSE, return 1 to caller
     |
   next(gen) #2 (resumes HERE, not from top)
     v
run: 'B: after first, before second yield'
     |
   yield 2  <--------- PAUSE, return 2 to caller
     |
   next(gen) #3 (resumes HERE)
     v
run: 'C: after second yield, function ending'
     |
[function body ends] -- StopIteration raised automatically
```

## Part 11 Practice Exercises

**Beginner:** At which lifecycle stage does a generator function's body actually begin executing for the first time?

*Answer:* Stage 3: the very first next() call on the generator object, not the call to the generator function itself (Stage 2).

**Intermediate:** In the lifecycle_demo() example, why does 'C: after second yield, function ending' only print once, on the third next() call, and not earlier?

*Answer:* The print statement for 'C: ...' sits, in source order, after the second yield 2 statement. Execution only reaches that line once it has already resumed past the second yield, which happens specifically on the third next() call in this example (the first two calls each pause exactly at a yield before reaching that line). Because execution genuinely pauses and does not run ahead past a yield until asked to resume, no code physically located after a not-yet-reached yield can execute before that point is reached.

**Advanced:** Predict, with justification based on the eight-stage lifecycle, exactly what happens if you call next() a fourth time on the gen object from the lifecycle_demo() example (i.e., after the StopIteration from the third call has already been raised and caught once).

*Answer:* It raises StopIteration again. Once a generator's body has run to completion (Stage 8), the generator is permanently exhausted: there is no remaining code left to resume into, no further yield statements to pause at, and no mechanism to reset the generator object back to an earlier state. A well-behaved iterator, per Part 4.1's protocol, is expected to continue raising StopIteration on every subsequent __next__() call once exhausted, rather than raising it once and then behaving unpredictably afterward, and CPython's generator implementation follows this expectation precisely.

# Part 12. Generator Expressions

A generator expression is a compact, single-line way to create a generator, using syntax that looks almost identical to a list comprehension but with parentheses instead of square brackets. This part covers their syntax, contrasts them precisely with list comprehensions, and gives clear guidance on when to reach for each.

## 12.1 Syntax

In [34]:
squares_list = [x ** 2 for x in range(5)]        # list comprehension: EAGER
squares_gen = (x ** 2 for x in range(5))         # generator expression: LAZY
print(squares_list)
print(squares_gen)
print(list(squares_gen))

[0, 1, 4, 9, 16]
<generator object <genexpr> at 0x7f913041d220>
[0, 1, 4, 9, 16]


| Code / Item | Explanation |
|---|---|
| `[x ** 2 for x in range(5)]` | Square brackets: builds the entire list immediately, holding all five results in memory at once. |
| `(x ** 2 for x in range(5))` | Parentheses: creates a generator object instantly, computing nothing yet; each value is produced only when actually requested. |
| `print(squares_gen)` | Prints something like <generator object <genexpr> at 0x...>, not the values themselves, because nothing has been computed yet. |

## 12.2 Generator Expressions vs List Comprehensions: Behavior

In [35]:
gen = (x for x in range(3))
print(sum(gen))     # 3: consumes the generator fully
print(sum(gen))     # 0: the SAME generator is now exhausted, nothing left to sum

lst = [x for x in range(3)]
print(sum(lst))     # 3
print(sum(lst))     # 3 again: a list can be reused freely, it is not 'consumed'

3
0
3
3


> **Note:** This is the single most important practical difference: a list comprehension's result can be iterated over as many times as you like; a generator expression's result can only be iterated over once, ever, because it is a genuine iterator (Part 3), and iterators do not reset.

## 12.3 Passing a Generator Expression Directly Into a Function

When a generator expression is the sole argument to a function call, the surrounding parentheses can be omitted entirely, a small but extremely common syntactic convenience.

In [36]:
numbers = [1, 2, 3, 4, 5]
total = sum(x ** 2 for x in numbers)     # no extra parentheses needed
print(total)

# equivalent, but with explicit, separate parentheses:
total2 = sum((x ** 2 for x in numbers))
print(total2)

55
55


## 12.4 Memory Usage: A Concrete Demonstration

In [37]:
import sys
list_comp = [x for x in range(100_000)]
gen_exp = (x for x in range(100_000))
print("List comprehension bytes:", sys.getsizeof(list_comp))
print("Generator expression bytes:", sys.getsizeof(gen_exp))

List comprehension bytes: 800984
Generator expression bytes: 192


> **Note:** The generator expression's reported size stays small and constant regardless of range(100_000) versus, say, range(100_000_000), because it never materializes its results; the list comprehension's size scales directly with how many elements it actually holds. Part 13 expands on this with further comparisons.

## 12.5 When to Use Each

- Use a list comprehension when you need to access elements multiple times, need random access by index, need len(), or need to reuse the same collection of results repeatedly.
- Use a generator expression when you only need to iterate over the results once, especially in a single pass to a function like sum(), max(), any(), or a for loop, and particularly when the source data is large or comes from an expensive or unbounded computation.
- As a rule of thumb: if you find yourself immediately wrapping a generator expression in list(...) right after creating it, seriously consider whether you actually needed a list comprehension in the first place instead.

## 12.6 Generator Expression vs List Comprehension

| Aspect | List Comprehension [ ] | Generator Expression ( ) |
|---|---|---|
| Evaluation | Eager: computed immediately, in full | Lazy: computed one value at a time, on demand |
| Memory usage | Proportional to the number of elements | Constant, minimal, regardless of length |
| Reusability | Can be iterated over any number of times | Can be iterated over exactly once; then exhausted |
| Supports indexing/len() | Yes | No |
| Typical use | Small-to-medium, reusable, randomly-accessed data | Large or unbounded data, single-pass pipelines |

## Part 12 Practice Exercises

**Beginner:** What is the only syntactic difference between a list comprehension and an equivalent generator expression?

*Answer:* The enclosing brackets: square brackets [ ] for a list comprehension versus parentheses ( ) for a generator expression; the inner 'for ... in ...' syntax is otherwise identical.

**Intermediate:** Given gen = (x * 2 for x in range(3)); list(gen); result = list(gen), what does result equal, and why?

*Answer:* result equals [], an empty list. The first list(gen) call fully drains the generator, advancing it past every value it will ever produce. A generator expression's underlying generator object cannot be reset or replayed, so the second list(gen) call finds nothing left to produce and returns an empty list.

**Advanced:** Explain why max(x**2 for x in range(1_000_000) if x % 7 == 0) is considered a strong, idiomatic use case for a generator expression rather than a list comprehension, referencing both memory and the number of times the result is consumed.

*Answer:* This expression is consumed exactly once, by max(), which only ever needs to scan through the candidate values a single time to find the largest one; it never needs random access, indexing, or a second pass over the results. Using a list comprehension here, [x**2 for x in range(1_000_000) if x % 7 == 0], would force Python to first construct and hold an entire intermediate list of roughly 142,857 squared values in memory, purely so that max() could then immediately throw that whole list away after one linear scan. The generator expression version produces and discards one squared value at a time as max() consumes it, so at any given moment only a single candidate value (plus minimal loop state) needs to exist in memory, regardless of whether range() were 1,000,000 or 1,000,000,000; this is precisely the 'single-pass, large-or-unbounded-source' scenario the rule of thumb in 12.5 describes.

# Part 13. Memory Efficiency

This part makes the memory case for generators completely concrete: measured comparisons between list, tuple and generator for representing the same sequence of data, and realistic large-data scenarios where the difference is the difference between a script that runs and one that crashes.

## 13.1 List, Tuple and Generator: A Measured Comparison

In [38]:
import sys
n = 1_000_000

as_list = list(range(n))
as_tuple = tuple(range(n))
as_generator = (x for x in range(n))

print("list:     ", sys.getsizeof(as_list), "bytes")
print("tuple:    ", sys.getsizeof(as_tuple), "bytes")
print("generator:", sys.getsizeof(as_generator), "bytes")

list:      8000056 bytes
tuple:     8000040 bytes
generator: 192 bytes


| Code / Item | Explanation |
|---|---|
| `sys.getsizeof(as_list)` | Scales directly with n: a list of a million integers needs memory for a million stored references, growing linearly as n grows. |
| `sys.getsizeof(as_tuple)` | Also scales with n, similarly to list, though tuples typically have marginally less per-element overhead than lists. |
| `sys.getsizeof(as_generator)` | Stays a small, fixed, constant size regardless of n, whether n is 1,000 or 1,000,000,000, because the generator object stores only its paused execution state, not any produced values. |

## 13.2 Eager vs Lazy Evaluation, Restated for Memory

Eager evaluation means computing and storing an entire result up front, all at once, before any of it is actually used; this is what a list comprehension, list(), or a normal function's return does. Lazy evaluation means deferring the computation of each value until the exact moment it is actually needed, and never holding more than the current value (plus minimal bookkeeping state) in memory at once; this is what a generator does. The memory cost of eager evaluation is proportional to the total size of the result; the memory cost of lazy evaluation is constant, independent of the total size of the result.

## 13.3 Streaming and Processing Data Incrementally

Streaming, in the context of iteration, means processing a sequence of data items one (or a small batch) at a time as they arrive or are produced, rather than waiting to collect the entire sequence first. Generators are Python's natural vehicle for streaming: a generator-based pipeline can begin producing useful output after processing only the very first item, without ever needing the full dataset to exist in memory simultaneously.

## 13.4 A Realistic Large-Data Example: Summing a Huge Sequence

In [39]:
def eager_sum_of_squares(n):
    values = [x ** 2 for x in range(n)]   # ALL n squares held in memory at once
    return sum(values)

def lazy_sum_of_squares(n):
    return sum(x ** 2 for x in range(n))  # ONE square in memory at a time

n = 2_000_000
print(eager_sum_of_squares(n))
print(lazy_sum_of_squares(n))

2666664666667000000
2666664666667000000


> **Note:** Both functions compute the identical final numeric result; only their memory profile while computing it differs. eager_sum_of_squares briefly needs enough memory to hold two million integers simultaneously; lazy_sum_of_squares never needs more than a small, constant handful of values in memory at any single moment, regardless of how large n grows.

## 13.5 When Memory Efficiency Genuinely Matters

- Processing files or datasets larger than available RAM (Part 15): a generator can process a multi-gigabyte log file in constant memory; loading it as a list cannot.
- Infinite or unbounded sequences: a generator can represent 'the next N Fibonacci numbers, for any N you eventually ask for' without ever needing a defined endpoint; a list fundamentally cannot represent an infinite sequence at all.
- Long-running production pipelines and services (Parts 16 through 18) where holding intermediate results in memory would accumulate and eventually exhaust available memory over time.
- Any situation where only a small fraction of a potentially large computed sequence will actually be consumed, since a generator only computes what is actually asked for, while an eagerly built list computes everything regardless of how much is later used.

## Part 13 Practice Exercises

**Beginner:** Between a list of one million integers and a generator that will eventually produce one million integers, which one uses a constant, small amount of memory regardless of the million?

*Answer:* The generator; its memory footprint does not scale with how many values it will eventually produce, only the list's does.

**Intermediate:** Explain, in terms of eager vs lazy evaluation, why eager_sum_of_squares(n) from 13.4 could raise a MemoryError for a sufficiently large n, while lazy_sum_of_squares(n) would not, for the exact same n.

*Answer:* eager_sum_of_squares first builds a complete list of n squared values via a list comprehension before summing it, meaning all n values must simultaneously exist in memory at the point the sum() call runs; for a sufficiently large n, the memory required to hold that entire list can exceed what is available, raising MemoryError. lazy_sum_of_squares instead passes a generator expression directly to sum(), so squares are computed and immediately consumed (added to a running total, then discarded) one at a time; at any given moment, only one squared value (plus the running total and minimal loop state) needs to exist in memory, so the memory required does not grow with n at all, allowing it to complete successfully for values of n that would exhaust memory in the eager version.

**Advanced:** A colleague argues that since both eager_sum_of_squares(n) and lazy_sum_of_squares(n) ultimately compute and return the exact same numeric answer, there is no meaningful difference between them, and picking either is purely a matter of style. Construct a rebuttal grounded in the distinction between eager and lazy evaluation.

*Answer:* The two functions are equivalent only in their final output, not in their resource behavior while producing it, and for sufficiently large n that distinction is the difference between a program that completes and one that crashes. eager_sum_of_squares performs eager evaluation: it fully materializes an intermediate list of n squared integers before summing it, so its peak memory usage is directly proportional to n, and for a large enough n it will exhaust available memory and raise MemoryError regardless of how fast the machine's processor is. lazy_sum_of_squares performs lazy evaluation via a generator expression: it computes each squared value only at the exact moment sum() is ready to add it to the running total, immediately discards that value once consumed, and never holds more than a small, constant amount of data in memory at once, so its peak memory usage is independent of n entirely. This is not a stylistic preference; it is a fundamental difference in space complexity, O(n) for the eager version versus O(1) for the lazy version, and for any n large enough to matter, that difference determines whether the program can run at all, not merely how it looks in source code.

# Part 14. Lazy Evaluation

Lazy evaluation has been mentioned throughout Parts 9 through 13 as the underlying principle that makes generators memory-efficient. This part treats it as a first-class topic in its own right: a precise definition, a contrast with eager evaluation, concrete benefits and genuine limitations, and performance considerations that go beyond memory alone.

## 14.1 What Lazy Evaluation Means, Precisely

Lazy evaluation is a strategy where a computation is deferred (not performed) until the exact moment its result is actually required, rather than being performed proactively as soon as it becomes possible. In Python, generators, generator expressions, range, map(), filter(), zip(), and dictionary views (Part 30) are all lazy or lazy-like in this sense: creating one of them does little or no actual work; the real computation happens incrementally, only as values are actually pulled out.

## 14.2 Eager Evaluation, Contrasted Directly

Eager evaluation performs a computation as soon as it is triggered, in full, whether or not the entire result will ever actually be used. A list comprehension, str.split(), and an ordinary function call with a return statement are all eager: the moment they run, all of their work is done and their entire result exists, ready or not.

In [40]:
def eager_double(numbers):
    return [n * 2 for n in numbers]   # computes EVERY doubled value immediately

def lazy_double(numbers):
    return (n * 2 for n in numbers)   # computes NOTHING until values are pulled

result_eager = eager_double([1, 2, 3])     # all three doublings happen right here
result_lazy = lazy_double([1, 2, 3])       # nothing has been doubled yet
print(result_eager)
print(next(result_lazy))                    # the FIRST doubling happens only now, on demand

[2, 4, 6]
2


## 14.3 Benefits of Lazy Evaluation

- Constant, minimal memory usage for arbitrarily large or unbounded sequences (Part 13).
- The ability to represent genuinely infinite sequences meaningfully, such as 'all natural numbers' or 'an endless stream of sensor readings', which cannot be represented as any kind of eagerly-built collection at all.
- Avoided wasted computation: if only the first few results of a lazy sequence are ever actually consumed (for example, via itertools.islice, Part 24), the remaining values are simply never computed at all, saving both time and memory.
- Naturally composable pipelines: several lazy operations (a generator expression feeding into filter(), feeding into map()) can be chained together, and the entire pipeline still only does as much work as the final consumer actually asks for, one item at a time.

## 14.4 Limitations of Lazy Evaluation

- A lazy sequence generally cannot be indexed directly (no gen[5]) and has no defined len() without being consumed first, since it may not even know its own length in advance, or may be infinite.
- A lazy sequence can typically be consumed only once; once exhausted, it cannot be replayed or reset (Part 12.2), unlike a list which can be iterated over repeatedly.
- Errors inside a lazy computation are deferred too: a bug in the logic that produces value number 999 will not surface until value 999 is actually requested, potentially far away in the code, and far later in time, from where the lazy sequence was originally defined, which can make debugging less immediately obvious than with eager code that fails immediately.
- For genuinely small, fixed-size datasets that will be accessed repeatedly, the overhead of generator machinery (Part 33 measures this precisely) can make lazy evaluation slightly slower in wall-clock time than simply computing everything eagerly once and reusing the result.

## 14.5 Performance Considerations Beyond Memory

It is a common misconception that 'lazy' automatically means 'faster'. Lazy evaluation's primary, guaranteed advantage is memory efficiency and the ability to handle unbounded data; it does not guarantee a speed advantage, and for small, fully-consumed datasets, it can sometimes be marginally slower due to the per-value overhead of repeatedly pausing and resuming generator frames, compared to a tight, eagerly-computed loop. Part 33 is devoted entirely to measuring and clarifying exactly this trade-off with real benchmarks.

## Part 14 Practice Exercises

**Beginner:** In one sentence, what does lazy evaluation mean?

*Answer:* Deferring a computation until the exact moment its result is actually needed, rather than performing it proactively as soon as possible.

**Intermediate:** Name one genuine limitation of lazy evaluation that has nothing to do with memory usage.

*Answer:* Any of: a lazy sequence generally cannot be indexed or measured with len() without first being consumed; it can typically only be iterated over once; errors inside it are deferred until the relevant value is actually requested, which can make debugging harder; or, for small fully-consumed datasets, its per-value overhead can make it marginally slower than an eager equivalent.

**Advanced:** A junior developer claims 'I switched this function from a list comprehension to a generator expression, so it must now be faster.' Explain why this claim is not necessarily true, and describe the one scenario where switching to a generator expression is guaranteed to help, regardless of raw speed.

*Answer:* Lazy evaluation's defining, guaranteed benefit is memory efficiency (bounded, constant memory usage) and the ability to represent unbounded sequences, not raw execution speed; a generator expression that is fully consumed exactly once, in full, can in fact run marginally slower in wall-clock time than an equivalent list comprehension, because each value produced by a generator involves pausing and resuming a generator frame, which carries a small but real per-value overhead compared to a tight, straightforward loop building a list directly. The claim would only be reliably true, in terms of memory rather than raw speed, in the scenario where either the full sequence is very large (or unbounded) and only used in a single pass, or where only a small portion of a potentially large sequence is ever actually consumed (for instance, via early termination with break, or itertools.islice); in either of those cases, the generator's incremental, on-demand computation avoids constructing (or even beginning to construct) the full, unused result, which the list comprehension version could not avoid.

# Part 15. Generators and Files

This part shows generators solving a genuinely common, practical problem: processing files (plain text, logs, CSV, JSON-lines) that may be far larger than available memory, one record at a time, without ever loading the entire file at once. Note: opening and reading files themselves is covered in the companion File Handling handbook; this part focuses specifically on the iteration/generator layer built on top of file objects.

## 15.1 The Problem: Loading an Entire File at Once

In [41]:
def count_long_lines_eager(path):
    with open(path) as f:
        lines = f.readlines()          # loads the ENTIRE file into memory as a list
    return sum(1 for line in lines if len(line) > 80)

> **Note:** For a small file this is completely fine; for a multi-gigabyte log file, f.readlines() attempts to hold every single line in memory simultaneously as a list, which can exhaust available memory or simply be needlessly wasteful when only a running count is actually needed.

## 15.2 The Generator-Based Fix: Process One Line at a Time

In [42]:
def count_long_lines_lazy(path):
    with open(path) as f:
        return sum(1 for line in f if len(line) > 80)   # f itself is iterated lazily, line by line

| Code / Item | Explanation |
|---|---|
| `sum(1 for line in f if len(line) > 80)` | Iterates directly over the open file object f, which (Part 31 explains why) yields one line at a time on demand; at most one line's worth of text needs to exist in memory at any given moment, regardless of the file's total size. |

## 15.3 A Generator Function That Filters and Transforms Log Lines

In [43]:
def parse_error_lines(path):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if "ERROR" in line:
                yield line

# Usage (assuming a real log file path):
# for error_line in parse_error_lines("app.log"):
#     print(error_line)
print("parse_error_lines is itself a generator function; calling it creates a generator instantly.")

parse_error_lines is itself a generator function; calling it creates a generator instantly.


> **Note:** This generator never holds the whole log file, or even the whole list of error lines, in memory at once; each qualifying line is produced, consumed by whatever calls next() on it (directly or via a for loop), and then can be discarded before the next line is even read from disk.

## 15.4 Streaming a CSV File Record by Record (Without the csv Module)

For illustration, the following builds a minimal, generator-based CSV row parser using only string splitting; in real production code you would use the standard library's csv module (covered in the File Handling handbook), which itself is iterable and yields rows lazily, following exactly this same underlying pattern.

In [44]:
def read_csv_rows(path):
    with open(path) as f:
        header = next(f).strip().split(",")
        for line in f:
            values = line.strip().split(",")
            yield dict(zip(header, values))

# Usage:
# for row in read_csv_rows("sales.csv"):
#     print(row["amount"])
print("read_csv_rows yields one dict per row, never the whole file as a list of dicts.")

read_csv_rows yields one dict per row, never the whole file as a list of dicts.


| Code / Item | Explanation |
|---|---|
| `header = next(f).strip().split(",")` | Manually pulls just the very first line from the file (the header row) using next() directly on the file object, before the for loop begins consuming the remaining lines. |
| `yield dict(zip(header, values))` | Combines the header names with each row's values into a dict, and yields it immediately, one row at a time, rather than accumulating a list of all rows. |

## 15.5 Streaming JSON Lines (One JSON Record Per Line)

A JSON Lines (.jsonl) file, a common format for large datasets and logs, stores one complete, independent JSON object per line, specifically so that it CAN be processed one record at a time without ever needing to parse the file as a single giant JSON document.

In [45]:
import json

def read_jsonl(path):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

print("Each call to next() on read_jsonl(path) parses and returns exactly one JSON record.")

Each call to next() on read_jsonl(path) parses and returns exactly one JSON record.


## 15.6 Large-File Processing: Putting It Together

In [46]:
def average_of_field(path, field):
    total = 0
    count = 0
    for record in read_jsonl(path):     # streams the file, one JSON object at a time
        if field in record:
            total += record[field]
            count += 1
    return total / count if count else None

print("average_of_field can process a multi-gigabyte .jsonl file in constant memory.")

average_of_field can process a multi-gigabyte .jsonl file in constant memory.


## Part 15 Practice Exercises

**Beginner:** Why does 'for line in f:' (where f is an open file object) avoid loading the entire file into memory at once, unlike f.readlines()?

*Answer:* An open file object is itself iterable in a lazy way: each iteration reads and yields just the next single line from disk, discarding it once the loop body has processed it, rather than reading and storing every line up front the way f.readlines() explicitly does by building a complete list of every line.

**Intermediate:** Rewrite parse_error_lines(path) from 15.3 so that it also yields a running line-number alongside each matching error line, as a tuple (line_number, line_text), using enumerate() (Part 27 covers enumerate() in full).

*Answer:* def parse_error_lines_numbered(path):
    with open(path) as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if "ERROR" in line:
                yield (line_number, line)
Wrapping the file object in enumerate(f, start=1) preserves the exact same one-line-at-a-time, generator-based streaming behavior, while additionally tracking a 1-based line counter alongside each line, with no extra memory cost since enumerate() itself is lazy (Part 27).

**Advanced:** Explain why read_csv_rows(path) from 15.4 calls next(f) once, manually, before starting its for loop, rather than simply checking 'if this is the first row' inside the loop body on every iteration.

*Answer:* Calling next(f) once before the loop consumes exactly the first line of the file (the header row) immediately and unconditionally, advancing the file object's own internal read position past it, so that the subsequent 'for line in f:' loop begins its very first iteration already positioned at the second line of the file, the first actual data row. This is more efficient and clearer than checking a condition like 'if line_number == 1' inside the loop body on every single iteration for the entire rest of the file, a check that would need to run needlessly on every remaining data row just to handle one specific line at the very start; separating the one-time header extraction from the per-row processing loop keeps each part of the function doing exactly one clear job.

# Part 16. Generators and Data Science

This part applies generators to practical Data Science workflows: preprocessing pipelines, batch processing, streaming transformations, and a brief, honest look at how this relates to pandas, without turning into a pandas course.

## 16.1 Generator-Based Data Preprocessing Pipelines

A very common, powerful pattern is to chain several small generator functions together, each performing one transformation step, so that a single record flows through the entire pipeline, one step at a time, before the next record even begins. This mirrors the Unix pipe philosophy directly.

In [47]:
def raw_records(n):
    for i in range(n):
        yield {"id": i, "value": i * 1.5, "valid": i % 5 != 0}

def filter_valid(records):
    for r in records:
        if r["valid"]:
            yield r

def add_normalized(records, max_value):
    for r in records:
        r["normalized"] = r["value"] / max_value
        yield r

pipeline = add_normalized(filter_valid(raw_records(10)), max_value=15)
for record in pipeline:
    print(record)

{'id': 1, 'value': 1.5, 'valid': True, 'normalized': 0.1}
{'id': 2, 'value': 3.0, 'valid': True, 'normalized': 0.2}
{'id': 3, 'value': 4.5, 'valid': True, 'normalized': 0.3}
{'id': 4, 'value': 6.0, 'valid': True, 'normalized': 0.4}
{'id': 6, 'value': 9.0, 'valid': True, 'normalized': 0.6}
{'id': 7, 'value': 10.5, 'valid': True, 'normalized': 0.7}
{'id': 8, 'value': 12.0, 'valid': True, 'normalized': 0.8}
{'id': 9, 'value': 13.5, 'valid': True, 'normalized': 0.9}


| Code / Item | Explanation |
|---|---|
| `pipeline = add_normalized(filter_valid(raw_records(10)), max_value=15)` | Chains three generator functions together; critically, NONE of them run any actual code yet at this line, exactly per the lazy-creation rule from Part 9.4. |
| `for record in pipeline:` | This is the only point where any work actually happens: each iteration pulls one record through the entire chain, from raw_records() through filter_valid() through add_normalized(), one record at a time, rather than materializing an intermediate list at each stage. |

## 16.2 Batch Processing With a Generator

It is extremely common in data pipelines to need fixed-size batches (chunks) of records rather than one record at a time, for example to insert 1,000 rows into a database per query instead of one row per query.

In [48]:
def batched(iterable, batch_size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch   # yield any leftover, smaller final batch

for batch in batched(range(10), 3):
    print(batch)

[0, 1, 2]
[3, 4, 5]
[6, 7, 8]
[9]


| Code / Item | Explanation |
|---|---|
| `if len(batch) == batch_size: yield batch; batch = []` | Yields a full batch the instant it reaches the target size, then immediately resets the accumulator so the next batch can start collecting fresh. |
| `if batch: yield batch (after the loop)` | Handles the final, possibly-smaller leftover batch (here, [9], since 10 does not divide evenly by 3), which would otherwise be silently dropped if this check were omitted. |

> **Note:** Since Python 3.12, itertools.batched() provides this exact behavior as a built-in; it is implemented here from scratch specifically to make the underlying mechanism explicit.

## 16.3 Streaming Data Transformations at Scale

In [49]:
def clean_and_convert(records):
    for r in records:
        cleaned = {k: (v.strip() if isinstance(v, str) else v) for k, v in r.items()}
        yield cleaned

raw = [{"name": "  Ada ", "score": 90}, {"name": " Bo", "score": 85}]
for cleaned in clean_and_convert(raw):
    print(cleaned)

{'name': 'Ada', 'score': 90}
{'name': 'Bo', 'score': 85}


## 16.4 ETL Pipelines: Extract, Transform, Load, as Generators

A classic ETL pipeline maps naturally onto three chained generator functions: extract() yields raw records from a source (a file, an API, a database cursor), transform() yields cleaned/reshaped records, and load() consumes the final stream, writing each record to its destination as it arrives, rather than ever holding the full dataset in memory across all three stages simultaneously.

In [50]:
def extract(n):
    for i in range(n):
        yield {"id": i, "raw_value": str(i * 10)}

def transform(records):
    for r in records:
        yield {"id": r["id"], "value": int(r["raw_value"])}

def load(records):
    loaded_count = 0
    for r in records:
        # in real code: write r to a database, warehouse, or file here
        loaded_count += 1
    return loaded_count

total_loaded = load(transform(extract(1000)))
print("Records loaded:", total_loaded)

Records loaded: 1000


## 16.5 Memory-Efficient Transformations: Realistic Example

In [51]:
def large_dataset(n):
    for i in range(n):
        yield {"id": i, "amount": (i % 97) * 3.3}

def total_amount_above(records, threshold):
    return sum(r["amount"] for r in records if r["amount"] > threshold)

print(total_amount_above(large_dataset(1_000_000), threshold=250))

61439578.199999996


> **Note:** This computes an aggregate over a simulated one-million-record dataset while never holding more than one record (plus a running total) in memory at any point; the exact same code pattern scales identically whether large_dataset(n) yields from an in-memory range, a database cursor, or a streamed file, per Parts 15 and 24.

## 16.6 Brief Relationship to pandas

pandas' read_csv() (and similar readers) support a chunksize parameter, which turns the return value into an iterator that yields successive DataFrame chunks lazily, rather than loading the entire file into one giant DataFrame at once; this is pandas' own built-in application of exactly the generator/lazy-evaluation principles this handbook covers, applied specifically to tabular data too large to fit comfortably in memory as a single DataFrame. This handbook stops at that connection deliberately; pandas' own chunked-processing API deserves, and typically has, its own dedicated material.

## Part 16 Practice Exercises

**Beginner:** In the batched() generator from 16.2, what happens to any leftover items that don't fill a complete, full-sized batch?

*Answer:* They are collected into a final, smaller batch and yielded once, after the main loop finishes, via the 'if batch: yield batch' check; without that check, a leftover partial batch would be silently discarded.

**Intermediate:** In the ETL pipeline example (16.4), explain why calling extract(1000) does not, by itself, actually read or produce any of the 1000 records yet.

*Answer:* extract is a generator function; per Part 9.4, calling a generator function only creates a generator object with its body paused before the first line, it does not begin executing that body. The actual work of producing each record only happens once something downstream, ultimately the for loop inside load(), starts requesting values via next(), pulling exactly one record through extract(), then transform(), at a time.

**Advanced:** Explain, referencing both 16.1's pipeline pattern and Part 13's memory analysis, why chaining several generator functions together (extract -> transform -> load) uses dramatically less peak memory than an equivalent pipeline written with ordinary functions that each return a fully-built list.

*Answer:* In the generator-chained version, calling extract(n), then wrapping it in transform(...), then load(...), builds a chain of paused generator objects with no data flowing through any of them yet; the moment the for loop inside load() begins, it requests one value from transform(), which in turn requests one value from extract(), which produces exactly one raw record, which is transformed, then loaded, and then that single record can be discarded (garbage collected) before the next one is ever produced, so at any instant only a small, constant number of records exist in memory across the entire three-stage pipeline, regardless of n. If each stage were instead an ordinary function returning a fully-built list, extract(n) would first construct and return a complete list of n raw records held entirely in memory; transform() would then need to hold both that entire input list AND build a complete second list of n transformed records simultaneously; and load() would need the complete transformed list available before processing any of it. Peak memory in that eager version would be proportional to n (and, briefly, to multiple full copies of the dataset existing at once during the handoff between stages), while the generator-chained version's peak memory stays constant regardless of how large n grows, exactly mirroring the eager-vs-lazy memory analysis from Part 13.4.

# Part 17. Generators and Machine Learning

Machine learning training pipelines are, at their core, iteration: repeatedly pulling batches of examples and feeding them to a model. This part explains how generators map directly onto batch generation, training pipelines, streaming datasets, and data augmentation, and how this connects conceptually to real ML data-loading tools.

## 17.1 Batch Generation for Training

Training a model rarely processes one example, or the entire dataset, at a time; it processes mini-batches, a fixed number of examples per training step. A generator is a natural fit for producing these mini-batches without ever needing every example loaded into memory simultaneously.

In [52]:
def mini_batches(features, labels, batch_size):
    n = len(features)
    for start in range(0, n, batch_size):
        end = start + batch_size
        yield features[start:end], labels[start:end]

features = list(range(10))
labels = [x % 2 for x in features]
for batch_x, batch_y in mini_batches(features, labels, batch_size=4):
    print(batch_x, batch_y)

[0, 1, 2, 3] [0, 1, 0, 1]
[4, 5, 6, 7] [0, 1, 0, 1]
[8, 9] [0, 1]


## 17.2 An Infinite Training Data Generator (Epochs)

A training loop typically needs to cycle through the dataset repeatedly, once per epoch, for many epochs. A generator can express 'repeat this dataset forever, in mini-batches' directly, which a finite list fundamentally cannot represent on its own.

In [53]:
import itertools

def infinite_batches(data, batch_size):
    while True:                              # loop forever: a new epoch each time around
        for start in range(0, len(data), batch_size):
            yield data[start:start + batch_size]

data = list(range(6))
gen = infinite_batches(data, batch_size=4)
for _ in range(4):        # simulate pulling 4 batches from an unending stream
    print(next(gen))

[0, 1, 2, 3]
[4, 5]
[0, 1, 2, 3]
[4, 5]


> **Note:** This is precisely how many real training loops work: 'for step in range(total_training_steps): batch = next(training_generator)', pulling exactly as many batches as needed, regardless of how many total steps that training run happens to require, without ever pre-computing a fixed, finite list of every batch in advance.

## 17.3 Streaming Datasets Too Large to Fit in Memory

Combining the file-streaming techniques from Part 15 with the batching pattern from 17.1 directly yields a pipeline that can train on a dataset far larger than available RAM, since only one batch's worth of data needs to exist in memory at any given moment.

In [54]:
def stream_training_examples(records_generator, batch_size):
    batch = []
    for record in records_generator:
        batch.append(record)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

# Usage sketch (combining with Part 15's read_jsonl):
# for batch in stream_training_examples(read_jsonl("huge_dataset.jsonl"), batch_size=32):
#     train_step(batch)
print("stream_training_examples never holds more than one batch worth of data in memory.")

stream_training_examples never holds more than one batch worth of data in memory.


## 17.4 Data Augmentation as a Generator

Data augmentation, producing modified variants of training examples (rotations, noise, synonym substitution) on the fly, is naturally expressed as a generator wrapping another generator: each incoming example can yield one or more augmented variants, without ever needing to precompute and store every augmented variant of the entire dataset ahead of time.

In [55]:
import random
random.seed(0)

def augment(records):
    for r in records:
        yield r                                          # the original
        yield {**r, "value": r["value"] * random.uniform(0.9, 1.1)}   # one noisy variant

original = [{"id": 1, "value": 10.0}, {"id": 2, "value": 20.0}]
for example in augment(original):
    print(example)

{'id': 1, 'value': 10.0}
{'id': 1, 'value': 10.688843703050097}
{'id': 2, 'value': 20.0}
{'id': 2, 'value': 21.03181761176121}


| Code / Item | Explanation |
|---|---|
| `yield r; yield {**r, ...}` | A single generator function producing MORE than one output value per input value is a perfectly normal, common pattern; each incoming record here results in two separate yielded values before the loop moves to the next incoming record. |

## 17.5 Memory-Efficient Preprocessing Pipelines for ML

Exactly the ETL pattern from Part 16.4, chained generator functions for extract/transform/load, applies directly to ML preprocessing: tokenization, normalization, feature extraction and augmentation can each be one generator stage in a chain, feeding a final batching stage, feeding the training loop, with the entire pipeline operating in bounded, constant memory regardless of total dataset size.

## 17.6 Relationship to Real ML Data Loaders

Frameworks such as PyTorch's DataLoader (built on a Dataset or IterableDataset) and TensorFlow's tf.data.Dataset are, conceptually, production-grade, heavily optimized implementations of exactly the ideas in this part: lazy, batched, potentially-shuffled, potentially-parallelized iteration over training data, often built internally using Python generators or objects that follow the same iterator protocol from Part 4. Understanding plain Python generators, as covered throughout this handbook, is directly transferable background for understanding how those framework-level data loaders behave and why they are designed the way they are; this handbook deliberately stops short of framework-specific API details, which belong in dedicated ML framework material.

## Part 17 Practice Exercises

**Beginner:** Why is a generator, rather than a pre-built list, a natural fit for representing 'an endless stream of training batches, one per epoch, forever'?

*Answer:* A list is inherently finite; it cannot represent an unbounded, ongoing sequence. A generator using 'while True:' can represent exactly that: it produces batches indefinitely, on demand, computing each new batch (or epoch's worth of batches) only when the training loop actually asks for the next one, with no upper bound baked in.

**Intermediate:** In mini_batches() from 17.1, explain what happens on the final iteration if len(features) is not evenly divisible by batch_size, using range(0, n, batch_size) and slicing.

*Answer:* Python slicing never raises an error for an end index that runs past the end of a sequence; it simply returns whatever elements remain, which may be fewer than requested. So on the final iteration of the range(0, n, batch_size) loop, 'features[start:end]' and 'labels[start:end]' simply return a shorter, final partial batch containing whatever examples are left, rather than raising an IndexError or silently dropping the leftover examples; every example in the original dataset therefore still ends up included in exactly one batch, even though the very last batch may be smaller than batch_size.

**Advanced:** Explain, in terms of memory and lazy evaluation, why combining Part 15's file-streaming generators with Part 17's batching generators (as sketched in 17.3) allows training on a dataset that is larger than the machine's available RAM, while precomputing 'all_batches = list(stream_training_examples(...))' first would not.

*Answer:* read_jsonl() (Part 15) is itself a generator that reads and parses exactly one JSON record from disk at a time, yielding it immediately and never holding more than that single record (plus the open file handle's own small internal buffer) in memory. stream_training_examples() (17.3) consumes that generator lazily too, accumulating records only until it has collected exactly one batch's worth, at which point it yields that one batch and discards its accumulator, so at any given moment across the entire combined pipeline, at most one batch's worth of records needs to exist in memory, a bounded, constant amount regardless of how many total records the underlying file contains. Wrapping the whole thing in 'list(...)', however, forces every single batch produced by the entire pipeline to be collected and held simultaneously in one big list before training even starts, which requires memory proportional to the ENTIRE dataset's size all at once, precisely defeating the purpose of the streaming, batch-by-batch design and reintroducing the exact same out-of-memory risk that motivated using generators for this in the first place.

# Part 18. Generators and AI Engineering

This part covers how generators show up specifically in AI engineering work: streaming model output, processing large documents for retrieval pipelines, and general event/batch processing patterns, while being careful to draw a clear, technically precise line between a Python generator (an in-process language construct) and application-level network streaming (bytes moving between processes or machines over a connection).

## 18.1 Streaming AI Responses: The Conceptual Pattern

Many AI systems (chat completions, text generation) produce output incrementally, token by token or chunk by chunk, rather than waiting to generate an entire response before returning anything. Inside a Python client or server handling this, a generator is a natural way to model 'here is the next chunk of the response, as it becomes available', letting calling code start processing (or displaying) partial output immediately rather than waiting for the entire response to finish generating.

In [56]:
import time

def simulated_token_stream(tokens):
    for token in tokens:
        # In a real system, this line would represent waiting for the
        # next token to actually arrive from a model or network connection.
        yield token

for token in simulated_token_stream(["The", " quick", " brown", " fox"]):
    print(token, end="", flush=True)
print()

The

 quick

 brown

 fox

## 18.2 Python Generators vs Application-Level Network Streaming: A Critical Distinction

It is essential to be precise here: a Python generator, by itself, is purely an in-process control-flow construct; it pauses and resumes a single function's execution within one running Python process, and produces ordinary Python objects. Network streaming (such as HTTP chunked transfer encoding, or a WebSocket connection) is an entirely different, lower-level mechanism: bytes being sent incrementally across a network connection between two separate processes, potentially on two separate machines, governed by network protocols that have nothing to do with Python's yield keyword.

The two concepts connect, but do not collapse into each other, in a very common and important pattern: a network client library often wraps an incoming stream of network bytes/events with a Python generator (or an async generator, Part 32) purely as a convenient, Pythonic interface, so that application code can simply write 'for chunk in response.iter_content():' or 'async for event in stream:' and let the library handle the actual underlying network mechanics, while the calling code only ever interacts with ordinary Python iteration syntax.

| Code / Item | Explanation |
|---|---|
| `Python generator (yield)` | In-process pause/resume of one function's execution; produces Python objects; no networking involved by itself. |
| `Network streaming (HTTP chunked, WebSockets, SSE)` | Bytes sent incrementally across a network connection between separate processes/machines; governed by network/transport protocols, independent of any specific programming language. |

## 18.3 Large Document Processing for Retrieval Pipelines

Retrieval-augmented systems commonly need to split large documents into smaller chunks (for embedding and indexing) without loading and holding an entire corpus of documents in memory simultaneously. A generator-based chunking pipeline, combining ideas directly from Part 15 (file streaming) and Part 16 (transformation pipelines), is a natural fit.

In [57]:
def chunk_text(text, chunk_size, overlap=0):
    start = 0
    while start < len(text):
        end = start + chunk_size
        yield text[start:end]
        start += chunk_size - overlap

def chunk_documents(documents, chunk_size):
    for doc_id, text in documents:
        for chunk in chunk_text(text, chunk_size):
            yield {"doc_id": doc_id, "chunk": chunk}

docs = [(1, "a" * 25), (2, "b" * 15)]
for piece in chunk_documents(docs, chunk_size=10):
    print(piece["doc_id"], len(piece["chunk"]))

1 10
1 10
1 5
2 10
2 5


> **Note:** chunk_documents() nests one generator function inside another (an inner 'for chunk in chunk_text(...)' loop inside an outer generator function); this is a routine, idiomatic pattern, and Part 19's yield from provides an even more direct way to express exactly this kind of delegation.

## 18.4 Batch Processing and Event Processing

The same batching pattern from Part 16.2 and Part 17.1 applies directly to AI engineering workloads: batching embedding-API requests, batching records for a vector database upsert, or processing a stream of incoming events (webhook calls, queue messages) one at a time or in small batches, all follow the identical generator-based accumulate-and-yield pattern already established.

## 18.5 Summary: Where Generators Fit in an AI Engineering Stack

- Inside a single process: generators are the natural tool for lazily processing large documents, batching records, and building memory-efficient preprocessing/retrieval pipelines.
- At a process/network boundary: an async generator (Part 32) is commonly used as the Pythonic wrapper AROUND an underlying network stream, not as a replacement for the network mechanism itself.
- The core discipline (lazy, one-item-at-a-time processing, bounded memory) is identical in both cases; what differs is simply whether the 'next item' is computed locally in-process or arrives from across a network connection.

## Part 18 Practice Exercises

**Beginner:** Is a Python generator, by itself, a networking mechanism?

*Answer:* No. A Python generator is purely an in-process construct for pausing and resuming a function's execution; it produces ordinary Python objects and has no inherent connection to sending data across a network.

**Intermediate:** In chunk_text(text, chunk_size, overlap), explain what the overlap parameter does to the boundaries between successive yielded chunks.

*Answer:* Each iteration advances start by 'chunk_size - overlap' instead of by the full chunk_size. When overlap is greater than zero, this means the starting position of each new chunk moves forward by less than one full chunk's width, so the final 'overlap' characters of one chunk are also included again as the first characters of the next chunk, a common technique in document-chunking for retrieval systems to avoid losing context that happens to fall exactly on a chunk boundary.

**Advanced:** A teammate says 'we're using an async generator to stream the AI model's response, so our system is now doing network streaming.' Using the distinction from 18.2, explain precisely what is and is not accurate about this statement.

*Answer:* It is accurate that an async generator is very commonly used as the Python-level interface for consuming a streamed response, and if the underlying response genuinely arrives incrementally over a network connection (such as HTTP chunked transfer or Server-Sent Events from an external AI API), then yes, actual network streaming is occurring at the transport layer. What would be inaccurate is any implication that the async generator itself IS the network streaming mechanism, or that simply using 'async def' with yield automatically causes data to be sent incrementally over a network; the async generator is only the in-process Python construct that pauses and resumes to hand out each already-arrived piece of data to the rest of the application, in a convenient, iterable form. The actual incremental delivery of bytes across the network is handled by an entirely separate layer, typically an HTTP client library or socket-level code sitting underneath the async generator, and if that underlying transport were not actually streaming (for example, if the client library secretly buffered the entire response before ever handing any of it to the async generator), wrapping the result in an async generator afterward would not retroactively make the network transfer itself incremental; the two concepts remain distinct even when they are used together correctly.

# Part 19. yield from

yield from is a dedicated syntax for delegating part (or all) of a generator's work to another iterable or generator, automatically forwarding every value it produces. This part builds up from the simplest possible case to genuinely nested generator composition.

## 19.1 The Problem yield from Solves

In [58]:
def without_yield_from():
    for x in [1, 2, 3]:
        yield x            # manually re-yielding every value from another iterable

def with_yield_from():
    yield from [1, 2, 3]    # identical result, more direct

print(list(without_yield_from()))
print(list(with_yield_from()))

[1, 2, 3]


[1, 2, 3]


## 19.2 Delegating to Another Iterable

yield from accepts any iterable, not only another generator: a list, a tuple, a string, a range, or a file object all work directly.

In [59]:
def flatten_two_lists(a, b):
    yield from a
    yield from b

print(list(flatten_two_lists([1, 2], [3, 4, 5])))

[1, 2, 3, 4, 5]


## 19.3 Delegating to Another Generator

In [60]:
def inner_gen():
    yield "a"
    yield "b"

def outer_gen():
    yield "start"
    yield from inner_gen()   # forwards EVERY value inner_gen() produces, in order
    yield "end"

print(list(outer_gen()))

['start', 'a', 'b', 'end']


| Code / Item | Explanation |
|---|---|
| `yield from inner_gen()` | Fully drives inner_gen() to completion, forwarding each of its yielded values ('a', then 'b') out through outer_gen() as though outer_gen() had yielded them itself, before outer_gen()'s own code continues to its next line. |

## 19.4 Nested Generators: Flattening Arbitrarily Deep Structures

In [61]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # RECURSIVE delegation for arbitrary nesting depth
        else:
            yield item

data = [1, [2, 3, [4, 5]], 6, [[7]]]
print(list(flatten(data)))

[1, 2, 3, 4, 5, 6, 7]


| Code / Item | Explanation |
|---|---|
| `yield from flatten(item)` | A generator function delegating to a recursive call of itself; this correctly handles lists nested to any depth, since each recursive call handles exactly one more level, forwarding everything it eventually produces back up through every enclosing call. |

## 19.5 Generator Composition: Building Pipelines With yield from

yield from is also commonly used to compose several processing stages into a single, combined generator function, an alternative, sometimes more readable style to the explicitly-chained functions shown in Part 16.1.

In [62]:
def evens(iterable):
    for x in iterable:
        if x % 2 == 0:
            yield x

def doubled(iterable):
    for x in iterable:
        yield x * 2

def pipeline(iterable):
    yield from doubled(evens(iterable))

print(list(pipeline(range(10))))

[0, 4, 8, 12, 16]


## Part 19 Practice Exercises

**Beginner:** What does 'yield from [1, 2, 3]' inside a generator function accomplish, in terms of individual yield statements?

*Answer:* It is equivalent to writing 'yield 1', then 'yield 2', then 'yield 3' explicitly; yield from forwards every value from the given iterable, one at a time, in order, as though each had been yielded directly.

**Intermediate:** In outer_gen() from 19.3, in what order do the values 'start', 'a', 'b', 'end' get yielded, and why?

*Answer:* In exactly that order: 'start', 'a', 'b', 'end'. Execution runs top to bottom as usual: 'yield "start"' happens first; then 'yield from inner_gen()' fully delegates to inner_gen(), which yields 'a' and then 'b', both of which are forwarded out through outer_gen() before outer_gen()'s own code proceeds any further; only once inner_gen() is fully exhausted does outer_gen() continue past the yield from line to its final 'yield "end"'.

**Advanced:** Explain why flatten(nested) in 19.4 needs to be a recursive generator function using yield from flatten(item), rather than being written as a single flat loop with no recursion, to correctly handle a list nested to an arbitrary, unknown depth.

*Answer:* A single flat loop can only ever inspect one fixed level of nesting directly; it has no general way to 'reach inside' an item that is itself a list containing further, arbitrarily-nested lists, without essentially reimplementing recursion manually (for example, using an explicit stack). By having flatten() call itself recursively on any item that is itself a list, and using yield from to forward everything that recursive call eventually produces, each level of nesting is handled by exactly one additional recursive call, and because yield from correctly forwards values through any number of chained delegations, a value nested five levels deep gets forwarded up through five separate yield from calls, one per level, arriving at the very top exactly the same way a value nested only one level deep would; this generality, correctly handling any nesting depth without the code needing to know that depth in advance, is precisely what makes the recursive, yield from-based approach necessary rather than optional here.

# Part 20. Generator Return Values

Generators can use return, but it behaves quite differently from return in an ordinary function. This part explains exactly what return does inside a generator, how its value is exposed via StopIteration.value, how to capture it, and how yield from can retrieve it directly.

## 20.1 return Inside a Generator: What Actually Happens

Using 'return value' inside a generator function does NOT make the generator return that value the way an ordinary function would. Instead, it immediately ends the generator (exactly like reaching the end of the function body normally would), and the StopIteration exception that gets raised as a result carries that value, attached to the exception itself, as its .value attribute.

In [63]:
def gen_with_return():
    yield 1
    yield 2
    return "all done"

gen = gen_with_return()
print(next(gen))
print(next(gen))
try:
    next(gen)
except StopIteration as e:
    print("Caught StopIteration, value:", e.value)

1
2
Caught StopIteration, value: all done


| Code / Item | Explanation |
|---|---|
| `return "all done"` | Does NOT get yielded as a normal value; a for loop consuming this generator would never see 'all done' appear as one of its produced items. |
| `except StopIteration as e: ... e.value` | The returned value is instead attached to the StopIteration exception's .value attribute, accessible only by explicitly catching that exception, exactly as demonstrated here. |

## 20.2 Why a for Loop Never Sees a Generator's Return Value

Since a for loop's implicit exception handling (Part 3.6, Part 6.3) simply catches StopIteration and breaks out of the loop silently, it has no built-in mechanism to expose that exception's .value attribute to your code; the return value is effectively invisible unless you either drive the generator manually with explicit next()/try/except code (as in 20.1), or use yield from specifically to retrieve it (20.3).

## 20.3 Capturing a Generator's Return Value With yield from

yield from is the one construct that DOES give you direct, convenient access to a delegated generator's return value: the entire 'yield from some_generator()' expression itself evaluates to that generator's return value, once it is fully exhausted.

In [64]:
def inner():
    yield "a"
    yield "b"
    return "inner is finished"

def outer():
    result = yield from inner()   # 'result' captures inner()'s return value directly
    yield f"inner returned: {result}"

for value in outer():
    print(value)

a
b
inner returned: inner is finished


| Code / Item | Explanation |
|---|---|
| `result = yield from inner()` | The yield from expression forwards 'a' and 'b' out of outer() exactly as before, but ALSO, once inner() finishes, assigns inner()'s return value directly to result, which outer()'s own code can then use immediately, here to build one final yielded message. |

## 20.4 A Practical Use: Aggregating Results From a Sub-Pipeline

In [65]:
def process_batch(items):
    total = 0
    for item in items:
        yield item * 2
        total += item
    return total   # a summary value, available only via StopIteration.value or yield from

def run_and_report(items):
    total = yield from process_batch(items)
    print(f"Batch total (of original values): {total}")

list(run_and_report([1, 2, 3]))   # driving it with list() to actually run it to completion

Batch total (of original values): 6


[2, 4, 6]

## Part 20 Practice Exercises

**Beginner:** Does 'return "done"' inside a generator function cause a for loop consuming that generator to receive "done" as one of its iterated values?

*Answer:* No. The return value is never produced as a yielded value; a for loop simply sees the generator end (StopIteration is raised internally and silently caught), with no visibility into the returned value at all through the for loop itself.

**Intermediate:** How can you retrieve the value a generator returned, if you are NOT using yield from to delegate to it?

*Answer:* By manually driving the generator with next() inside a try/except block, and reading the .value attribute off the caught StopIteration exception, exactly as shown in 20.1: 'try: next(gen)\nexcept StopIteration as e: result = e.value'.

**Advanced:** In run_and_report(items) from 20.4, explain why the code uses 'list(run_and_report([1, 2, 3]))' to drive the generator, rather than simply calling 'run_and_report([1, 2, 3])' on its own line.

*Answer:* run_and_report is itself a generator function (it contains a yield from expression, which counts as containing yield, per Part 9.1's definition), so simply calling run_and_report([1, 2, 3]) only creates a generator object; none of its body, including the 'yield from process_batch(items)' line and the subsequent print() statement, would actually execute at all, exactly as established in Part 9.4's lazy-creation rule. Wrapping the call in list(...) forces the returned generator to actually be driven to completion, by repeatedly calling next() on it internally until it raises StopIteration, which is what causes 'yield from process_batch(items)' to actually run process_batch() to completion, capture its returned total, and then execute the print() statement showing that total; without some form of actually consuming the outer generator, none of that code would ever run.

# Part 21. Sending Values Into Generators (.send())

Generators can do more than just produce values outward; they can also receive values from the caller through the same yield expression that produces them, using the .send() method. This part builds this idea up carefully from a first, minimal example to genuinely practical patterns.

## 21.1 yield as an Expression, Not Just a Statement

So far, every example has used yield purely as a statement that produces a value. yield is actually an expression, meaning 'value = yield something' is valid syntax: it yields 'something' outward, pauses, and once resumed, whatever was passed in via .send() becomes the value of the entire yield expression, assigned to value.

## 21.2 A First, Minimal .send() Example

In [66]:
def echo():
    while True:
        received = yield
        print("Received:", received)

gen = echo()
next(gen)          # REQUIRED: prime the generator, running it to the first yield
gen.send("hello")
gen.send("world")

Received: hello
Received: world


| Code / Item | Explanation |
|---|---|
| `next(gen) (priming)` | A generator must be advanced to its first yield expression before .send() can deliver a value into it; calling .send(value) on a brand-new, not-yet-started generator (other than .send(None), which behaves like priming) raises TypeError. |
| `gen.send("hello")` | Resumes execution at the currently-paused 'yield' expression, making that expression evaluate to "hello", which gets assigned to received; execution then continues until the next yield (here, looping back to the top and pausing at yield again). |

## 21.3 Why Priming Is Required

In [67]:
def unprimed():
    x = yield
    print(x)

gen = unprimed()
try:
    gen.send("too early")
except TypeError as e:
    print("TypeError:", e)

TypeError: can't send non-None value to a just-started generator


> **Note:** A brand-new generator's execution is paused BEFORE its first line, not at a yield expression yet; there is no yield expression currently waiting to receive a sent value, which is exactly why .send(value) with a non-None value fails until at least one next() (or an equivalent gen.send(None)) has advanced execution to the first actual yield.

## 21.4 A Practical, Stateful .send() Pattern: A Configurable Accumulator

In [68]:
def accumulator():
    total = 0
    while True:
        amount = yield total
        if amount:
            total += amount

acc = accumulator()
print(next(acc))          # 0: primes, gets the initial total
print(acc.send(10))       # 10
print(acc.send(5))        # 15
print(acc.send(-3))       # 12: negative amounts work too, it is ordinary addition

0
10
15
12


## 21.5 send() vs next(): They Are Closely Related

gen.send(None) and next(gen) are functionally identical: next(gen) is essentially a convenient shorthand for gen.send(None), 'resume execution, delivering no particular value into the paused yield expression'. This is why the very first call used to prime a generator can be written as either next(gen) or gen.send(None), interchangeably.

In [69]:
def demo():
    x = yield "first"
    print("received:", x)
    yield "second"

gen = demo()
print(next(gen))         # primes, same as gen.send(None) would be
print(gen.send("value"))

first
received: value
second


## 21.6 Practical Applications of .send()

- Coroutine-style generators that maintain running state and react to values fed in incrementally, such as the accumulator() pattern above, or a moving-average calculator.
- Simple, cooperative pipeline stages where a downstream generator can occasionally send configuration or control values back upstream (an advanced, less common pattern).
- Historically, .send() and yield-based coroutines were the foundation Python's async/await syntax was built on top of; modern async code (Part 32) uses async/await directly rather than .send(), but understanding .send() makes the underlying mechanism far less mysterious.

## Part 21 Practice Exercises

**Beginner:** Before you can call gen.send(some_value) with a real, non-None value, what must you do first?

*Answer:* Prime the generator by calling next(gen) once (or equivalently gen.send(None)), advancing its execution to its first yield expression.

**Intermediate:** What is the relationship between next(gen) and gen.send(None)?

*Answer:* They are functionally equivalent; next(gen) resumes the generator, delivering no meaningful value into the currently-paused yield expression, which is exactly what gen.send(None) does explicitly. next() can be thought of as shorthand specifically for the 'send nothing in particular' case.

**Advanced:** Write a generator function moving_average() that receives numbers via successive .send() calls and always yields the average of all numbers received so far (starting at 0 for no numbers yet). Demonstrate it with three .send() calls, and explain why the running count and running total must both be preserved as local state between calls.

*Answer:* def moving_average():
    total = 0
    count = 0
    average = 0
    while True:
        value = yield average
        if value is not None:
            total += value
            count += 1
            average = total / count

gen = moving_average()
next(gen)                  # prime
print(gen.send(10))        # 10.0
print(gen.send(20))        # 15.0
print(gen.send(30))        # 20.0
Both total and count must persist across every .send() call because computing a correct running average requires knowing both the sum of everything received so far AND how many values have been received so far; if either were reset or recomputed from scratch on each call, the function would have no way to know the history of previously sent values, since each .send() call only delivers the single newest value at a time, not the entire history. This works correctly only because, per Part 10.3, a generator's local variables are genuinely preserved across every pause and resume on the same generator object, letting total and count accumulate exactly like a running total would inside a long-lived object's instance attributes, but expressed here purely through a paused function's local scope instead.

# Part 22. .throw() and .close()

Beyond producing and receiving values, a generator can also have an exception injected into it (.throw()) or be explicitly shut down early (.close()). This part explains both, including the GeneratorExit exception that close() relies on, and when these features are genuinely useful in practice.

## 22.1 .throw(): Injecting an Exception Into a Paused Generator

gen.throw(exc_type) resumes a paused generator, but instead of delivering a normal value at the paused yield expression, it raises the given exception AT that exact point inside the generator's code, as though that yield expression itself had raised it. The generator's own code can catch it with a normal try/except, exactly as it could catch any other exception.

In [70]:
def resilient_gen():
    while True:
        try:
            value = yield
            print("Got:", value)
        except ValueError:
            print("Recovered from a ValueError, continuing")

gen = resilient_gen()
next(gen)
gen.send("normal value")
gen.throw(ValueError)
gen.send("still works after recovering")

Got: normal value
Recovered from a ValueError, continuing
Got: still works after recovering


| Code / Item | Explanation |
|---|---|
| `gen.throw(ValueError)` | Resumes the generator, but raises ValueError exactly at the paused 'yield' expression, inside the generator's own try/except block, which catches it and prints a recovery message, then the loop continues normally. |
| `gen.send("still works ...") afterward` | Proves the generator survived the injected exception and is still running normally, since its except block caught the exception rather than letting it propagate and end the generator. |

## 22.2 What Happens If the Generator Does NOT Catch the Thrown Exception

In [71]:
def fragile_gen():
    yield "start"
    yield "more"

gen = fragile_gen()
next(gen)
try:
    gen.throw(RuntimeError("injected failure"))
except RuntimeError as e:
    print("Propagated out to the caller:", e)

Propagated out to the caller: injected failure


> **Note:** If the generator's code has no matching except block at the point where the exception was injected, the exception propagates out of the .throw() call itself to whatever code called .throw(), exactly like an uncaught exception anywhere else, and the generator is left in a finished, unusable state afterward.

## 22.3 .close(): Explicitly Shutting Down a Generator Early

gen.close() is used to deliberately terminate a generator that will never be fully consumed, most commonly when only some of a generator's values were needed and the rest should be abandoned cleanly, especially important if the generator holds a resource (an open file, a database connection) that needs proper cleanup.

In [72]:
def resource_using_gen():
    print("Resource acquired")
    try:
        yield 1
        yield 2
        yield 3
    finally:
        print("Resource released")   # guaranteed to run, even if closed early

gen = resource_using_gen()
print(next(gen))
gen.close()      # abandon the rest; the 'finally' block still runs, cleaning up

Resource acquired
1
Resource released


## 22.4 GeneratorExit: How .close() Actually Works Internally

Calling .close() works by internally raising a special exception, GeneratorExit, at the generator's currently-paused yield expression, very similarly to how .throw() works, but specifically to signal 'please stop now'. A generator's own try/finally block (as in 22.3) still runs normally in response, guaranteeing cleanup code executes; a generator's own code should generally NOT catch and suppress GeneratorExit and continue yielding more values afterward, since doing so violates the expected close() contract, and Python raises RuntimeError if a generator attempts to yield again after receiving GeneratorExit.

## 22.5 When These Features Are Genuinely Useful

- Ensuring cleanup code (closing a file, releasing a lock, closing a network connection) runs correctly even when a generator-based resource is abandoned early, via .close() combined with try/finally, exactly the pattern in 22.3.
- Testing and simulating error conditions inside a long-running generator-based pipeline, using .throw() to verify the pipeline's own error-handling logic behaves correctly.
- Advanced coroutine-style patterns where a generator needs to be told, from the outside, 'stop what you're doing and clean up' or 'here's an error condition to react to', without simply letting the generator run to natural completion.

## Part 22 Practice Exercises

**Beginner:** What exception does Python internally raise inside a generator when .close() is called on it?

*Answer:* GeneratorExit.

**Intermediate:** In resource_using_gen() from 22.3, why does 'Resource released' print even though gen.close() is called after only the first value (1) has been retrieved, well before the generator would naturally finish?

*Answer:* gen.close() raises GeneratorExit at the generator's currently-paused yield 1 expression. Because that yield sits inside a try/finally block, Python guarantees the finally clause runs regardless of how the try block's execution ends, whether it completes normally, raises an uncaught exception, or, as here, is interrupted by GeneratorExit from a close() call; so 'Resource released' executes as the generator shuts down, exactly as intended for reliable resource cleanup, even though yield 2 and yield 3 are never reached at all.

**Advanced:** Explain why a generator's code generally should NOT catch GeneratorExit and then continue yielding additional values afterward, referencing what .close() is meant to guarantee to its caller.

*Answer:* The entire purpose of .close() is to give calling code a reliable way to say 'I am done consuming this generator; please stop and clean up', with the expectation that once close() returns, the generator is genuinely finished and will not produce any further values. If a generator's code caught GeneratorExit inside a try/except (rather than only using try/finally for cleanup) and then proceeded to yield another value anyway, it would violate that guarantee: the caller that called close(), believing the generator had shut down, could not safely assume that; Python enforces this expectation directly by raising RuntimeError if a generator does attempt to yield again after being sent GeneratorExit, specifically to catch and surface this misuse loudly, rather than silently allowing a 'closed' generator to keep producing values as though nothing had happened.

# Part 23. Generators vs Custom Iterator Classes

Parts 7 and 8 built custom iterator classes by hand; Parts 9 through 22 built the exact same kind of functionality using generator functions and expressions. This part compares the two approaches directly, so you can choose deliberately rather than by habit.

## 23.1 The Same Iterator, Two Ways

In [73]:
# As a custom iterator class (Part 7 style):
class CounterClass:
    def __init__(self, start, end):
        self.current = start
        self.end = end
    def __iter__(self):
        return self
    def __next__(self):
        if self.current > self.end:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

# As a generator function:
def counter_gen(start, end):
    current = start
    while current <= end:
        yield current
        current += 1

print(list(CounterClass(1, 5)))
print(list(counter_gen(1, 5)))

[1, 2, 3, 4, 5]
[1, 2, 3, 4, 5]


> **Note:** Both produce identical output; the generator version needs no explicit __iter__()/__next__() methods, no manually-tracked self.current attribute, and no explicit StopIteration statement, since while simply ending achieves the same effect automatically.

## 23.2 Simplicity and Readability

A generator function reads like ordinary, linear procedural code with pauses inserted (yield), which is usually dramatically easier to read and write correctly than the equivalent class, especially once any nontrivial state or branching logic is involved, as Part 7.6's ValidRecordIterator demonstrated. A custom iterator class becomes preferable mainly when the 'iterator' needs to expose additional methods or attributes beyond plain iteration (configuration, introspection, or being part of a larger object with other responsibilities).

## 23.3 State Management

A generator function's state (local variables) is managed entirely automatically by Python's generator machinery, with no explicit self.something bookkeeping required at all. A custom iterator class requires every piece of state to be manually declared as an instance attribute in __init__() and manually updated inside __next__(), which is more verbose and more error-prone (it is easy to forget to update one piece of state correctly).

## 23.4 Memory Usage

Both a generator object and a custom iterator instance are lightweight; neither one inherently uses significantly more memory than the other for equivalent logic. The memory-efficiency benefits discussed throughout Parts 13 and 14 apply equally to both approaches, since both are genuine, protocol-compliant iterators that produce values lazily, one at a time; the choice between them is not a memory trade-off.

## 23.5 Flexibility

A custom iterator class has an advantage when you need more than pure iteration: additional public methods, custom __repr__(), the ability to be reused as a base class with inheritance, or when the same object needs to serve multiple different roles beyond just being an iterator. A generator function is inherently limited to producing values through iteration alone; it cannot easily expose additional methods for callers to use, beyond the small set (.send(), .throw(), .close()) already built into every generator.

## 23.6 Performance

In CPython, generator functions are generally at least as fast as, and often somewhat faster than, an equivalent hand-written iterator class, because generator objects are implemented with lower per-value overhead than a full Python class instance with its own __dict__ and method-lookup machinery. Part 33 covers this with concrete benchmarking guidance rather than an unqualified claim.

## 23.7 Appropriate Use Cases: A Decision Guide

- Choose a generator function when: the logic is a straightforward, mostly linear sequence of steps producing values, no extra public methods/attributes are needed beyond iteration itself, and readability/maintainability matter most (this covers the large majority of real-world cases).
- Choose a generator expression when: the entire logic fits naturally in a single 'for ... in ... if ...' expression, typically feeding directly into another function (Part 12).
- Choose a custom iterator class when: the object needs additional methods or attributes beyond pure iteration, needs to participate in inheritance, needs a meaningful __repr__() or equality behavior, or when the iteration logic is genuinely complex enough that separating state (in __init__) from advancement logic (in __next__) as distinct, individually testable pieces is clearer than one continuous generator function body.

## 23.8 Generators vs Generator Expressions vs Custom Iterator Classes

| Aspect | Generator Function | Generator Expression | Custom Iterator Class |
|---|---|---|---|
| Syntax overhead | Low: a normal function with yield | Lowest: a single expression | Highest: a full class with 3 methods |
| State management | Automatic (local variables) | Automatic (implicit) | Manual (instance attributes) |
| Readability for complex logic | High | Low, if logic is nontrivial | Medium (state and logic are separated) |
| Can expose extra methods/attributes | No (beyond send/throw/close) | No | Yes, freely |
| Supports inheritance | No | No | Yes |
| Typical use case | Most real-world iteration logic | Simple, single-pass transforms | Iterators needing extra behavior/state structure |

## Part 23 Practice Exercises

**Beginner:** For most everyday iteration logic, which is generally recommended: a generator function or a hand-written custom iterator class?

*Answer:* A generator function; it is typically simpler, more readable, and requires far less boilerplate for the same behavior, per 23.2 and 23.7.

**Intermediate:** Name one genuine capability a custom iterator class has that a generator function does not.

*Answer:* A custom iterator class can expose additional public methods and attributes beyond pure iteration (for example, a .reset() method, a .progress property, or custom __repr__()/__eq__() behavior), and can participate in class inheritance; a generator function is limited to producing values via iteration plus the built-in .send()/.throw()/.close() methods.

**Advanced:** A team is building a DataBatchIterator that needs to: (a) iterate lazily over batches of a large dataset, AND (b) expose a public .batches_processed_so_far property that other parts of the system can inspect at any time, even mid-iteration, without consuming any values. Using the decision guide in 23.7, justify whether this is better implemented as a generator function or a custom iterator class.

*Answer:* This scenario is a strong case for a custom iterator class rather than a generator function. The core iteration logic itself (lazily producing batches) is something either approach could express reasonably well, but the requirement in (b), exposing a public, independently-inspectable property that other code can read at any arbitrary moment without affecting or consuming the iteration itself, is exactly the kind of 'extra method/attribute beyond pure iteration' that 23.5 and 23.7 identify as a genuine advantage unique to custom iterator classes. A generator function has no mechanism to expose an inspectable, externally-readable attribute like this at all; the only way to get information out of a generator mid-iteration is by consuming a value from it (advancing it) or by having it .send() information back (Part 21), neither of which matches the requirement to inspect a running counter passively, from outside, without affecting iteration progress. A custom class can trivially satisfy this by storing batches_processed_so_far as a plain instance attribute, updated inside __next__(), and exposed as a public property that any other code can read freely at any time, which directly and cleanly satisfies requirement (b) in a way no generator function construction can.

# Part 24. itertools: A Comprehensive Practical Guide

itertools is a standard-library module of fast, memory-efficient, composable building blocks for working with iterators. Every function in it is lazy, consistent with everything this handbook has covered, and this part walks through every commonly used one: purpose, syntax, example, expected output, a practical application, and a common mistake to avoid.

## 24.1 Infinite Iterators: count()

Purpose: produces an endless, evenly-spaced arithmetic sequence of numbers, starting from a given value.

In [74]:
import itertools
gen = itertools.count(10, 3)
print(next(gen), next(gen), next(gen))

10 13 16


> **Note:** Common mistake: passing count() directly to list() or sum() with no bound; since it never stops on its own, this will hang or exhaust memory. Always pair it with something bounding, such as itertools.islice() (24.16) or zip() (Part 26) against a finite iterable.

## 24.2 Infinite Iterators: cycle()

Purpose: repeats the elements of a given iterable endlessly, looping back to the start each time it is exhausted.

In [75]:
import itertools
gen = itertools.cycle(["A", "B", "C"])
print([next(gen) for _ in range(7)])

['A', 'B', 'C', 'A', 'B', 'C', 'A']


> **Note:** Practical application: assigning round-robin labels or colors to a sequence of items. Common mistake: identical to count(), never bound it directly with list().

## 24.3 Infinite Iterators: repeat()

Purpose: yields the same value over and over, either forever, or exactly n times if a count is given.

In [76]:
import itertools
print(list(itertools.repeat("x", 4)))
gen = itertools.repeat(0)   # infinite, unless bounded
print(next(gen), next(gen))

['x', 'x', 'x', 'x']
0 0


> **Note:** Practical application: providing a constant argument alongside a varying one in a call to map(), such as map(pow, [1,2,3], itertools.repeat(2)) to square each number.

## 24.4 Combinatoric Iterators: product()

Purpose: the Cartesian product of two or more iterables, equivalent to a set of nested for loops, but lazy.

In [77]:
import itertools
print(list(itertools.product([1, 2], ["a", "b"])))
print(list(itertools.product([0, 1], repeat=3)))

[(1, 'a'), (1, 'b'), (2, 'a'), (2, 'b')]
[(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]


| Code / Item | Explanation |
|---|---|
| `itertools.product([0, 1], repeat=3)` | The repeat keyword argument repeats the SAME iterable that many times in the product, here producing all 8 length-3 combinations of 0s and 1s, equivalent to product([0,1],[0,1],[0,1]). |

> **Note:** Common mistake: forgetting that product() grows combinatorially; product() over several large iterables can produce an astronomically large (or effectively infinite, if any input is infinite) number of combinations.

## 24.5 Combinatoric Iterators: permutations()

Purpose: all possible orderings of a given length r (default: full length) from an iterable, without repeating any element within one permutation.

In [78]:
import itertools
print(list(itertools.permutations(["A", "B", "C"])))
print(list(itertools.permutations(["A", "B", "C"], 2)))

[('A', 'B', 'C'), ('A', 'C', 'B'), ('B', 'A', 'C'), ('B', 'C', 'A'), ('C', 'A', 'B'), ('C', 'B', 'A')]
[('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]


## 24.6 Combinatoric Iterators: combinations()

Purpose: all possible selections of r elements from an iterable, in their original relative order, without regard to different orderings of the same selection (unlike permutations).

In [79]:
import itertools
print(list(itertools.combinations(["A", "B", "C"], 2)))

[('A', 'B'), ('A', 'C'), ('B', 'C')]


> **Note:** Practical application: generating all possible pairs of items to compare (for example, all pairs of products to A/B test, or all pairs of users to compute a similarity score between).

## 24.7 Combinatoric Iterators: combinations_with_replacement()

Purpose: like combinations(), but allows the same element to be selected more than once within a single combination.

In [80]:
import itertools
print(list(itertools.combinations_with_replacement(["A", "B"], 2)))

[('A', 'A'), ('A', 'B'), ('B', 'B')]


## 24.8 Terminating Iterators: accumulate()

Purpose: yields a running (cumulative) total by default, or the running result of any given two-argument function.

In [81]:
import itertools
import operator
print(list(itertools.accumulate([1, 2, 3, 4])))
print(list(itertools.accumulate([1, 2, 3, 4], operator.mul)))

[1, 3, 6, 10]
[1, 2, 6, 24]


## 24.9 Terminating Iterators: chain()

Purpose: iterates over several iterables in sequence, one after another, as though they were one single, combined iterable.

In [82]:
import itertools
print(list(itertools.chain([1, 2], (3, 4), "ab")))
print(list(itertools.chain.from_iterable([[1, 2], [3], [4, 5]])))

[1, 2, 3, 4, 'a', 'b']
[1, 2, 3, 4, 5]


| Code / Item | Explanation |
|---|---|
| `itertools.chain.from_iterable(...)` | A variant that takes a single iterable OF iterables and chains all of them, useful when you have a dynamically-sized list of lists rather than a fixed, known number of separate arguments. |

## 24.10 Terminating Iterators: compress()

Purpose: filters one iterable's elements according to the truthiness of the corresponding elements in a second, parallel selector iterable.

In [83]:
import itertools
data = ["a", "b", "c", "d"]
selectors = [1, 0, 1, 0]
print(list(itertools.compress(data, selectors)))

['a', 'c']


## 24.11 Terminating Iterators: dropwhile() and takewhile()

Purpose: dropwhile() skips elements as long as a predicate is true, then yields everything remaining, without checking the predicate again. takewhile() does the opposite: yields elements as long as the predicate is true, and stops the instant it first becomes false.

In [84]:
import itertools
nums = [1, 2, 3, 10, 2, 1]
print(list(itertools.dropwhile(lambda x: x < 5, nums)))
print(list(itertools.takewhile(lambda x: x < 5, nums)))

[10, 2, 1]
[1, 2, 3]


> **Note:** Common mistake: expecting dropwhile() to drop every element matching the predicate throughout the whole sequence; it only checks the predicate at the start, and STOPS checking the moment it first fails, yielding everything after that point regardless (notice the trailing 2, 1 in the dropwhile() output above, even though they are also less than 5).

## 24.12 Terminating Iterators: filterfalse()

Purpose: the logical opposite of the built-in filter(): keeps only elements for which the predicate is False.

In [85]:
import itertools
nums = range(10)
print(list(itertools.filterfalse(lambda x: x % 2 == 0, nums)))   # keeps ODD numbers

[1, 3, 5, 7, 9]


## 24.13 Terminating Iterators: groupby()

Purpose: groups consecutive elements that share the same key, per a given key function. CRITICALLY, it only groups CONSECUTIVE matching elements; input is almost always expected to be pre-sorted by the same key for groupby() to produce the grouping you actually want.

In [86]:
import itertools
data = [("fruit", "apple"), ("fruit", "banana"), ("veg", "carrot"), ("fruit", "kiwi")]
for key, group in itertools.groupby(data, key=lambda x: x[0]):
    print(key, list(group))

fruit [('fruit', 'apple'), ('fruit', 'banana')]
veg [('veg', 'carrot')]
fruit [('fruit', 'kiwi')]


> **Note:** Common mistake: NOT sorting the input by the grouping key first. Notice above that the final ('fruit', 'kiwi') forms its own SEPARATE group from the earlier fruit entries, because 'veg' appeared in between, breaking consecutiveness; the fix is 'sorted(data, key=lambda x: x[0])' before calling groupby().

## 24.14 Terminating Iterators: pairwise()

Purpose (Python 3.10+): yields successive overlapping pairs of elements from an iterable.

In [87]:
import itertools
print(list(itertools.pairwise([1, 2, 3, 4])))

[(1, 2), (2, 3), (3, 4)]


> **Note:** Practical application: computing differences or ratios between successive readings in a time series, such as day-over-day price changes.

## 24.15 Terminating Iterators: starmap()

Purpose: like map(), but unpacks each element of the input iterable as separate positional arguments to the function, rather than passing each element as a single argument.

In [88]:
import itertools
import operator
pairs = [(2, 3), (4, 5), (6, 2)]
print(list(itertools.starmap(operator.mul, pairs)))

[6, 20, 12]


> **Note:** This is exactly equivalent to '[func(*args) for args in pairs]'; use starmap() when your data already comes as tuples of arguments, and plain map() when it comes as single values.

## 24.16 Terminating Iterators: islice()

Purpose: applies list-slicing-style start/stop/step behavior directly to any iterator or iterable, lazily, without needing to support real slicing or consume the whole thing first.

In [89]:
import itertools
gen = itertools.count(1)
first_five = itertools.islice(gen, 5)
print(list(first_five))
print(list(itertools.islice(range(20), 2, 10, 3)))

[1, 2, 3, 4, 5]
[2, 5, 8]


> **Note:** This is precisely the standard, idiomatic way to safely bound an infinite iterator like count() or cycle(), as flagged in 24.1 and 24.2.

## 24.17 Terminating Iterators: tee()

Purpose: splits a single iterator into n independent iterators, each of which can be advanced separately without affecting the others.

In [90]:
import itertools
original = iter([1, 2, 3, 4])
a, b = itertools.tee(original, 2)
print(next(a), next(a))
print(next(b))

1 2
1


> **Note:** Common mistake: continuing to use the ORIGINAL iterator (original, above) after calling tee() on it; once tee()'d, the original should generally be considered consumed/off-limits, and only the returned copies should be used, since tee() and the original iterator share and compete for the same underlying data source internally.

## 24.18 Terminating Iterators: zip_longest()

Purpose: like the built-in zip(), but continues until the LONGEST input is exhausted, filling in a specified fill value (default None) for any shorter iterables that have already run out.

In [91]:
import itertools
names = ["Ada", "Bo", "Chi"]
scores = [90, 85]
print(list(itertools.zip_longest(names, scores, fillvalue=0)))

[('Ada', 90), ('Bo', 85), ('Chi', 0)]


> **Note:** Contrast directly with the built-in zip(), which STOPS at the shortest input instead (Part 26 covers this distinction, and Python 3.10's zip(strict=True), in full).

## 24.19 itertools Quick-Reference Summary

| Function | Category | One-Line Purpose |
|---|---|---|
| count(start, step) | Infinite | Endless arithmetic sequence. |
| cycle(iterable) | Infinite | Repeats an iterable's elements forever. |
| repeat(value, n) | Infinite | Yields the same value, forever or n times. |
| product(*iterables) | Combinatoric | Cartesian product (nested-loop equivalent). |
| permutations(iterable, r) | Combinatoric | All orderings of length r. |
| combinations(iterable, r) | Combinatoric | All r-length selections, order-independent. |
| combinations_with_replacement(iterable, r) | Combinatoric | Like combinations(), repeats allowed. |
| accumulate(iterable, func) | Terminating | Running (cumulative) result. |
| chain(*iterables) | Terminating | Iterates several iterables as one. |
| compress(data, selectors) | Terminating | Filters data by a parallel truthy/falsy mask. |
| dropwhile(pred, iterable) | Terminating | Skips a leading run matching pred, then yields the rest. |
| filterfalse(pred, iterable) | Terminating | Keeps elements where pred is False. |
| groupby(iterable, key) | Terminating | Groups CONSECUTIVE elements sharing a key. |
| islice(iterable, start, stop, step) | Terminating | Lazy slicing for any iterator. |
| pairwise(iterable) | Terminating | Successive overlapping pairs. |
| starmap(func, iterable_of_tuples) | Terminating | map() that unpacks each tuple as arguments. |
| takewhile(pred, iterable) | Terminating | Yields while pred is true; stops at first failure. |
| tee(iterable, n) | Terminating | Splits one iterator into n independent ones. |
| zip_longest(*iterables, fillvalue) | Terminating | Like zip(), but pads to the LONGEST input. |

## Part 24 Practice Exercises

**Beginner:** Which itertools function would you use to get the first 5 values from an infinite itertools.count() generator?

*Answer:* itertools.islice(gen, 5).

**Intermediate:** Explain why groupby() on data = [("a",1),("b",2),("a",3)] (grouped by the first element) produces THREE separate groups instead of two, and how to fix it.

*Answer:* groupby() only groups elements that are CONSECUTIVE in the input and share the same key; it does not scan ahead to find every matching element throughout the whole sequence. Here, the two ('a', ...) entries are not adjacent, since a ('b', 2) entry sits between them, so groupby() sees a run of one 'a', then a run of one 'b', then a run of one 'a' again, three separate consecutive runs, producing three groups rather than combining both 'a' entries into one. The fix is to sort the data by the same key first: 'sorted(data, key=lambda x: x[0])', which rearranges it so that every element sharing a key becomes consecutive before groupby() ever sees it.

**Advanced:** Using itertools, write an expression that produces all unique unordered pairs of elements from the list ['Ada', 'Bo', 'Chi', 'Dumi'] (each pair appearing exactly once, with no element paired with itself, and (A, B) not appearing separately from (B, A)), and explain which itertools function is the correct choice and why permutations() would be the wrong choice here.

*Answer:* list(itertools.combinations(['Ada', 'Bo', 'Chi', 'Dumi'], 2)) is the correct expression, producing exactly six pairs: [('Ada','Bo'), ('Ada','Chi'), ('Ada','Dumi'), ('Bo','Chi'), ('Bo','Dumi'), ('Chi','Dumi')]. combinations() is correct here because the requirement explicitly excludes both self-pairing and treating (A, B) and (B, A) as distinct, which is exactly what combinations() guarantees: it selects elements without regard to order and never repeats an element within one selection. permutations(['Ada','Bo','Chi','Dumi'], 2) would be the wrong choice because it treats different orderings of the same pair as genuinely distinct results, meaning it would include both ('Ada', 'Bo') AND ('Bo', 'Ada') as two separate entries in its output, which directly violates the requirement that each unordered pair appear exactly once.

# Part 25. itertools for Data Processing

This part applies itertools directly to realistic data-processing tasks: transformation, grouping, batching, generating combinations, cartesian products for parameter search, and streaming pipelines, showing itertools functions composed together rather than used in isolation.

## 25.1 Data Transformation With accumulate() and starmap()

In [92]:
import itertools, operator

daily_sales = [120, 85, 200, 150, 90]
running_total = list(itertools.accumulate(daily_sales))
print("Running total:", running_total)

prices_and_quantities = [(19.99, 3), (5.50, 10), (100.0, 1)]
line_totals = list(itertools.starmap(operator.mul, prices_and_quantities))
print("Line totals:", line_totals)

Running total: [120, 205, 405, 555, 645]
Line totals: [59.97, 55.0, 100.0]


## 25.2 Grouping Data With groupby() (Correctly Sorted First)

In [93]:
import itertools

transactions = [
    {"region": "West", "amount": 100},
    {"region": "East", "amount": 50},
    {"region": "West", "amount": 75},
    {"region": "East", "amount": 30},
]
transactions_sorted = sorted(transactions, key=lambda t: t["region"])
for region, group in itertools.groupby(transactions_sorted, key=lambda t: t["region"]):
    total = sum(t["amount"] for t in group)
    print(region, total)

East 80
West 175


> **Note:** Sorting by the same key BEFORE calling groupby() (per the warning in Part 24.13) is what guarantees every transaction for a given region ends up in the exact same, single group.

## 25.3 Batching a Data Stream With islice()

In [94]:
import itertools

def batched_islice(iterable, batch_size):
    it = iter(iterable)
    while True:
        batch = list(itertools.islice(it, batch_size))
        if not batch:
            return
        yield batch

for batch in batched_islice(range(11), 4):
    print(batch)

[0, 1, 2, 3]
[4, 5, 6, 7]
[8, 9, 10]


| Code / Item | Explanation |
|---|---|
| `list(itertools.islice(it, batch_size))` | Pulls the next batch_size items directly off the shared iterator it, lazily, without needing to slice the original iterable at all (which would not even be possible if iterable were, say, an unbounded generator). |

## 25.4 Cartesian Products for Parameter Search

product() is a natural fit for exhaustively generating every combination of hyperparameter values to try during a grid search, a common Data Science and Machine Learning task.

In [95]:
import itertools

learning_rates = [0.01, 0.1]
batch_sizes = [16, 32]
for lr, bs in itertools.product(learning_rates, batch_sizes):
    print(f"Trying learning_rate={lr}, batch_size={bs}")

Trying learning_rate=0.01, batch_size=16
Trying learning_rate=0.01, batch_size=32
Trying learning_rate=0.1, batch_size=16
Trying learning_rate=0.1, batch_size=32


## 25.5 Combinations for Pairwise Comparison Tasks

In [96]:
import itertools

products = ["A", "B", "C"]
for p1, p2 in itertools.combinations(products, 2):
    print(f"Compare {p1} vs {p2}")

Compare A vs B
Compare A vs C
Compare B vs C


## 25.6 A Streaming Pipeline Combining Several itertools Functions

In [97]:
import itertools

def sensor_readings():
    for i in itertools.count():
        yield i % 10   # a simulated, unending stream of readings

stream = sensor_readings()
first_20 = itertools.islice(stream, 20)
above_five = itertools.filterfalse(lambda x: x <= 5, first_20)
print(list(above_five))

[6, 7, 8, 9, 6, 7, 8, 9]


| Code / Item | Explanation |
|---|---|
| `itertools.islice(stream, 20)` | Bounds the otherwise-infinite sensor_readings() stream to just the first 20 readings, lazily. |
| `itertools.filterfalse(lambda x: x <= 5, first_20)` | Chains directly onto the bounded stream, keeping only readings greater than 5; nothing here materializes an intermediate list at any stage until list() is finally called at the very end. |

## 25.7 Large Dataset Processing: Combining Files, itertools, and Batching

In [98]:
import itertools

def parse_lines(lines):
    for line in lines:
        parts = line.strip().split(",")
        yield {"name": parts[0], "score": int(parts[1])}

raw_lines = ["Ada,90", "Bo,55", "Chi,95", "Dumi,60", "Efe,88"]
records = parse_lines(raw_lines)
high_scorers = itertools.dropwhile(lambda r: r["score"] < 80, records)
for r in itertools.islice(high_scorers, 2):
    print(r)

{'name': 'Ada', 'score': 90}
{'name': 'Bo', 'score': 55}


> **Note:** This chains a generator (parse_lines), an itertools filtering function (dropwhile), and an itertools bounding function (islice) into a single lazy pipeline, processing the exact same way whether raw_lines were five in-memory strings or a multi-gigabyte streamed file (Part 15).

## Part 25 Practice Exercises

**Beginner:** Which itertools function is the natural choice for generating every combination of two independent hyperparameter lists to try in a grid search?

*Answer:* itertools.product().

**Intermediate:** In 25.2's grouping example, what would go wrong if the sorted() call were removed, and transactions were grouped in their original order instead?

*Answer:* Since groupby() only merges CONSECUTIVE elements sharing the same key (Part 24.13), grouping the original, unsorted order (West, East, West, East) would produce four separate single-transaction groups instead of two properly merged region groups, because each region's entries are not adjacent to each other in the original list; the computed per-region totals would then be wrong, each reflecting only one transaction instead of the true total for that region.

**Advanced:** Using itertools.islice() and itertools.count(), write a generator function every_nth(iterable, n) that yields every nth item from ANY iterable (starting with the nth item, i.e. index n-1), without loading the entire iterable into memory, and explain why islice() alone (without wrapping it in a loop) cannot directly express 'every nth item, indefinitely' the way this function can.

*Answer:* def every_nth(iterable, n):
    it = iter(iterable)
    while True:
        chunk = list(itertools.islice(it, n))
        if not chunk:
            return
        yield chunk[-1]

print(list(every_nth(range(20), 3)))   # [2, 5, 8, 11, 14, 17]
A single islice() call only ever extracts one bounded slice (a fixed start/stop/step range) from an iterator in one shot; it has no built-in notion of 'repeat this slicing pattern forever across an iterator of unknown or unbounded length'. To get every nth element across the WHOLE iterable, every_nth() repeatedly asks islice() for the next chunk of exactly n items at a time from the same shared iterator it, and yields only the last item of each such chunk (which is exactly the nth, 2nth, 3nth, ... item overall), looping this pattern until islice() eventually returns an empty chunk (signaling the underlying iterator has been fully exhausted), which is precisely the kind of indefinite, until-exhaustion repetition that a single islice() call cannot express by itself, but that wrapping it in an outer generator loop, as shown here, achieves directly.

# Part 26. Iteration and zip()

zip() is one of the most commonly used iteration tools in Python for walking multiple sequences in lockstep. This part covers its lazy behavior, its default short-circuiting on unequal lengths, the strict=True safeguard, and practical data-processing applications.

## 26.1 zip(): Iterating Over Multiple Sequences Together

In [99]:
names = ["Ada", "Bo", "Chi"]
scores = [90, 85, 95]
for name, score in zip(names, scores):
    print(name, score)

Ada

 90
Bo 85
Chi 95


## 26.2 zip() Is Lazy

zip() itself returns an iterator, not a list; it produces each combined tuple only as it is requested, following exactly the same lazy principles as everything else in this handbook.

In [100]:
z = zip([1, 2, 3], ["a", "b", "c"])
print(z)
print(next(z))
print(list(z))


(1, 'a')
[(2, 'b'), (3, 'c')]


## 26.3 zip() and Unequal-Length Iterables: Default Behavior

By default, zip() stops the instant its SHORTEST input iterable is exhausted, silently discarding any remaining, unpaired elements from longer inputs.

In [101]:
names = ["Ada", "Bo", "Chi", "Dumi"]
scores = [90, 85]
result = list(zip(names, scores))
print(result)

[('Ada', 90), ('Bo', 85)]


> **Note:** Notice 'Chi' and 'Dumi' are silently dropped entirely, with no warning or error, because scores ran out first; this default behavior is a common, easy-to-miss source of subtly incorrect data-processing code.

## 26.4 zip(strict=True): Guarding Against Silent Data Loss

Since Python 3.10, passing strict=True makes zip() raise a ValueError immediately if the input iterables turn out to have different lengths, rather than silently truncating; this is strongly recommended whenever the inputs are genuinely expected to always be the same length, and a mismatch would indicate a real bug worth catching loudly.

In [102]:
names = ["Ada", "Bo", "Chi"]
scores = [90, 85]
try:
    list(zip(names, scores, strict=True))
except ValueError as e:
    print("ValueError:", e)

ValueError: zip() argument 2 is shorter than argument 1


## 26.5 Practical Data-Processing Applications

In [103]:
columns = ["name", "score", "passed"]
row_values = ["Ada", 92, True]
record = dict(zip(columns, row_values))
print(record)

list_a = [1, 2, 3]
list_b = [10, 20, 30]
sums = [a + b for a, b in zip(list_a, list_b)]
print(sums)

matrix = [[1, 2, 3], [4, 5, 6]]
transposed = list(zip(*matrix))
print(transposed)

{'name': 'Ada', 'score': 92, 'passed': True}
[11, 22, 33]
[(1, 4), (2, 5), (3, 6)]


| Code / Item | Explanation |
|---|---|
| `dict(zip(columns, row_values))` | A very common, idiomatic pattern: pairing a list of column names with a list of row values to build a labeled record, directly leveraging zip()'s lockstep pairing. |
| `list(zip(*matrix))` | The unpacking operator * spreads each row of matrix as a separate argument to zip(), which then pairs up corresponding elements across all rows, a common, idiomatic way to transpose a simple matrix represented as a list of lists. |

## Part 26 Practice Exercises

**Beginner:** By default, does zip() stop at the shortest or the longest of its input iterables?

*Answer:* The shortest; any extra, unpaired elements from longer iterables are silently discarded by default.

**Intermediate:** What does zip(a, b, strict=True) do differently from plain zip(a, b) when a and b have different lengths?

*Answer:* Plain zip(a, b) silently stops once the shorter of the two is exhausted, discarding the remaining elements of the longer one with no warning. zip(a, b, strict=True) instead raises a ValueError immediately once it detects that the iterables did not have matching lengths, surfacing the mismatch as an explicit, loud error rather than allowing it to pass silently.

**Advanced:** Explain, using a concrete data-processing scenario, why silently losing data via plain zip() on mismatched-length inputs can be more dangerous than an outright crash, and how strict=True changes that risk profile.

*Answer:* Consider pairing a list of customer_ids with a list of corresponding purchase_amounts pulled from two separate database queries that are supposed to always return the same number of rows in the same order; if one query silently returned one fewer row than expected (due to, say, a filtering bug or a race condition), plain zip(customer_ids, purchase_amounts) would simply produce one fewer pair than intended, with NO error, NO warning, and no visible indication that anything went wrong at all, the resulting dataset would just be quietly, incorrectly short by exactly one record, and every purchase amount from that point onward would end up mispaired with the wrong customer entirely, corrupting the rest of the dataset silently rather than only losing the one problematic record. An outright crash, in contrast, is immediately visible and forces investigation before any incorrect results can be used or shipped downstream, which is generally far safer than silent data corruption. zip(customer_ids, purchase_amounts, strict=True) converts this exact silent-corruption scenario into an immediate, loud ValueError the moment the two lists' lengths are found to disagree, which is precisely why strict=True is recommended whenever two iterables being zipped together are expected, by the logic of the program, to always match in length; any mismatch under that assumption represents a genuine bug that deserves to be caught immediately, not silently absorbed.

# Part 27. Iteration and enumerate()

enumerate() pairs each element of an iterable with a running index, and is the idiomatic, preferred alternative to manually maintaining a separate counter variable alongside a loop.

## 27.1 The Manual-Counter Pattern (What to Avoid)

In [104]:
fruits = ["mango", "pawpaw", "banana"]
index = 0
for fruit in fruits:
    print(index, fruit)
    index += 1

0

 mango
1 pawpaw
2 banana


## 27.2 enumerate(): The Idiomatic Replacement

In [105]:
fruits = ["mango", "pawpaw", "banana"]
for index, fruit in enumerate(fruits):
    print(index, fruit)

0 mango
1 pawpaw
2 banana


## 27.3 The start Argument

In [106]:
fruits = ["mango", "pawpaw", "banana"]
for position, fruit in enumerate(fruits, start=1):
    print(f"{position}. {fruit}")

1. mango
2. pawpaw
3. banana


## 27.4 enumerate() Is Lazy

In [107]:
e = enumerate(["a", "b", "c"])
print(e)
print(next(e))
print(list(e))

(0, 'a')
[(1, 'b'), (2, 'c')]


> **Note:** Like zip(), enumerate() returns an iterator, producing each (index, value) pair only as requested, not a pre-built list of pairs.

## 27.5 Why enumerate() Is Preferable to a Manual Counter

- It removes an entire class of off-by-one bugs: forgetting to initialize the counter, forgetting to increment it, or incrementing it in the wrong place relative to the loop body.
- It is more readable: 'for index, item in enumerate(items):' states the intent directly, rather than requiring the reader to trace a separate index variable's lifecycle across the loop.
- It works correctly and lazily on ANY iterable, not just sequences that happen to support indexing; a manual counter alongside 'for item in some_generator:' works too, but enumerate() expresses the exact same intent more directly and idiomatically.
- The start= argument directly handles the extremely common '1-based instead of 0-based numbering' need, without an extra +1 scattered throughout the loop body.

## 27.6 Practical Examples

In [108]:
lines = ["first line", "second line", "third line"]
for line_number, text in enumerate(lines, start=1):
    print(f"Line {line_number}: {text}")

items = ["apple", "banana", "cherry"]
index_map = {item: idx for idx, item in enumerate(items)}
print(index_map)

Line 1: first line
Line 2: second line
Line 3: third line
{'apple': 0, 'banana': 1, 'cherry': 2}


## Part 27 Practice Exercises

**Beginner:** What two things does each value produced by enumerate(my_list) contain?

*Answer:* A running index (an integer, starting at 0 by default) and the corresponding element from the original iterable, paired together as a tuple (index, element).

**Intermediate:** How would you use enumerate() to number a list of report lines starting from 100 instead of 0?

*Answer:* for number, line in enumerate(lines, start=100): ... - the start keyword argument sets the value the running index begins at.

**Advanced:** Explain why 'for index, item in enumerate(some_generator()):' works correctly even though some_generator() has no defined length and does not support indexing, and contrast this with why a manual approach using 'for index in range(len(some_generator())):' would fail entirely.

*Answer:* enumerate() only requires its argument to be iterable, meaning it can produce a sequence of values one at a time via the iteration protocol (Part 4); it never needs to know the total length in advance, nor does it ever use indexing ([]) on its argument at all; internally, it simply pairs each successively produced value with an internally-tracked running counter, incrementing that counter by one each time, which works identically regardless of whether the underlying values come from a list, a file, or a generator. 'range(len(some_generator()))' would fail immediately and completely, for two separate reasons: first, a generator object has no len() defined at all (attempting len() on it raises TypeError, since a generator does not know its own total length in advance, and may not even have a finite one); and second, even if a length were somehow known, using range(n) alongside a SEPARATE call to some_generator() to index into it manually would not work either, since a generator object also does not support indexing ([]) at all, only sequential iteration via next().

# Part 28. Iteration and map(), filter(), and Related Tools

This part focuses specifically on how map(), filter(), zip(), enumerate(), reversed() and sorted() interact with the iteration protocol: which are lazy, which materialize their result immediately, and why. This is not a general tutorial on these functions themselves (that belongs to the companion Functions handbook); the focus here is purely their iteration behavior.

## 28.1 map() and filter(): Lazy Iterators

In [109]:
doubled = map(lambda x: x * 2, [1, 2, 3])
print(doubled)
print(next(doubled))
print(list(doubled))

evens = filter(lambda x: x % 2 == 0, range(10))
print(evens)
print(list(evens))


2
[4, 6]
[0, 2, 4, 6, 8]


| Code / Item | Explanation |
|---|---|
| `map(lambda x: x * 2, [1, 2, 3])` | Returns a map object, a genuine lazy iterator; nothing is computed until values are actually requested from it, exactly like a generator expression. |
| `filter(lambda x: x % 2 == 0, range(10))` | Returns a filter object, likewise a lazy iterator; the predicate function is only ever called on elements as they are individually requested, not all at once up front. |

## 28.2 zip() and enumerate(): Also Lazy

As Parts 26.2 and 27.4 already demonstrated directly, both zip() and enumerate() return lazy iterators too, following the exact same on-demand production model as map() and filter().

## 28.3 reversed(): Lazy, But With a Requirement

reversed() also returns a lazy iterator, producing elements from the end of a sequence backward, one at a time, on demand. Unlike map()/filter()/zip()/enumerate(), which accept any iterable, reversed() specifically requires its argument to support either a __reversed__() method or both __len__() and indexing (i.e., generally a genuine sequence, per Part 18 of this handbook, not just any arbitrary iterable); it cannot reverse a plain, one-directional iterator such as a generator or a file object, since reversing requires knowing where the end is, which an arbitrary lazy iterator does not inherently expose.

In [110]:
r = reversed([1, 2, 3])
print(r)
print(list(r))
try:
    reversed(x for x in range(3))   # a generator has no defined reverse
except TypeError as e:
    print("TypeError:", e)

[3, 2, 1]
TypeError: 'generator' object is not reversible


## 28.4 sorted(): Eager, Not Lazy

sorted() is a deliberate, important exception among these tools: it is EAGER, not lazy. Sorting fundamentally requires seeing every element before any output can be produced at all (you cannot know an element belongs first in sorted order without having compared it against, or at least accounted for, the rest), so sorted() always fully consumes its input and returns a complete, materialized list, never a lazy iterator.

In [111]:
gen = (x for x in [3, 1, 2])
result = sorted(gen)
print(type(result))
print(result)

# Proof that sorted() FULLY consumes its generator input:
gen2 = (x for x in [5, 4])
sorted(gen2)
print(list(gen2))   # []: gen2 was completely drained by sorted() already

<class 'list'>
[1, 2, 3]
[]


## 28.5 Laziness Summary

| Function | Lazy or Eager? | Requirement on Its Input | Notes |
|---|---|---|---|
| map() | Lazy | Any iterable | Function applied one element at a time, on demand. |
| filter() | Lazy | Any iterable | Predicate applied one element at a time, on demand. |
| zip() | Lazy | Any iterables | Stops at shortest by default (Part 26). |
| enumerate() | Lazy | Any iterable | Pairs a running counter with each value. |
| reversed() | Lazy (in production), but needs upfront structure | A true sequence (or an object with __reversed__()) | Cannot reverse an arbitrary one-way iterator like a generator. |
| sorted() | EAGER | Any iterable | Must consume the ENTIRE input before producing any output; always returns a full list. |

## Part 28 Practice Exercises

**Beginner:** Which of map(), filter(), sorted() is the one exception that does NOT behave lazily?

*Answer:* sorted(); it always fully consumes its input immediately and returns a complete list, unlike map() and filter(), which are both lazy.

**Intermediate:** Why can't reversed() be used directly on a plain generator expression, such as reversed(x for x in range(5))?

*Answer:* reversed() requires its argument to either define a __reversed__() method or to be a genuine sequence supporting both len() and indexing, specifically because producing the LAST element first requires knowing where the sequence ends, information a plain, one-directional generator does not expose at all (it only knows how to produce its next value, never its length or its end, in advance); attempting to call reversed() on a generator raises TypeError as a direct result.

**Advanced:** Explain, using the specific requirement that sorting must see every element before producing any output, why sorted() cannot be implemented as a lazy iterator the way map() or filter() can, even in principle.

*Answer:* map() and filter() can each decide, and produce, their very next output value based ENTIRELY on the next single input value alone (applying a function to it, or testing a predicate on it), which is exactly why they can hand back one result immediately upon request without needing to see anything else first. Sorting is fundamentally different: determining which element belongs FIRST in the sorted output requires comparing it, directly or indirectly, against every other element in the entire input, since any element not yet seen could turn out to belong before the ones already examined; there is no way to guarantee an element's correct final position in a sorted sequence without having accounted for the complete input first. Because of this, no correct sorting algorithm can produce even its very first output element without first having consumed the entire input, which means sorted() is fundamentally incompatible with true, incremental, on-demand laziness as map() or filter() implement it; it can, at best, be eager (materializing the full input and full output before returning), which is exactly what Python's sorted() does.

# Part 29. Iteration and Comprehensions

Every comprehension (list, set, dict) and generator expression is built on the exact same iteration protocol as a for loop, since a comprehension is, at its core, syntactic sugar over iteration too. This part compares all four directly, with a focus on memory and performance.

## 29.1 The Four Comprehension Forms

In [112]:
nums = range(6)
list_comp = [x * x for x in nums]
set_comp = {x % 3 for x in nums}
dict_comp = {x: x * x for x in nums}
gen_exp = (x * x for x in nums)
print(list_comp)
print(set_comp)
print(dict_comp)
print(gen_exp)

[0, 1, 4, 9, 16, 25]
{0, 1, 2}
{0: 0, 1: 1, 2: 4, 3: 9, 4: 16, 5: 25}
<generator object <genexpr> at 0x7f913041ecf0>


## 29.2 What Every Comprehension Has in Common

Every one of these four forms iterates over its source using exactly the iteration protocol from Part 4: it obtains an iterator, repeatedly calls next() on it, and stops on StopIteration; the only difference between the four is what container (if any) is used to collect the resulting values, and whether that collection happens immediately (eager) or on demand (lazy).

## 29.3 List, Set and Dict Comprehensions Are All Eager

All three of list, set and dict comprehensions fully consume their source iterable immediately and build their entire respective container in memory before the comprehension expression finishes evaluating; they behave exactly like the equivalent list()/set()/dict() constructor call wrapped around a generator expression, in terms of eagerness.

## 29.4 Only the Generator Expression Is Lazy

The generator expression, using ( ) instead of [ ], { }, is the one form that defers computation, producing values only as they are requested, exactly as covered throughout Part 12.

## 29.5 Comprehension vs Generator Expression: Memory and Performance

In [113]:
import sys
n = 500_000
list_form = [x for x in range(n)]
gen_form = (x for x in range(n))
print("list comprehension:", sys.getsizeof(list_form), "bytes")
print("generator expression:", sys.getsizeof(gen_form), "bytes")

list comprehension:

 4167352 bytes
generator expression: 192 bytes


> **Note:** This is exactly the same measured contrast established in Part 12.4 and Part 13.1; comprehensions inherit the identical memory trade-off, since a list/set/dict comprehension is, functionally, 'a generator expression immediately and fully consumed into a concrete container'.

## 29.6 When to Choose Which Comprehension Form

- List comprehension: when order matters, duplicates are meaningful, and you need indexing, len(), or multiple passes over the result.
- Set comprehension: when you specifically need uniqueness and fast membership testing, and order/duplicates do not matter.
- Dict comprehension: when you are building a key-to-value mapping from some source data.
- Generator expression: when the result will only ever be consumed once, in a single pass, especially for large or unbounded source data, per the guidance in Part 12.5.

## Part 29 Practice Exercises

**Beginner:** Which of the four comprehension forms covered in this part is lazy rather than eager?

*Answer:* The generator expression, written with parentheses ( ).

**Intermediate:** Why do list, set and dict comprehensions all count as 'built on the iteration protocol', even though none of them use the word 'iterator' explicitly in their syntax?

*Answer:* Every comprehension form works by iterating over its source clause ('for x in some_iterable'), which under the hood follows exactly the same iter()/next()/StopIteration mechanism as an ordinary for loop (Part 3.6); the comprehension syntax is simply a compact, expression-based way of writing 'iterate over this source and collect each produced value into a container', with the specific container type (list, set, or dict) determined by which brackets are used, but the underlying act of walking through the source is identical to any other form of iteration in Python.

**Advanced:** A dataset of one million records needs to be checked for uniqueness of a particular 'email' field, and then, separately, an average needs to be computed over a 'score' field, but the raw records themselves are never needed again afterward. Using the guidance from 29.6, justify the most memory-appropriate comprehension/expression choice for each of these two separate tasks.

*Answer:* For checking uniqueness of the 'email' field, a set comprehension is the appropriate choice: 'unique_emails = {r["email"] for r in records}', because the task specifically requires uniqueness (duplicates should collapse together) and fast membership testing is exactly what a set is optimized for; a set comprehension also conveniently allows a follow-up check like 'len(unique_emails) == len(records)' to detect whether any duplicates existed at all. For computing the average of the 'score' field, a generator expression is the more memory-appropriate choice: 'total = sum(r["score"] for r in records); average = total / len(records)' (or, if records itself is also a one-time-use generator rather than a reusable list, computing len alongside the sum in a single pass), because this computation only needs to see each score value exactly once, in a single pass through sum(), with no need to retain any intermediate collection of scores afterward; using a list comprehension here ("[r['score'] for r in records]") would needlessly materialize and hold a full list of a million numeric values in memory purely to immediately sum them once and then discard the list entirely, which is exactly the wasteful eager-evaluation-for-a-single-pass-consumer pattern that Part 12.5's rule of thumb and Part 13's memory analysis both caution against.

# Part 30. Iteration With Dictionaries

This part focuses specifically on how iteration interacts with dict: the three view methods, their lazy-like, live behavior, and the important, common error of modifying a dictionary while iterating over it.

## 30.1 Iterating Over Keys (the Default)

In [114]:
student = {"name": "Ada", "age": 21, "active": True}
for key in student:          # iterates over KEYS by default
    print(key)
for key in student.keys():   # identical, but explicit
    print(key)

name
age
active
name
age
active


## 30.2 Iterating Over Values

In [115]:
student = {"name": "Ada", "age": 21, "active": True}
for value in student.values():
    print(value)

Ada
21
True


## 30.3 Iterating Over Items (Key-Value Pairs)

In [116]:
student = {"name": "Ada", "age": 21, "active": True}
for key, value in student.items():
    print(key, "->", value)

name -> Ada
age -> 21
active -> True


## 30.4 Dictionary Views: Live, Lazy-Like Windows Into the Dict

student.keys(), student.values() and student.items() do not return lists or copies; they return dictionary view objects, which are lazy-like, dynamic windows onto the dictionary's current, live contents. They do not materialize a separate collection up front; instead, they reflect the dictionary's actual current state every time they are inspected or iterated.

In [117]:
student = {"name": "Ada", "age": 21}
keys_view = student.keys()
print(keys_view)
student["active"] = True     # mutate the dict AFTER creating the view
print(keys_view)              # the view reflects the change automatically

dict_keys(['name', 'age'])
dict_keys(['name', 'age', 'active'])


| Code / Item | Explanation |
|---|---|
| `student["active"] = True followed by print(keys_view)` | The view shows 'active' too, even though it was added AFTER keys_view was created; this proves the view is a live, dynamic reflection of the dict's current state, not a frozen snapshot taken at the moment .keys() was called. |

## 30.5 Modifying a Dictionary During Iteration: The Classic Error

In [118]:
student = {"a": 1, "b": 2, "c": 3}
try:
    for key in student:
        if key == "b":
            del student[key]   # DANGEROUS: mutating the dict's size while iterating over it
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError:

 dictionary changed size during iteration


> **Note:** Python explicitly detects and raises RuntimeError: dictionary changed size during iteration when a dict's SIZE changes (a key added or removed) while a loop is actively iterating over it, specifically because doing so could otherwise silently corrupt the iteration or skip/repeat entries; this protection applies to iterating over the dict directly, its .keys(), and its .items() alike.

## 30.6 The Safe Approach: Iterate Over a Snapshot, Then Mutate

In [119]:
student = {"a": 1, "b": 2, "c": 3}
for key in list(student.keys()):   # list() takes a SNAPSHOT copy, safe to mutate the original during
    if key == "b":
        del student[key]
print(student)

{'a': 1, 'c': 3}


| Code / Item | Explanation |
|---|---|
| `for key in list(student.keys()):` | Wrapping the view in list() immediately materializes a separate, independent COPY of the current keys, at that moment; the subsequent loop iterates over that safe, static copy, so deleting from the original dict mid-loop no longer affects what the loop itself is iterating over, avoiding the RuntimeError entirely. |

## 30.7 A Safer Alternative: Build a New Dict Instead of Mutating in Place

In [120]:
student = {"a": 1, "b": 2, "c": 3}
filtered = {k: v for k, v in student.items() if k != "b"}
print(filtered)

{'a': 1, 'c': 3}


> **Note:** Often the cleanest, most idiomatic fix is to avoid mutating the original dictionary during iteration altogether, and instead build a brand-new dictionary (via a dict comprehension, Part 29) containing only the desired entries, which sidesteps the entire class of mutate-while-iterating problem.

## Part 30 Practice Exercises

**Beginner:** By default, does 'for x in some_dict:' iterate over the dictionary's keys, values, or items?

*Answer:* Its keys.

**Intermediate:** Why does adding a new key to a dict WHILE actively iterating over it raise a RuntimeError, but reassigning the VALUE of an EXISTING key during iteration does not?

*Answer:* Reassigning an existing key's value does not change the dictionary's SIZE or its set of keys at all; the internal structure the iteration is walking through remains unchanged in shape, so the iterator's position and expected entries remain valid throughout. Adding a new key (or deleting one) changes the dictionary's size, which can invalidate the internal state the iterator relies on to correctly walk through the dictionary's entries without skipping or repeating any, so Python proactively detects this specific kind of size-changing mutation during iteration and raises RuntimeError rather than risk silently producing incorrect, inconsistent iteration results.

**Advanced:** Explain why 'for key in list(student.keys()): ... del student[key]' is safe, while 'for key in student.keys(): ... del student[key]' (without list()) is not, in terms of what a dictionary view actually is.

*Answer:* student.keys() (without wrapping it in list()) returns a dictionary VIEW object, a live, dynamic window that reflects the dictionary's actual current state at every point during iteration, not a fixed, independent snapshot; when the loop deletes a key from student mid-iteration, that same live view immediately reflects the dictionary's new, smaller size and altered key set, which is exactly the size-changing mutation that Python's RuntimeError protection (30.5) is designed to catch, since the view being iterated over is, in effect, the dictionary itself changing shape out from under the ongoing iteration. Wrapping the call in list(student.keys()) instead immediately and fully materializes a separate, independent, ordinary list containing a COPY of whatever keys existed at that exact moment; once created, this list is entirely disconnected from the original dictionary, so subsequent deletions from student have no effect on it whatsoever, and the loop safely iterates over this static, unchanging copy while del student[key] freely mutates the real, separate dictionary object underneath, with no conflict between the two.

# Part 31. Iteration With Files

This part explains precisely why file objects are iterable, exactly what happens internally during 'for line in file:', and works through large-file processing patterns. Opening, reading and writing files themselves belong to the companion File Handling handbook; this part is purely about the iteration layer.

## 31.1 Why File Objects Are Iterable

An open file object, in text mode, implements the iterator protocol directly: it has both __iter__() (returning itself, exactly like the custom iterators in Part 3.4) and __next__(), which reads and returns the next line from the underlying file each time it is called, raising StopIteration once the end of the file is reached. This is precisely why a file object can be used directly in a for loop, with no explicit .readlines() or manual line-splitting required.

## 31.2 'for line in file:' Explained Internally

In [121]:
with open("demo_temp.txt", "w") as f:
    f.write("first\nsecond\nthird\n")

with open("demo_temp.txt") as f:
    it = iter(f)         # a file object's __iter__() simply returns itself
    print(it is f)
    print(next(it))       # 'first\n': manually pulling the first line
    print(next(it))       # 'second\n'

import os
os.remove("demo_temp.txt")

True
first

second



| Code / Item | Explanation |
|---|---|
| `it is f` | True: exactly like a custom iterator class (Part 3.4), a file object's own __iter__() method returns itself, since the file object IS its own iterator, not merely an iterable that produces a separate iterator. |
| `next(it) returning 'first\n'` | Confirms directly that a file object's __next__() returns one line at a time, including the trailing newline character, which is why .strip() is so commonly applied to each line during processing. |

## 31.3 The Equivalent Manual Loop

In [122]:
with open("demo_temp2.txt", "w") as f:
    f.write("a\nb\nc\n")

with open("demo_temp2.txt") as f:
    while True:
        try:
            line = next(f)
        except StopIteration:
            break
        print(repr(line))

import os
os.remove("demo_temp2.txt")

'a\n'
'b\n'
'c\n'


> **Note:** This is exactly the same five-step for-loop equivalence established generally in Part 3.6, applied specifically to a file object; 'for line in file:' is genuine syntactic sugar over precisely this iter()/next()/StopIteration pattern, with no separate, special file-reading mechanism involved.

## 31.4 Large-File Processing: Constant Memory, Regardless of File Size

In [123]:
with open("demo_large.txt", "w") as f:
    for i in range(50_000):
        f.write(f"record {i}\n")

def count_matching_lines(path, substring):
    count = 0
    with open(path) as f:
        for line in f:            # one line in memory at a time, regardless of total file size
            if substring in line:
                count += 1
    return count

print(count_matching_lines("demo_large.txt", "9"))

import os
os.remove("demo_large.txt")

17195


> **Note:** count_matching_lines() uses exactly constant memory (one line's worth, plus a small integer counter) whether the file it is given has 50 lines or 50 billion lines, precisely because iterating a file object never loads more than the current line into memory, exactly mirroring the memory analysis from Part 13 applied specifically to files, and directly enabling everything demonstrated in Part 15.

## 31.5 Combining File Iteration With Generators for Reusable Pipelines

In [124]:
def read_and_strip(path):
    with open(path) as f:
        for line in f:
            yield line.strip()

with open("demo_pipeline.txt", "w") as f:
    f.write("  padded line 1  \n line 2\n\nline 4\n")

non_empty_lines = (line for line in read_and_strip("demo_pipeline.txt") if line)
for line in non_empty_lines:
    print(repr(line))

import os
os.remove("demo_pipeline.txt")

'padded line 1'
'line 2'
'line 4'


| Code / Item | Explanation |
|---|---|
| `read_and_strip(path)` | A generator function that WRAPS the file object's own built-in iteration, adding a transformation step (stripping whitespace) to every line; this is precisely the generator-based transformation pipeline pattern from Part 16.1, applied directly to a real file. |

## Part 31 Practice Exercises

**Beginner:** What two dunder methods does an open text-mode file object implement that make it directly usable in a for loop?

*Answer:* __iter__() (which returns the file object itself) and __next__() (which reads and returns the next line, raising StopIteration at the end of the file).

**Intermediate:** Why does next(f), where f is an open file object, return a string that typically ends with a newline character (\n)?

*Answer:* A file object's __next__() implementation reads up to and including the next newline character in the underlying file each time it is called, returning that entire line, newline included, as the produced value; this is simply how line-based reading is defined for text-mode file iteration, which is exactly why calling .strip() (or .rstrip('\n')) on each line is such a common, near-automatic step in code that processes file lines, to remove that trailing newline before further processing.

**Advanced:** Explain why count_matching_lines(path, substring) in 31.4 uses a constant, small amount of memory regardless of whether the target file contains 50 lines or 50 million lines, contrasting this directly with what would happen if the function instead began with 'lines = open(path).readlines()'.

*Answer:* Because count_matching_lines() iterates directly over the open file object f with 'for line in f:', at any given moment during execution only the single line currently being examined (plus the small integer counter) needs to exist in memory; each line is read from disk, checked, and then can be discarded (garbage collected) before the very next line is even read, exactly mirroring the constant-memory, lazy-evaluation pattern established generally in Part 13 and Part 14. If the function instead began with 'lines = open(path).readlines()', it would force Python to read the ENTIRE file from disk immediately and hold every single line simultaneously as one large list in memory before the counting loop even begins; for a 50-line file this difference is negligible, but for a 50-million-line file, readlines() would need to allocate memory proportional to the entire file's size all at once, which could easily exhaust available memory or simply be dramatically slower and more wasteful than necessary for a task that, by its nature, only ever needs to look at one line at a time to compute its final count.

# Part 32. Async Iterators and Async Generators

Everything so far in this handbook has covered synchronous iteration: each next() call runs, blocking, until it produces a value. Asynchronous iteration extends the same conceptual protocol to situations where producing the next value might require WAITING (for a network response, a file operation, a timer) without blocking the entire program while it waits. This part introduces async iteration precisely, building directly on the synchronous protocol already mastered.

## 32.1 Why Async Iteration Exists

A synchronous __next__() call must run to completion, including any waiting, before control returns to the caller; if producing the next value requires waiting on a slow network response, the entire program is blocked, unable to do anything else, for that entire wait. Async iteration allows a coroutine (an async function) to pause specifically at points where it is genuinely waiting on something external, letting Python's event loop run OTHER code during that wait, then resume the iteration once the awaited operation completes.

## 32.2 The Async Iteration Protocol: __aiter__ and __anext__

The async iteration protocol directly mirrors the synchronous one from Part 4, with async-aware counterparts: __aiter__(self) (analogous to __iter__()) must return an async iterator, and __anext__(self) (analogous to __next__()) must be an async method (a coroutine) that returns the next value, or raises StopAsyncIteration (analogous to StopIteration) once exhausted.

## 32.3 async for: The Async Equivalent of a for Loop

async for x in async_iterable: is the direct async counterpart of a regular for loop, and can only be used inside an async def function. Internally, it performs exactly the synchronous for-loop pattern from Part 3.6, but using __aiter__()/__anext__()/StopAsyncIteration/await instead of their synchronous counterparts.

## 32.4 A Custom Async Iterator Class

In [125]:
import asyncio

class AsyncCounter:
    def __init__(self, limit):
        self.current = 0
        self.limit = limit

    def __aiter__(self):
        return self

    async def __anext__(self):
        if self.current >= self.limit:
            raise StopAsyncIteration
        await asyncio.sleep(0)   # simulates a real async wait (e.g. network I/O)
        value = self.current
        self.current += 1
        return value

async def main():
    async for number in AsyncCounter(3):
        print(number)

await main()

0
1
2


| Code / Item | Explanation |
|---|---|
| `def __aiter__(self): return self` | Directly mirrors the synchronous convention from Part 3.4: an async iterator's own __aiter__() returns itself. |
| `async def __anext__(self): ...` | Defined with async def, making it a coroutine; this is what allows it to genuinely 'await' something (a real network call, a database query) without blocking the rest of the program while doing so. |
| `await asyncio.sleep(0)` | Stands in for any real asynchronous wait; in production code this would typically be an awaited network request, database call, or similar I/O operation instead. |

> **Note:** This example uses 'await main()' directly, assuming a context (such as a Jupyter notebook, which has a running event loop already) that supports top-level await; in a plain .py script, this would instead be run via 'asyncio.run(main())'.

## 32.5 Async Generators: The Convenient Alternative

Exactly as a synchronous generator function (Part 9) is the convenient, low-boilerplate alternative to a hand-written custom iterator class, an async generator function, an async def function containing yield, is the convenient alternative to a hand-written async iterator class. Calling it returns an async generator object directly, without needing to write __aiter__()/__anext__() by hand at all.

In [126]:
import asyncio

async def async_counter(limit):
    current = 0
    while current < limit:
        await asyncio.sleep(0)
        yield current
        current += 1

async def main2():
    async for number in async_counter(3):
        print(number)

await main2()

0
1
2


| Code / Item | Explanation |
|---|---|
| `async def async_counter(limit): ... yield current` | The combination of async def with a yield statement inside is what specifically makes this an ASYNC generator function; calling it produces an async generator object directly, with __aiter__/__anext__ handled automatically, exactly paralleling how a plain generator function (Part 9) automatically satisfies the synchronous iterator protocol. |

## 32.6 StopAsyncIteration

StopAsyncIteration is the async-protocol counterpart to StopIteration (Part 6): it signals 'no more values' for an async iterator, and, exactly like its synchronous counterpart, is normally invisible, since 'async for' catches it internally, exactly mirroring how a plain for loop silently catches StopIteration.

## 32.7 Practical Applications

- APIs and network streams: consuming a paginated API response, or a server-sent-events / WebSocket stream, where each 'next item' genuinely requires waiting on network I/O, without blocking other work the program might be doing concurrently.
- AI applications: consuming a streamed model response token-by-token from an async API client, exactly the pattern previewed conceptually in Part 18.1, now with the actual async mechanism that makes non-blocking waiting for each token possible.
- Real-time data: processing a live feed of sensor readings, market data ticks, or log events arriving asynchronously, where the producer's timing is not under the consumer's control.
- Any I/O-bound pipeline where synchronous iteration would otherwise force the entire program to sit idle, blocked, while waiting for each next piece of data to arrive.

## 32.8 Synchronous vs Asynchronous Iteration Protocol

| Concept | Synchronous | Asynchronous |
|---|---|---|
| Loop construct | for x in iterable: | async for x in async_iterable: (inside async def) |
| Iterable method | __iter__() | __aiter__() |
| Advance method | __next__() | async __anext__() (a coroutine) |
| Exhaustion signal | StopIteration | StopAsyncIteration |
| Generator function | def f(): ... yield x | async def f(): ... yield x |

## Part 32 Practice Exercises

**Beginner:** What is the async counterpart of the StopIteration exception?

*Answer:* StopAsyncIteration.

**Intermediate:** What two things must be true about a function's definition for calling it to produce an async generator object?

*Answer:* It must be defined with async def (making it a coroutine function), AND its body must contain at least one yield statement; async def alone (with no yield) defines an ordinary coroutine function, not an async generator, and yield alone (without async def) defines an ordinary synchronous generator function, not an async one.

**Advanced:** Explain, using the concept of blocking versus non-blocking waits, why a synchronous generator with 'time.sleep(1)' inside its loop and an async generator with 'await asyncio.sleep(1)' inside its loop behave very differently in a larger program that is also trying to do other concurrent work while the generator produces its values.

*Answer:* A synchronous generator's __next__() call runs, uninterrupted, until it either yields a value or finishes; time.sleep(1) inside it BLOCKS the entire single-threaded program for that full second, during which nothing else in the program, no other task, no other coroutine, can make any progress at all, since the whole interpreter is simply paused waiting on that sleep call. An async generator's __anext__() is itself a coroutine; when it executes 'await asyncio.sleep(1)', it does not block the whole program, it instead yields control back to the async event loop for that second, explicitly signaling 'I am waiting; feel free to run other pending tasks in the meantime', which allows the event loop to work on other coroutines, handle other async iterators, or respond to other events during that same second, only resuming this particular generator once its one-second wait has genuinely elapsed. This is the entire practical point of async iteration: it lets a single-threaded Python program remain productive and responsive while individual pieces of work are waiting on something external, whereas the synchronous version has no such mechanism and simply halts everything during any blocking wait.

# Part 33. Iterators and Performance

This part directly addresses a common misconception flagged already in Part 14.5: that generators/iterators are simply, automatically 'faster'. Their real, guaranteed advantage is memory efficiency; this part measures actual execution time in several scenarios to show precisely when generators help, when they do not, and how to benchmark iteration code yourself.

## 33.1 What Generators Do and Do Not Guarantee

Memory usage: GUARANTEED to be lower (or equal) than the eager equivalent for large or unbounded data, per Part 13. Execution speed: NOT guaranteed to be faster at all; it depends heavily on the specific consumption pattern, as the rest of this part demonstrates with real measurements.

## 33.2 Benchmarking Setup: timeit

Python's standard library timeit module is the standard, reliable tool for measuring small code snippets' execution time, running them repeatedly to smooth out noise from other system activity.

In [127]:
import timeit

def using_list():
    return sum([x * x for x in range(10_000)])

def using_generator():
    return sum(x * x for x in range(10_000))

list_time = timeit.timeit(using_list, number=100)
gen_time = timeit.timeit(using_generator, number=100)
print(f"List comprehension: {list_time:.4f}s")
print(f"Generator expression: {gen_time:.4f}s")

List comprehension: 0.0409s
Generator expression: 0.0413s


## 33.3 Scenario 1: Small, Fully-Consumed Data (Generators Can Be Slightly Slower)

For a small, fixed-size sequence that will be fully consumed exactly once regardless of approach, a generator's per-value pause/resume overhead can make it marginally SLOWER in wall-clock time than a straightforward list comprehension, even though both use essentially the same peak memory for a small n; the code in 33.2 typically demonstrates this directly.

## 33.4 Scenario 2: Only Partially Consuming a Large Sequence (Generators Win Decisively)

In [128]:
import timeit

def find_first_eager(n, target):
    values = [x * x for x in range(n)]   # computes ALL n squares, even though most are unused
    for v in values:
        if v > target:
            return v

def find_first_lazy(n, target):
    values = (x * x for x in range(n))   # computes squares ONE AT A TIME, stops as soon as found
    for v in values:
        if v > target:
            return v

n = 5_000_000
target = 100   # found almost immediately; the vast majority of squares are never needed

eager_time = timeit.timeit(lambda: find_first_eager(n, target), number=5)
lazy_time = timeit.timeit(lambda: find_first_lazy(n, target), number=5)
print(f"Eager (builds full list first): {eager_time:.4f}s")
print(f"Lazy (stops as soon as found): {lazy_time:.4f}s")

Eager (builds full list first): 2.2355s
Lazy (stops as soon as found): 0.0000s


> **Note:** Here the lazy generator version is typically dramatically faster, not merely more memory-efficient, because it never wastes any time computing the millions of squared values that are never actually needed once a match is found early; the eager version wastes both memory AND time building the entire, mostly-unused list first.

## 33.5 Scenario 3: Memory Pressure Causing System-Level Slowdown

For genuinely huge datasets that approach or exceed available RAM, the eager, list-based approach can become dramatically slower than a generator-based approach not due to Python-level overhead at all, but due to operating-system-level effects: excessive memory allocation, garbage collection pressure, or the operating system resorting to disk-backed virtual memory (swapping), any of which can slow a program down by orders of magnitude, far beyond any difference attributable to generator pause/resume overhead itself.

## 33.6 Generator Overhead: Where It Comes From

Each individual next() call on a generator involves resuming a suspended Python frame, which carries genuine, measurable per-call overhead compared to a tight loop appending directly into a list in a single, uninterrupted pass. This overhead is normally negligible in absolute terms (nanoseconds to low microseconds per call), and becomes a meaningful factor mainly when a very large NUMBER of individual next() calls happen in a hot, performance-critical loop, and the entire sequence is fully, eagerly needed anyway.

## 33.7 Practical Guidance: When Each Approach Wins

- Use a generator/generator expression when: the data is large or unbounded, only part of the sequence will actually be consumed (early termination, break, islice()), or memory constraints are a genuine concern, per Part 13.
- A list comprehension may be equally fast or faster when: the data is small, the ENTIRE sequence will be consumed regardless, and the result needs to be accessed multiple times or randomly indexed (which would otherwise require converting a generator to a list anyway, incurring the exact same memory cost eventually).
- When in doubt for genuinely performance-sensitive code, benchmark with timeit or a proper profiler on YOUR actual data and access pattern, rather than assuming either approach is universally faster; Parts 33.3 and 33.4 show directly how the same general shape of code can favor either approach depending entirely on the consumption pattern.

## Part 33 Practice Exercises

**Beginner:** Is it accurate to say 'generators are always faster than lists'?

*Answer:* No. Generators guarantee lower or equal memory usage for large or unbounded data; they do not guarantee faster execution time, and can sometimes be marginally slower for small, fully-consumed sequences due to per-value pause/resume overhead.

**Intermediate:** In the find_first_eager() vs find_first_lazy() comparison (33.4), explain specifically WHY the lazy version is typically so much faster here, in terms of wasted work, not just wasted memory.

*Answer:* find_first_eager() unconditionally computes ALL n squared values via a list comprehension before it ever begins searching for one that exceeds the target, even though, in this scenario, a qualifying value is found almost immediately (target=100 is a small threshold), meaning the vast majority of those computed squares are never even inspected, let alone used, representing pure wasted computational work, not merely wasted memory. find_first_lazy() computes each squared value one at a time, checking it against the target immediately, and stops computing ANY further values the instant a match is found; it therefore performs only as much computation as is genuinely necessary to find the answer, skipping the computation of every squared value that would come after the match entirely, which is why it saves real, measurable execution time here, not only memory.

**Advanced:** A colleague benchmarks 'sum(x for x in range(1000))' against 'sum([x for x in range(1000)])' a thousand times each, finds the generator expression is very slightly slower on average, and concludes 'generators are basically pointless, we should always use list comprehensions'. Explain what is wrong with generalizing this specific benchmark result into that broad conclusion.

*Answer:* The benchmark described tests exactly the scenario identified in 33.3 as the ONE case where a list comprehension can have a genuine speed edge: a small (1000-element), fully-consumed-in-a-single-pass sequence, with the result never needed more than once and never partially discarded early; in that narrow scenario, the generator's small per-value overhead is not offset by any memory or wasted-work savings, since the list version also only ever computes exactly 1000 values and only holds a small amount of memory regardless. This result says nothing at all about the very different scenarios covered in 33.4 (early termination on a huge, mostly-unneeded sequence, where lazy evaluation avoids massive wasted computation) or Part 13 (datasets too large to fit comfortably in memory as a full list, where the eager version might not even complete successfully at all, or might trigger the severe memory-pressure slowdowns described in 33.5). Generalizing one narrow benchmark, small input, full consumption, single pass, into a universal rule ignores the entire class of scenarios (large or unbounded data, partial/early consumption, genuine memory constraints) where generators provide decisive, sometimes dramatic advantages in both memory and, as 33.4 demonstrates, sometimes execution time as well; the correct, defensible conclusion is the more nuanced guidance in 33.7, not a blanket dismissal of generators.

# Part 34. Common Iteration Mistakes

This part catalogs the ten most common iteration-related mistakes in real Python code, each with incorrect code, an explanation of exactly what goes wrong, corrected code, and the underlying principle that prevents it going forward.

## 34.1 Mistake: Confusing an Iterable With an Iterator

In [129]:
numbers = [1, 2, 3]
try:
    next(numbers)   # WRONG: a list is an ITERABLE, not an ITERATOR
except TypeError as e:
    print("TypeError:", e)

it = iter(numbers)   # CORRECT: explicitly get an iterator first
print(next(it))

TypeError: 'list' object is not an iterator
1


> **Note:** Problem: next() requires an ITERATOR (something with __next__()); a list only implements __iter__(), not __next__() (Part 2 vs Part 3). Solution: always call iter() on an iterable first if you intend to call next() on the result directly.

## 34.2 Mistake: Calling next() Incorrectly (Wrong Number of Arguments, or on the Wrong Object)

In [130]:
gen = (x for x in range(3))
try:
    next(gen, 0, 0)   # WRONG: next() accepts at most 2 arguments
except TypeError as e:
    print("TypeError:", e)

print(next(gen, 0))   # CORRECT: iterator, then optional default

TypeError: next expected at most 2 arguments, got 3
0


## 34.3 Mistake: Exhausting an Iterator, Then Being Surprised It's Empty

In [131]:
gen = (x for x in range(3))
first_pass = list(gen)
second_pass = list(gen)   # WRONG assumption: expecting the same 3 values again
print(first_pass, second_pass)

[0, 1, 2] []


> **Note:** Problem: an iterator (including a generator) cannot be replayed once exhausted (Part 4.2, Part 12.2). Solution: if you need to iterate multiple times, store the results in a list/tuple the first time, or recreate a fresh generator/iterator for each pass.

## 34.4 Mistake: Reusing an Already-Exhausted Iterator Object Across Functions

In [132]:
def process(data_iterator):
    return sum(data_iterator)

shared_gen = (x for x in range(5))
total1 = process(shared_gen)
total2 = process(shared_gen)   # WRONG: shared_gen is already exhausted from the first call
print(total1, total2)

10 0


> **Note:** Problem: passing the SAME generator object into two separate calls means the second call receives an already-drained iterator. Solution: pass a fresh generator (or the original iterable, recreated) into each call, e.g. 'process((x for x in range(5)))' twice, or pass the underlying list/range directly if repeated iteration is genuinely needed.

## 34.5 Mistake: An Infinite Generator With No Termination Guard

In [133]:
def infinite_counter():
    i = 0
    while True:
        yield i
        i += 1

# WRONG (would hang forever if actually run): list(infinite_counter())

import itertools
safe_result = list(itertools.islice(infinite_counter(), 5))   # CORRECT: explicitly bounded
print(safe_result)

[0, 1, 2, 3, 4]


## 34.6 Mistake: Modifying a List While Iterating Over It

In [134]:
numbers = [1, 2, 3, 4, 5]
for n in numbers:
    if n % 2 == 0:
        numbers.remove(n)   # WRONG: mutating the list's SIZE while iterating over it
print(numbers)   # SILENTLY WRONG: skips some elements, no error raised at all

[1, 3, 5]


| Code / Item | Explanation |
|---|---|
| `numbers.remove(n) inside the loop` | Because a list iterator tracks position by numeric index, removing an element shifts every later element one position earlier, causing the NEXT index the iterator visits to skip over an element entirely; unlike dict (Part 30.5), this does NOT raise an error, it just silently produces an incorrect result. |

> **Note:** Correct fix: iterate over a copy, or build a new list instead: 'numbers = [n for n in numbers if n % 2 != 0]', or 'for n in list(numbers): ... numbers.remove(n)', mirroring the safe-snapshot pattern from Part 30.6.

## 34.7 Mistake: Excessive, Hard-to-Follow Generator Nesting

In [135]:
def stage1(it):
    for x in it: yield x + 1
def stage2(it):
    for x in it: yield x * 2
def stage3(it):
    for x in it: yield x - 3
def stage4(it):
    for x in it: yield x ** 2

# WORKS, but is hard to read and debug when something goes wrong deep inside:
result = list(stage4(stage3(stage2(stage1(range(3))))))
print(result)

[1, 1, 9]


> **Note:** Problem: deeply nested generator chains are not incorrect, but can become genuinely difficult to read, debug, and trace an error through, since a bug inside stage2 might only surface as an unexpected value several stages later. Solution: for long pipelines, consider naming each intermediate stage as a separate variable ('step1 = stage1(range(3)); step2 = stage2(step1); ...'), or clearly documenting each stage's contract, to keep the pipeline debuggable.

## 34.8 Mistake: Misunderstanding Lazy Evaluation (Assuming Work Happens at Definition Time)

In [136]:
def get_expensive_data():
    print("Computing expensive data...")
    return (x ** 2 for x in range(3))

gen = get_expensive_data()   # 'Computing expensive data...' DOES print (the function itself ran)
print("Generator created")
result = list(gen)             # but the (x**2 for x...) expression's VALUES are only computed HERE
print(result)

Computing expensive data...
Generator created
[0, 1, 4]


> **Note:** Subtlety: calling get_expensive_data() DOES run its own print statement immediately (it is an ordinary function, not a generator function, since it uses return, not yield); it is specifically the generator EXPRESSION it returns whose individual values remain uncomputed until actually consumed. Conflating 'the function ran' with 'all its lazy contents were computed' is a common source of confusion; be precise about exactly which layer is lazy.

## 34.9 Mistake: Assuming a Generator Can Be Indexed

In [137]:
gen = (x for x in range(10))
try:
    print(gen[3])   # WRONG: generators do not support indexing at all
except TypeError as e:
    print("TypeError:", e)

import itertools
value = next(itertools.islice(gen, 3, 4), None)   # CORRECT way to get 'the 4th value', consuming up to it
print(value)

TypeError: 'generator' object is not subscriptable
3


> **Note:** Problem: a generator is a pure, one-directional iterator; it has no __getitem__() and no defined 'position 3' independent of having already consumed positions 0 through 2. Solution: use itertools.islice() to skip to (and consume) the desired position, or convert to a list first if repeated indexed access is genuinely needed (at the cost of losing laziness).

## 34.10 Mistake: Consuming a Generator Unintentionally

In [138]:
def get_values():
    return (x for x in range(5))

values = get_values()
has_data = bool(list(values))   # WRONG: this line ACCIDENTALLY consumes 'values' entirely
print(has_data)
print(list(values))              # empty now: values was already drained by the check above

True
[]


> **Note:** Problem: any operation that iterates over a generator, even one that looks like a simple, read-only check (converting to a list just to test truthiness), fully and permanently consumes it as a side effect. Solution: if you need to check 'does this have any items' without losing the data, either materialize it into a list ONCE and reuse that list for both the check and later use, or use a dedicated peek pattern (Part 5.6) that preserves the remaining iterator.

## Part 34 Practice Exercises

**Beginner:** Between calling next() directly on a list, versus calling next() on iter(a_list), which one works without raising an error?

*Answer:* next(iter(a_list)) works, because iter() first converts the iterable list into a genuine iterator, which is what next() requires; calling next() directly on the list itself raises TypeError, since a list is an iterable but not itself an iterator.

**Intermediate:** In 34.6's list-mutation-while-iterating example, explain precisely why the final numbers list still contains some even numbers, even though the loop's intent was clearly to remove all of them.

*Answer:* A list's iterator tracks position purely by numeric index, advancing by one each time. When numbers.remove(n) removes an element, every element after it shifts one position to the left (down by one index) to fill the gap, but the iterator's internal position counter does not know this happened and simply advances to the next index as usual; this causes it to SKIP over whatever element shifted into the position the iterator was about to visit next, since that position was already 'used up' logically. Concretely, when the loop removes the element at index 1 (2), the old element at index 2 (3) shifts down to index 1, but the iterator moves on to index 2, which is now the element that used to be at index 3 (4), so the value 3 is skipped over and never evaluated by the if condition at all, allowing some even numbers to survive if the pattern of removals happens to skip past them this way.

**Advanced:** Using the principle from 34.10, explain why a helper function that does 'if some_generator: process(some_generator)' (relying on a generator object's inherent truthiness) is not actually a safe way to check 'does this generator have any values', and what happens if that check is attempted directly.

*Answer:* A generator object, unlike a list or a string, does not become falsy when it has no remaining values to produce; a generator object is ALWAYS truthy, regardless of whether it is fully exhausted, partially consumed, or brand new, because generator objects do not implement the kind of emptiness check (via __len__ or __bool__) that empty lists or empty strings do. So 'if some_generator:' would always evaluate to True, unconditionally, and would NOT correctly detect an already-exhausted or genuinely empty generator at all, silently masking the exact scenario the check was presumably meant to catch; worse, even if a different check that DID actually attempt to peek at or iterate the generator were used instead (as in 34.10's example), that peek operation would itself consume at least the first item (or more) from the generator as an unavoidable side effect of checking, permanently altering what remains available for whatever code runs afterward, unless the peeked value(s) are deliberately saved and re-supplied to the rest of the pipeline, exactly as the peek_first() pattern from Part 5.6 demonstrates doing correctly.

# Part 35. Real-World Projects

This part presents ten complete, working projects that apply the entire handbook's content together. Each includes the full code, a walk-through of its architecture, and notes on the specific concepts from earlier parts it relies on.

## 35.1 Project: Custom Number Iterator

Architecture: a single iterator class (Part 7) supporting configurable start, stop and step, used directly in a for loop.

In [139]:
class NumberIterator:
    def __init__(self, start, stop, step=1):
        self.current = start
        self.stop = stop
        self.step = step
    def __iter__(self):
        return self
    def __next__(self):
        if (self.step > 0 and self.current >= self.stop) or (self.step < 0 and self.current <= self.stop):
            raise StopIteration
        value = self.current
        self.current += self.step
        return value

print(list(NumberIterator(0, 20, 5)))

[0, 5, 10, 15]


## 35.2 Project: Fibonacci Generator

Architecture: a generator function (Part 9-10) with a configurable count, using tuple assignment to track the running pair of values.

In [140]:
def fibonacci_gen(count):
    a, b = 0, 1
    for _ in range(count):
        yield a
        a, b = b, a + b

print(list(fibonacci_gen(10)))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


## 35.3 Project: Large File Processor (Word Frequency Counter)

Architecture: streams a text file line by line (Part 15, Part 31), tokenizes each line lazily, and accumulates word counts using constant memory relative to the number of lines (though the vocabulary dict itself grows with unique words).

In [141]:
import collections

def word_frequency(path):
    counts = collections.Counter()
    with open(path) as f:
        for line in f:
            for word in line.lower().split():
                counts[word.strip(".,!?;:\"'")] += 1
    return counts

with open("demo_wordfreq.txt", "w") as f:
    f.write("the quick fox. The QUICK dog! the fox runs.\n")

print(word_frequency("demo_wordfreq.txt").most_common(3))

import os
os.remove("demo_wordfreq.txt")

[('the', 3), ('quick', 2), ('fox', 2)]


## 35.4 Project: Log Streaming Generator (Filtering by Severity)

Architecture: a chained generator pipeline (Part 16.1) that parses, filters, and formats log lines from a streamed file.

In [142]:
def parse_logs(path):
    with open(path) as f:
        for line in f:
            parts = line.strip().split(" ", 1)
            if len(parts) == 2:
                yield {"level": parts[0], "message": parts[1]}

def filter_by_level(records, level):
    for r in records:
        if r["level"] == level:
            yield r

with open("demo_app.log", "w") as f:
    f.write("INFO startup complete\nERROR disk full\nINFO request served\nERROR timeout\n")

for error in filter_by_level(parse_logs("demo_app.log"), "ERROR"):
    print(error)

import os
os.remove("demo_app.log")

{'level': 'ERROR', 'message': 'disk full'}
{'level': 'ERROR', 'message': 'timeout'}


## 35.5 Project: CSV Streaming Pipeline (Extract, Transform, Aggregate)

Architecture: combines Part 15's CSV streaming with Part 16's ETL pattern to compute an aggregate without ever loading the full file.

In [143]:
def read_csv(path):
    with open(path) as f:
        header = next(f).strip().split(",")
        for line in f:
            yield dict(zip(header, line.strip().split(",")))

def total_revenue(rows):
    return sum(int(r["quantity"]) * float(r["price"]) for r in rows)

with open("demo_sales.csv", "w") as f:
    f.write("product,quantity,price\nWidget,3,9.99\nGadget,1,49.99\nWidget,2,9.99\n")

print(round(total_revenue(read_csv("demo_sales.csv")), 2))

import os
os.remove("demo_sales.csv")

99.94


## 35.6 Project: Memory-Efficient Data Processing System

Architecture: a multi-stage generator pipeline (validate -> deduplicate -> transform) processing a simulated large dataset in constant memory (aside from the deduplication set, which scales with unique keys).

In [144]:
def source(n):
    for i in range(n):
        yield {"id": i % 700, "value": i}   # deliberately introduces duplicate ids

def validate(records):
    for r in records:
        if r["value"] >= 0:
            yield r

def deduplicate(records):
    seen_ids = set()
    for r in records:
        if r["id"] not in seen_ids:
            seen_ids.add(r["id"])
            yield r

pipeline = deduplicate(validate(source(5000)))
print("Unique records:", sum(1 for _ in pipeline))

Unique records: 700


## 35.7 Project: Batch Data Generator for Machine Learning

Architecture: combines Part 17's mini-batch pattern with shuffling, producing an infinite stream of shuffled mini-batches suitable for a training loop.

In [145]:
import random
random.seed(1)

def training_batches(features, labels, batch_size):
    n = len(features)
    indices = list(range(n))
    while True:
        random.shuffle(indices)
        for start in range(0, n, batch_size):
            batch_idx = indices[start:start + batch_size]
            yield [features[i] for i in batch_idx], [labels[i] for i in batch_idx]

features = list(range(10))
labels = [x % 2 for x in features]
gen = training_batches(features, labels, batch_size=4)
for _ in range(3):
    print(next(gen))

([6, 8, 9, 7], [0, 0, 1, 1])
([5, 3, 0, 4], [1, 1, 0, 0])
([1, 2], [1, 0])


## 35.8 Project: Data Transformation Pipeline (Cleaning + Enrichment)

Architecture: a chained pipeline of small, single-purpose generator stages (Part 16.1), each independently testable.

In [146]:
def strip_strings(records):
    for r in records:
        yield {k: (v.strip() if isinstance(v, str) else v) for k, v in r.items()}

def enrich_with_category(records):
    for r in records:
        r["category"] = "premium" if r.get("price", 0) > 50 else "standard"
        yield r

raw = [{"name": " Widget ", "price": 9.99}, {"name": "Gadget", "price": 59.99}]
for r in enrich_with_category(strip_strings(raw)):
    print(r)

{'name': 'Widget', 'price': 9.99, 'category': 'standard'}
{'name': 'Gadget', 'price': 59.99, 'category': 'premium'}


## 35.9 Project: Streaming AI Data Pipeline (Simulated Token Stream)

Architecture: a generator (Part 18.1) simulating incremental model output, wrapped in a formatter stage that accumulates and prints progressively.

In [147]:
def simulated_model_stream(full_text):
    for word in full_text.split():
        yield word + " "

def display_stream(token_stream):
    accumulated = ""
    for token in token_stream:
        accumulated += token
        yield accumulated

for partial in display_stream(simulated_model_stream("The quick brown fox jumps")):
    print(partial)

The 
The quick 
The quick brown 
The quick brown fox 
The quick brown fox jumps 


## 35.10 Project: Async Data Streaming System (Simulated)

Architecture: an async generator (Part 32.5) simulating records arriving with realistic async delays, consumed with async for.

In [148]:
import asyncio

async def async_data_stream(n):
    for i in range(n):
        await asyncio.sleep(0)   # simulates an async wait per record (e.g. network I/O)
        yield {"id": i, "payload": i * 10}

async def consume():
    results = []
    async for record in async_data_stream(5):
        results.append(record)
    return results

results = await consume()
print(results)

[{'id': 0, 'payload': 0}, {'id': 1, 'payload': 10}, {'id': 2, 'payload': 20}, {'id': 3, 'payload': 30}, {'id': 4, 'payload': 40}]


## Part 35 Practice Exercises

**Beginner:** Which project in this part demonstrates using collections.Counter alongside a generator-based line-by-line file read?

*Answer:* Project 35.3, the Large File Processor (Word Frequency Counter).

**Intermediate:** In Project 35.6 (Memory-Efficient Data Processing System), explain why the deduplicate() stage's memory usage is NOT perfectly constant, unlike validate()'s.

*Answer:* validate() only ever needs to look at the single record currently passing through it to decide whether to yield it, requiring no memory that grows with the amount of data processed. deduplicate(), however, must remember every unique id it has already seen, in its seen_ids set, in order to correctly recognize and skip future duplicates, so its memory usage grows in proportion to the number of DISTINCT ids encountered so far, even though it still only holds one full record at a time otherwise; this is a common, realistic caveat: many real streaming pipelines have at least one stage (deduplication, a running aggregate keyed by category, a join with a lookup table) whose memory scales with the cardinality of some tracked dimension, even when the rest of the pipeline is fully constant-memory.

**Advanced:** Combine ideas from Project 35.5 (CSV Streaming Pipeline) and Project 35.7 (Batch Data Generator) to sketch, in code, a pipeline that reads an arbitrarily large CSV file of labeled training examples and yields infinite, shuffled mini-batches directly from the streamed file, without ever loading the entire CSV into memory as a single list.

*Answer:* def read_csv_stream(path):
    with open(path) as f:
        header = next(f).strip().split(",")
        for line in f:
            yield dict(zip(header, line.strip().split(",")))

def reservoir_shuffle_batches(record_stream, batch_size, buffer_size):
    import random
    buffer = []
    for record in record_stream:
        buffer.append(record)
        if len(buffer) >= buffer_size:
            random.shuffle(buffer)
            while len(buffer) >= batch_size:
                yield buffer[:batch_size]
                buffer = buffer[batch_size:]
    if buffer:
        random.shuffle(buffer)
        yield buffer
This sketch avoids ever materializing the full CSV as one list by combining two lazy stages: read_csv_stream() (from 35.5's pattern) yields one row dict at a time directly from disk, and reservoir_shuffle_batches() only ever holds a bounded buffer of at most buffer_size records at once, a form of approximate, memory-bounded shuffling (a true perfect, unbiased shuffle would require seeing the entire dataset at once, which is precisely the constraint being avoided here), periodically shuffling and draining that bounded buffer into fixed-size batches; peak memory across the whole pipeline stays proportional to buffer_size, a chosen constant, rather than to the total size of the underlying CSV file, satisfying the requirement even for a file far larger than available RAM, directly combining the file-streaming discipline of Part 15/35.5 with the batching discipline of Part 17/35.7.

# Part 36. Interview Questions

This part collects interview-style questions across every category this handbook covers, each with a complete, interview-ready answer. They are organized by category rather than by difficulty alone, so you can review by topic.

## Part 36 Practice Exercises

**Beginner:** What is the difference between an iterable and an iterator?

*Answer:* An iterable is any object that implements __iter__() and can produce an iterator when asked; examples include list, dict, and str. An iterator is the object actually produced (or used) to walk through the data: it implements both __iter__() (returning itself) and __next__() (producing the next value, or raising StopIteration when exhausted). Every iterator is also an iterable, but not every iterable is itself an iterator.

**Beginner:** What does the for loop do internally when you write 'for x in y:'?

*Answer:* It calls iter(y) once to obtain an iterator, then repeatedly calls next() on that iterator, binding each returned value to x and running the loop body, until next() raises StopIteration, at which point the loop catches that exception internally and ends cleanly.

**Beginner:** What keyword turns an ordinary function into a generator function?

*Answer:* yield. Its mere presence anywhere in a function's body causes calling that function to return a generator object instead of running the body immediately.

**Beginner:** What is the difference between a list comprehension and a generator expression, syntactically and behaviorally?

*Answer:* Syntactically, a list comprehension uses square brackets [ ] and a generator expression uses parentheses ( ). Behaviorally, a list comprehension is eager (computes and stores the full result immediately), while a generator expression is lazy (computes each value only on demand, and can only be iterated once).

**Intermediate:** Explain exactly what happens when a generator function hits a yield statement.

*Answer:* Execution pauses at that exact point: the value in the yield expression is handed back to whoever called next(), and the ENTIRE local execution state of the function, every local variable, the current position in the code, any active loop state, is preserved on the generator object. The function does not restart from the top on the next call; it resumes from immediately after that same yield statement, with all local state intact.

**Intermediate:** Why can an iterator generally only be consumed once?

*Answer:* An iterator's job is to track its own position/progress as internal state; once __next__() has been called enough times to exhaust the underlying data and raise StopIteration, that state reflects 'fully consumed' permanently, with no built-in mechanism to reset it back to the beginning. Getting a fresh pass over the same data requires creating a brand-new iterator (often by calling iter() again on the original iterable, or by calling the generator function again).

**Intermediate:** What is the difference between StopIteration and StopAsyncIteration?

*Answer:* StopIteration is raised by a synchronous iterator's __next__() method to signal there are no more values; it is caught automatically by an ordinary for loop. StopAsyncIteration is its asynchronous counterpart, raised by an async iterator's __anext__() coroutine, and is caught automatically by an async for loop instead.

**Intermediate:** What does yield from do, and give one concrete use case.

*Answer:* yield from delegates iteration to another iterable or generator, automatically forwarding every value it produces, one at a time, as though the outer generator had yielded each one itself; it also makes the delegated generator's eventual return value available as the value of the yield from expression itself. A concrete use case is recursively flattening an arbitrarily nested list structure, where each recursive call handles one level of nesting and yield from forwards everything it finds upward through every enclosing call.

**Intermediate:** Why does zip() stop at the shortest input by default, and how do you change that behavior?

*Answer:* By design, zip() pairs corresponding elements across all its input iterables and simply has nothing left to pair once its shortest input is exhausted, so it stops there, silently discarding leftover elements from any longer inputs. Passing strict=True (Python 3.10+) changes this: zip() will instead raise a ValueError immediately if the inputs turn out to have different lengths, rather than silently truncating.

**Advanced:** Explain the difference between .send(value) and next() on a generator, and why priming is required before the first real .send().

*Answer:* next(gen) is equivalent to gen.send(None): it resumes the generator, delivering no meaningful value into whatever yield expression is currently paused. gen.send(value) resumes the generator AND makes the currently-paused yield expression evaluate to that specific value, which the generator's own code can then use, typically assigning it to a local variable. Priming (an initial next(gen) or gen.send(None)) is required because a brand-new generator's execution is paused before its very first line, not yet at any yield expression; there is nothing yet for a real .send(value) to deliver its value into, so attempting a non-None .send() before priming raises TypeError.

**Advanced:** What does 'return value' do inside a generator function, and how would you retrieve that value?

*Answer:* It immediately ends the generator (equivalent to reaching the end of the function body), and the resulting StopIteration exception carries value as its .value attribute; a for loop consuming the generator never sees this value at all, since it silently catches StopIteration without inspecting it. To retrieve it, either manually drive the generator with next() inside a try/except StopIteration block and read e.value from the caught exception, or delegate to the generator with 'result = yield from the_generator()' inside another generator function, which captures the returned value directly as the value of the yield from expression.

**Advanced:** Compare a generator function to a hand-written custom iterator class implementing the same logic. When would you prefer each?

*Answer:* A generator function achieves the same result (a fully protocol-compliant iterator) with far less boilerplate: no explicit __iter__()/__next__() methods, no manually-tracked instance attributes for state, since Python's generator machinery manages all of that automatically from ordinary local variables and control flow. A custom iterator class becomes preferable specifically when the object needs to expose additional public methods or attributes beyond pure iteration (configuration, introspection, custom __repr__, participation in inheritance), none of which a generator function can provide beyond its built-in send()/throw()/close(). For the large majority of ordinary iteration logic, the generator function is simpler, more readable, and generally at least as fast, and is the recommended default.

**Code Output:** What does the following print? gen = (x for x in range(3)); print(list(gen)); print(list(gen))

*Answer:* [0, 1, 2] then []. The first list(gen) call fully drains the generator; the second call finds nothing left, since the same generator object cannot be replayed, and returns an empty list.

**Code Output:** What does the following print? def f():\n    yield 1\n    return 99\n    yield 2\ng = f(); print(next(g)); \ntry:\n    next(g)\nexcept StopIteration as e:\n    print(e.value)

*Answer:* 1 then 99. The first next(g) reaches 'yield 1' and pauses, returning 1. The second next(g) resumes, hits 'return 99', which ends the generator; the resulting StopIteration carries 99 as its .value, printed here, and the line 'yield 2' after the return is unreachable code, never executed at all.

**Debugging:** This code is supposed to remove all negative numbers from a list, but silently leaves some in: 'for n in nums: if n < 0: nums.remove(n)'. What is the bug and how do you fix it?

*Answer:* The bug is mutating nums (via .remove()) while a for loop is actively iterating over it; since a list iterator tracks position by numeric index, removing an element shifts subsequent elements one position earlier, causing the iterator to skip over whatever element shifted into the position it is about to visit next, silently missing some negative numbers depending on their arrangement. The fix is to avoid mutating the list being iterated: either iterate over a copy ('for n in list(nums): if n < 0: nums.remove(n)') or, more idiomatically, build a new list instead ('nums = [n for n in nums if n >= 0]').

**Debugging:** A developer writes a generator-based accumulator, calls gen.send(5) as the VERY FIRST operation on a freshly created generator object, and gets a TypeError. Why, and what is the fix?

*Answer:* A brand-new generator's execution is paused before its very first line of code runs, meaning it has not yet reached any yield expression that a sent value could be delivered into; calling .send() with a real, non-None value on an unprimed generator raises TypeError: can't send non-None value to a just-started generator. The fix is to prime the generator first with a single next(gen) (or gen.send(None)) call, advancing it to its first yield expression, before making any subsequent .send(actual_value) calls.

**Memory/Performance:** Why does 'sum(x**2 for x in range(10_000_000))' use dramatically less peak memory than 'sum([x**2 for x in range(10_000_000)])', even though both compute the identical final number?

*Answer:* The generator expression version computes each squared value one at a time, immediately adds it to sum()'s running total, and discards it, so at any given moment only a small, constant amount of memory (one value plus the running total) is in use, regardless of how large the range is. The list comprehension version first constructs and fully holds an entire list of ten million squared integers in memory simultaneously, before sum() ever begins adding them up, so its peak memory usage scales directly, linearly, with the size of the range, which can be many megabytes of memory that the lazy version never needs to allocate at all.

**Generator Questions:** Can a generator function contain a return statement with no value, and if so, what does it do?

*Answer:* Yes; a bare 'return' (with no value) inside a generator function behaves exactly like reaching the natural end of the function body: it immediately ends the generator, raising StopIteration, but with its .value attribute left as None (the default), since no explicit value was returned.

**Iterator Protocol Questions:** Why must an iterator's __iter__() method return self, rather than, say, a new iterator instance?

*Answer:* Returning self is what satisfies the requirement that every iterator is ALSO a valid iterable (Part 3.4), which is specifically what allows an iterator object to be used directly in a for loop, or passed into iter() again harmlessly and get itself back unchanged; if __iter__() instead returned a different, fresh object each time, that would technically violate the expected iterator contract and could cause confusing behavior, such as 'for x in some_iterator:' behaving inconsistently with manually calling next() on that same object.

**Data Science Questions:** In a Data Science preprocessing pipeline reading a 50GB CSV file, why would you choose a generator-based, line-by-line processing approach over pandas.read_csv() loading the whole file at once?

*Answer:* A 50GB file very likely exceeds available RAM on most machines, so pandas.read_csv() without a chunksize argument would attempt to load the entire dataset into memory as one DataFrame at once, which could raise a MemoryError or severely degrade performance through memory pressure and swapping. A generator-based, line-by-line (or chunked) approach processes the file incrementally, using bounded, roughly constant memory, regardless of the file's total size, allowing the exact same code to handle a 50GB file, or a 500GB file, successfully, provided the actual computation being performed (such as a running aggregate) does not itself require holding the entire dataset in memory; pandas.read_csv(chunksize=...) itself is, not coincidentally, an application of exactly this same generator-based, lazy-chunk-iteration principle.

**AI Engineering Questions:** When streaming a large language model's response token by token in a Python client, why is an async generator typically preferred over a plain synchronous generator?

*Answer:* Retrieving each successive token from a remote model API typically requires waiting on network I/O, which could take a noticeable, variable amount of time per token; a plain synchronous generator's __next__() call would BLOCK the entire single-threaded program for that entire wait, preventing it from doing any other useful work (handling other requests, running other coroutines) during that time. An async generator's __anext__() is itself a coroutine that can 'await' the network operation, explicitly yielding control back to the event loop during the wait, so other work can proceed concurrently, only resuming this particular stream once the awaited network operation genuinely completes; this non-blocking behavior is essential for a responsive, concurrent application handling potentially many simultaneous streaming requests.

# Part 37. Consolidated Practical Exercises

Every part of this handbook already ends with its own beginner, intermediate and advanced exercises focused on that part's specific material. This final exercises part adds a small set of integrative challenge problems that deliberately combine multiple parts together, as a capstone review, plus one debugging challenge.

## Part 37 Practice Exercises

**Challenge:** Build a generator function paginated_records(records, page_size) that, combined with itertools.islice, lets a caller retrieve any specific PAGE of records (e.g. 'give me page 3') from a large, one-time-use record generator, without loading the whole thing into a list first. Discuss the trade-off this design makes.

*Answer:* import itertools

def get_page(record_generator, page_size, page_number):
    start = page_number * page_size
    return list(itertools.islice(record_generator, start, start + page_size))

def records(n):
    for i in range(n):
        yield {"id": i}

gen = records(100)
print(get_page(gen, page_size=10, page_number=0))   # first page
print(get_page(gen, page_size=10, page_number=0))   # NOTE: continues from where gen left off, not page 1 again!
The key trade-off, worth discussing explicitly: because gen is a genuine, one-time-use iterator (Part 4.2), islice(record_generator, start, start+page_size) does NOT rewind to an absolute position each call; it advances from wherever the SAME shared generator object currently sits. This means repeated calls with the SAME page_number on the SAME generator object do NOT return the same page again, unlike true random-access pagination over a list or database query; this generator-based approach trades true random-access re-fetchability (which only a list, or a fresh query per page, can provide) for the ability to page through data that may be too large, or too expensive, to hold entirely in memory or fetch again from scratch, and is appropriate specifically for a strictly forward-only, single-pass consumption pattern.

**Challenge:** Combine itertools.groupby(), a generator-based file reader, and a dict comprehension to compute per-category totals from a simulated CSV stream of (category, amount) rows, WITHOUT loading the whole file into memory, and WITHOUT requiring the file to already be sorted by category (hint: this requires a different, non-groupby-based accumulation strategy for the unsorted case, since groupby() only merges consecutive matches).

*Answer:* def read_transactions(lines):
    for line in lines:
        category, amount = line.strip().split(",")
        yield category, float(amount)

def totals_by_category(transactions):
    totals = {}
    for category, amount in transactions:
        totals[category] = totals.get(category, 0.0) + amount
    return totals

lines = ["food,12.50", "transport,5.00", "food,8.25", "entertainment,20.00", "transport,3.50"]
print(totals_by_category(read_transactions(lines)))
This deliberately does NOT use groupby() for the accumulation itself, precisely because the data is not pre-sorted by category and Part 24.13/25.2 established that groupby() only merges CONSECUTIVE matching keys; sorting the entire stream first would require materializing it fully in memory anyway (per Part 28.4's explanation of why sorted() is inherently eager), defeating the memory-bounded goal. Instead, totals_by_category() uses a plain dict with dict.get(key, 0.0) as a running-total accumulator (Part 9.2 of the Data Types handbook's pattern), which correctly handles categories arriving in ANY order while still processing the transactions stream lazily, one row at a time; its only non-constant memory cost is the totals dict itself, which scales with the number of DISTINCT categories, not the number of transactions, a much smaller and more acceptable growth than holding every individual transaction in memory.

**Debugging Challenge:** The following code is meant to print each unique word in a large document exactly once, in the order first encountered, using a generator, but it has a subtle bug. Find it and fix it. def unique_words(path):\n    seen = set()\n    with open(path) as f:\n        for line in f:\n            for word in line.split():\n                if word not in seen:\n                    yield word\n\n# elsewhere:\nresult1 = list(unique_words('doc.txt'))\nresult2 = list(unique_words('doc.txt'))\nprint(len(result1), len(result2))

*Answer:* The bug is NOT inside unique_words() itself, which is correctly implemented (it never adds words to seen at all, which is actually a SEPARATE bug worth flagging: 'seen.add(word)' is missing entirely, so a word appearing twice on the SAME call would incorrectly be yielded twice within one pass; the fix for that is adding 'seen.add(word)' right after the yield, inside the if block). Separately, the calling code's structure itself reveals a common misunderstanding (Part 34.3/34.4): unique_words('doc.txt') is called TWICE, creating two entirely SEPARATE, independent generator objects and, in this specific case, two independent seen sets too, so both result1 and result2 will actually be correct and identical to each other, since each call reads the file from the beginning as a fresh, independent generator; there is no cross-contamination here since each call gets a brand new local seen set. The genuinely broken bug to fix is the missing 'seen.add(word)': def unique_words(path):\n    seen = set()\n    with open(path) as f:\n        for line in f:\n            for word in line.split():\n                if word not in seen:\n                    seen.add(word)\n                    yield word\nWithout this line, seen never actually accumulates anything, so the 'if word not in seen' check is always True, and EVERY occurrence of every word gets yielded, not just the first, defeating the entire purpose of the function despite it looking superficially correct and despite the two separate calls each behaving consistently with each other.

# Part 38. Complete Iteration Cheat Sheet

A single, consolidated set of reference tables covering every core concept in this handbook, for fast lookup and revision.

## 38.1 Core Concepts

| Term | Definition | Key Requirement |
|---|---|---|
| Iterable | An object that can produce an iterator | Implements __iter__() |
| Iterator | An object that produces values one at a time and tracks position | Implements __iter__() (returns self) and __next__() |
| Generator | A special iterator created automatically from a function containing yield | def with yield anywhere in the body |
| Generator expression | A lazily-evaluated, single-expression generator | (expr for x in iterable [if cond]) |
| iter() | Built-in function; calls obj.__iter__() | iter(iterable) -> iterator |
| next() | Built-in function; calls it.__next__() | next(iterator, default=optional) |

## 38.2 yield and Related Keywords

| Construct | Meaning | Notes |
|---|---|---|
| yield value | Pauses the generator, produces value, preserves all local state | Resumable, unlike return |
| yield | Pauses, produces None, and can RECEIVE a value via .send() | Used as an expression: x = yield |
| yield from iterable_or_gen | Delegates to another iterable/generator, forwarding all its values | Its own value is the delegate's return value |
| return value (in a generator) | Ends the generator; value becomes StopIteration.value | NOT visible to a normal for loop |
| StopIteration | Signals 'no more values'; raised automatically or manually | Caught implicitly by for loops |

## 38.3 Generator Object Methods

| Method | Purpose | Requires Priming First? |
|---|---|---|
| next(gen) / gen.send(None) | Resume, deliver nothing meaningful in | No (this IS the priming call) |
| gen.send(value) | Resume, deliver value into the paused yield expression | Yes, for any non-None value |
| gen.throw(exc_type) | Resume, raise exc_type at the paused yield expression | Yes |
| gen.close() | Raise GeneratorExit at the paused yield; expects the generator to stop | Yes, to have any code to interrupt |

## 38.4 itertools Quick Reference

| Function | Category | Purpose |
|---|---|---|
| count(start, step) | Infinite | Endless arithmetic sequence |
| cycle(iterable) | Infinite | Repeats forever |
| repeat(value, n) | Infinite | Same value, forever or n times |
| product(*its) | Combinatoric | Cartesian product |
| permutations(it, r) | Combinatoric | All orderings of length r |
| combinations(it, r) | Combinatoric | All r-selections, order-independent |
| combinations_with_replacement(it, r) | Combinatoric | Like combinations, repeats allowed |
| accumulate(it, func) | Terminating | Running/cumulative result |
| chain(*its) | Terminating | Iterates several as one |
| compress(data, selectors) | Terminating | Filters by a parallel mask |
| dropwhile(pred, it) / takewhile(pred, it) | Terminating | Skip-then-yield-rest / yield-until-false |
| filterfalse(pred, it) | Terminating | Keeps elements where pred is False |
| groupby(it, key) | Terminating | Groups CONSECUTIVE matching elements |
| islice(it, start, stop, step) | Terminating | Lazy slicing for any iterator |
| pairwise(it) | Terminating | Overlapping successive pairs |
| starmap(func, it_of_tuples) | Terminating | map() that unpacks tuple arguments |
| tee(it, n) | Terminating | Splits into n independent iterators |
| zip_longest(*its, fillvalue) | Terminating | Like zip(), pads to the LONGEST input |

## 38.5 Built-in Iteration Tools

| Tool | Lazy? | Purpose |
|---|---|---|
| zip(*its, strict=False) | Yes | Pairs elements across iterables; stops at shortest unless strict=True |
| enumerate(it, start=0) | Yes | Pairs each element with a running index |
| map(func, it) | Yes | Applies func to each element on demand |
| filter(func, it) | Yes | Keeps elements where func(x) is truthy, on demand |
| reversed(seq) | Yes (needs a real sequence) | Iterates from the end backward |
| sorted(it, key, reverse) | NO - Eager | Fully consumes input; returns a complete sorted list |

## 38.6 Async Iteration

| Concept | Sync Equivalent | Purpose |
|---|---|---|
| async for x in ait: | for x in it: | Async loop; must be inside async def |
| __aiter__(self) | __iter__(self) | Returns an async iterator (usually self) |
| async def __anext__(self) | __next__(self) | Coroutine producing the next value or raising StopAsyncIteration |
| StopAsyncIteration | StopIteration | Signals 'no more values', async iteration |
| async def f(): ... yield x | def f(): ... yield x | Async generator function |

## 38.7 Dictionary Iteration

| Expression | Produces | Live View? |
|---|---|---|
| for k in d: | Keys | N/A (consumed directly) |
| d.keys() | A view of keys | Yes |
| d.values() | A view of values | Yes |
| d.items() | A view of (key, value) tuples | Yes |
| list(d.keys()) | A static COPY of keys, safe to mutate d during iteration afterward | No |

# Conclusion

Every idea in this handbook traces back to one small, precise contract: an iterable can produce an iterator, and an iterator produces values one at a time until it raises StopIteration. Custom iterator classes implement that contract by hand; generators implement it automatically from ordinary function syntax plus yield; generator expressions implement it in a single line; itertools provides a rich library of fast, composable, lazy building blocks on top of it; and async iterators extend the exact same contract to situations that require non-blocking waiting.

The practical payoff of understanding this contract deeply is memory-bounded, composable, production-grade data processing: pipelines that scale to files, datasets, and streams far larger than available memory, training loops that can pull mini-batches indefinitely, and AI systems that can process streamed output incrementally, all built from the same handful of ideas covered here.

This handbook deliberately did not duplicate your companion Data Types, Methods, Functions, Operators, Keywords, OOP, Exception Handling, File Handling, and Modules & Packages references; use them alongside this one. Where this handbook explained how data flows through a program one value at a time, those references explain the language features that flow acts upon. Together, they form a complete, professional foundation for Python development.

---

**Kajola Gbenga | Data Scientist & AI Engineer**